# FINDS Lab — 전체 실증분석 Master Notebook

산업별 주가수익률의 금리민감도 분석 전체를 하나의 노트북에 모은다. `FINAL_paper_analysis.ipynb`의 마크다운·코드 셀은 원문 그대로 포함하고, FINAL에 싣지 않은 보조분석은 관련 절 뒤에 `X.A`로 붙인다.

## 0. Notebook 안내

셀은 세 종류로 구성된다.

- **FINAL 본문**: `FINAL_paper_analysis.ipynb`의 셀을 순서·내용 그대로 옮긴 것. Baseline·H1·H2·H3·종합·Appendix가 여기 해당한다.
- **X.A (Final Notebook 미포함)**: FINAL에는 없지만 01–10 원본 노트북에 있는 분석. 코드 셀은 원본을 그대로 가져오되, 이미 계산된 패널·계수표는 다시 읽지 않고 재사용한다.
- **신규 마크다운**: X.A 절의 도입·관찰문. 원본 노트북의 문장은 옮기지 않고 새로 쓴다.

목차: 1 EDA · 2 Baseline(+2.A) · 3 H1(+3.A) · 4 H2(+4.A) · 5 H3(+5.A) · 6 종합 · 7 Appendix · 8 전체 분석 Index.

# 산업별 주가수익률의 금리민감도: 최종 실증분석

2016–2025년 일별 자료를 이용해 시장수익률을 통제한 산업별 금리민감도를 추정하고, 수익률곡선 요인·금리 방향·산업 재무특성으로 분석을 확장한다. 결과는 관측된 금리변화와 산업수익률의 조건부 관련성으로 해석하며 인과효과로 확대하지 않는다.

**계수 해석 원칙.** 노트북 전체에서 금리 및 수익률곡선 요인 계수의 경제적 크기는 10bp 변화를 기준으로 예상 산업수익률 변화(%)로 서술한다. 원계수와 SE, $t$, $p$, 95% 신뢰구간, $R^2$, 표본수는 결과표에만 둔다.

| 단계 | 질문 | 역할 |
|---|---|---|
| **Baseline** | 산업별 3년 금리베타가 서로 다른가? | 산업 간 차이 확인 |
| **H1** | 차이는 수익률곡선의 어느 요인에서 나타나는가? | 금리요인 분해 |
| **H2** | 금리 상승과 하락에 대한 반응은 다른가? | 방향 비대칭 검정 |
| **H3** | 산업 재무특성과 금리베타는 어떻게 정렬되는가? | 탐색적 해석 |

Baseline에서 산업 간 차이를 확인한 뒤 H1에서 만기와 수익률곡선 요인을 비교한다. H2는 상승·하락 계수의 동일성을 검정하고, H3는 산업 단위 순위관계를 탐색한다. 본문에 필요한 결과를 먼저 제시하고 세부 민감도와 기술적 진단은 Appendix에 둔다.


In [1]:
# 저장소 루트와 분석 환경 설정
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())

_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is None:
    raise FileNotFoundError("config.py와 requirements.txt가 있는 저장소 루트를 찾지 못했습니다.")
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))

import config
from src import models

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from itertools import combinations, permutations
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)
config.setup_korean_font(verbose=False)
plt.rcParams["axes.unicode_minus"] = False

NEG_COLOR, POS_COLOR, NS_COLOR = "#c0392b", "#1f6feb", "#9aa0a6"


def add_bh_correction(result, group_cols=("model",), p_col="p_value", alpha=0.05):
    """검정족(group_cols) 단위로 BH(FDR) 보정 p-value를 추가한다."""
    out = result.copy()
    group_cols = list(group_cols)
    out["p_value_bh"] = np.nan
    _by = group_cols[0] if len(group_cols) == 1 else group_cols
    for _, sub in out.groupby(_by):
        idx = sub.index
        out.loc[idx, "p_value_bh"] = multipletests(
            out.loc[idx, p_col].to_numpy(dtype=float), method="fdr_bh"
        )[1]
    out["significant_5pct_bh"] = out["p_value_bh"] < alpha
    return out


HAC_LAGS = config.HAC_MAXLAGS
print("분석 환경 준비 완료 · HAC lag =", HAC_LAGS, "· NS lambda =", config.NS_LAMBDA)


분석 환경 준비 완료 · HAC lag = 5 · NS lambda = 0.5


## 1. EDA — Final Notebook 미포함

FINAL은 Baseline부터 시작한다. 이 절은 `03_eda_industry.ipynb`의 패널 구조·기초통계·시계열·상관관계 확인을 Baseline 이전에 옮겨 둔다.

### 1.1 데이터·표본 개요

확인할 내용: 패널이 산업×거래일 단위로 균형을 이루는지, 날짜 정렬과 (날짜, 산업) 중복 여부, 항등식(term_spread=y10−y3, d_term_spread=d_y10−d_y3)이 수치오차 범위에서 성립하는지.

In [2]:
# [셀 1] 라이브러리 · 한글폰트 · 경로
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
import seaborn as sns

# 저장소 루트를 찾아 config를 불러온다 (실행 위치/편집기와 무관하게 동작).
# 저장소 루트에서 `pip install -e .` 를 한 번 실행했다면 아래 탐색은 건너뛴다.
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더 (Jupyter/Lab/Colab)
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
# 마커를 하나만 보면 다른 폴더의 동명 config.py 를 잘못 집을 수 있다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
# 이미 sys.path에 있더라도 맨 앞으로 올린다. cwd(예: 바탕화면)가 앞에 있으면
# 같은 이름의 다른 config.py 가 먼저 잡히기 때문이다.
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(
        f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}\n"
        f"저장소 루트({_root})에서 `pip install -e .` 를 실행하세요."
    )


PANEL_CSV = config.PROCESSED_DIR / "industry_rate_panel_daily.csv"   # 00 -> 01에서 저장한 최종 데이터
FIGDIR = str(config.FIGDIR); TABDIR = str(config.TABDIR)
os.makedirs(FIGDIR, exist_ok=True); os.makedirs(TABDIR, exist_ok=True)

# 한글 폰트 (환경별 후보)
for fp in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
           "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
           "C:/Windows/Fonts/malgun.ttf", "/Library/Fonts/AppleGothic.ttf"]:
    if os.path.exists(fp):
        fm.fontManager.addfont(fp); plt.rcParams["font.family"] = fm.FontProperties(fname=fp).get_name(); break
else:
    for c in ["Malgun Gothic","AppleGothic","NanumGothic"]:
        if any(c in f.name for f in fm.fontManager.ttflist): plt.rcParams["font.family"]=c; break
plt.rcParams["axes.unicode_minus"] = False

VARS = ["industry_return","kospi_return","d_y3","d_y10","term_spread","d_term_spread"]
UNIT = {"industry_return":"소수","kospi_return":"소수","d_y3":"%p","d_y10":"%p","term_spread":"%p","d_term_spread":"%p"}

In [3]:
# [셀 2] 데이터 로드 · 정렬
df = pd.read_csv(PANEL_CSV, encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["industry","date"]).reset_index(drop=True)
industries = sorted(df["industry"].unique())
daily = df.drop_duplicates("date").sort_values("date").reset_index(drop=True)  # 날짜당 1행(KOSPI·금리)
print(df.shape, "| 산업", len(industries), "| 거래일", daily.shape[0])

(17164, 12) | 산업 7 | 거래일 2452


In [4]:
# [셀 6] 데이터 검증
chk={}
chk["date_dtype"]=str(df["date"].dtype)
chk["sorted_within_industry"]=bool(df.groupby("industry")["date"].apply(lambda s:s.is_monotonic_increasing).all())
chk["dup_date_industry"]=int(df.duplicated(["date","industry"]).sum())
chk["balanced_panel"]=bool(df["industry"].value_counts().nunique()==1)
chk["missing"]=df[VARS].isna().sum().to_dict()
chk["ir_missing_beyond_first_day"]=int(df[df["date"]!=df["date"].min()]["industry_return"].isna().sum())
chk["identity_d_term_spread_err"]=float((df["d_term_spread"]-(df["d_y10"]-df["d_y3"])).abs().max())
chk["identity_term_spread_err"]=float((df["term_spread"]-(df["y10"]-df["y3"])).abs().max())
chk["outliers_absz_gt4"]={v:int(((df[v]-df[v].mean())/df[v].std()).abs().gt(4).sum()) for v in VARS}
for k,v in chk.items(): print(k,":",v)

date_dtype : datetime64[ns]
sorted_within_industry : True
dup_date_industry : 0
balanced_panel : True
missing : {'industry_return': 7, 'kospi_return': 7, 'd_y3': 7, 'd_y10': 7, 'term_spread': 0, 'd_term_spread': 7}
ir_missing_beyond_first_day : 0
identity_d_term_spread_err : 1.1796119636642288e-16
identity_term_spread_err : 2.220446049250313e-16
outliers_absz_gt4 : {'industry_return': 84, 'kospi_return': 91, 'd_y3': 119, 'd_y10': 77, 'term_spread': 0, 'd_term_spread': 98}


**관찰.** 패널은 7개 산업 × 동일 거래일수의 균형 패널이며, 날짜는 산업별로 오름차순 정렬되어 있고 (날짜, 산업) 중복은 없다. 두 항등식의 최대 절대오차는 수치연산 오차 수준이다. `industry_return`은 각 산업의 표본 첫 거래일에서만 결측이며, `|z|>4` 이상치는 소수 존재한다.

### 1.2 기초통계

확인할 내용: 산업수익률·KOSPI 수익률·금리변화·장단기차의 평균·표준편차·범위, 단위 차이(수익률은 소수, 금리 관련 변수는 %p).

In [5]:
# [셀 3] 기초통계 (평균/표준편차/최소/최대/중앙/결측)
def describe(s):
    return pd.Series({"mean":s.mean(),"std":s.std(),"min":s.min(),"max":s.max(),
                      "median":s.median(),"n_missing":s.isna().sum()})
rows = {"industry_return": describe(df["industry_return"])}
for v in ["kospi_return","d_y3","d_y10","term_spread","d_term_spread"]:
    rows[v] = describe(daily[v])
stats = pd.DataFrame(rows).T; stats["unit"] = [UNIT[v] for v in stats.index]; stats.index.name="variable"
stats.to_csv(f"{TABDIR}/descriptive_statistics.csv", encoding="utf-8-sig")

by_ind = df.groupby("industry")["industry_return"].apply(describe).unstack().reindex(industries)
by_ind.index.name="industry"
by_ind.to_csv(f"{TABDIR}/descriptive_statistics_by_industry.csv", encoding="utf-8-sig")
display(stats.round(6)); display(by_ind.round(6))

,mean,std,min,max,median,n_missing,unit
variable,,,,,,,
industry_return,0.000386,0.016382,-0.133038,0.147541,0.000145,7.0,소수
kospi_return,0.000383,0.011111,-0.087677,0.086012,0.000699,1.0,소수
d_y3,0.000538,0.038799,-0.268000,0.349000,0.000000,1.0,%p
d_y10,0.000547,0.041304,-0.196000,0.321000,0.000000,1.0,%p
term_spread,0.322668,0.220517,-0.213000,1.042000,0.326000,0.0,%p
d_term_spread,0.000009,0.021281,-0.163000,0.117000,0.000000,1.0,%p


,mean,std,min,max,median,n_missing
industry,,,,,,
건설업,0.000134,0.017870,-0.118374,0.147541,-0.000159,1.0
금융업,0.000350,0.013031,-0.103676,0.112835,0.000324,1.0
운수장비,0.000543,0.016231,-0.105239,0.125083,0.000155,1.0
의약품,0.000457,0.018051,-0.117294,0.122435,-0.000288,1.0
전기전자,0.000812,0.015792,-0.095694,0.109055,0.000628,1.0
철강금속,0.000405,0.018379,-0.133038,0.126327,0.000033,1.0
화학,-0.000001,0.014600,-0.115122,0.090169,0.000199,1.0


**관찰.** 산업수익률과 KOSPI 수익률은 평균이 0에 가깝고 일별 변동폭은 산업마다 다르다. 금리 변화(d_y3, d_y10)의 표준편차는 3년물보다 10년물에서 작다.

### 1.3 시계열

확인할 내용: 산업수익률·KOSPI·금리변화·장단기차가 표본기간(2016–2025) 전체에서 특정 구간에 쏠려 있지 않은지, 금리 급변동 구간이 존재하는지.

In [6]:
# [셀 4-1] 산업별 수익률 facet
n=len(industries); ncol=2; nrow=int(np.ceil(n/ncol))
fig,axes=plt.subplots(nrow,ncol,figsize=(14,3.0*nrow),sharex=True); axes=axes.flatten()
for i,ind in enumerate(industries):
    sub=df[df["industry"]==ind]
    axes[i].plot(sub["date"],sub["industry_return"],lw=0.6); axes[i].axhline(0,color="grey",lw=0.5)
    axes[i].set_title(ind); axes[i].set_ylabel("수익률(소수)")
for j in range(n,len(axes)): axes[j].axis("off")
fig.suptitle("산업별 일별 수익률 (industry_return, 단위: 소수)",y=1.01)
fig.tight_layout(); fig.savefig(f"{FIGDIR}/industry_return_timeseries.png",dpi=130,bbox_inches="tight"); plt.show()

**관찰.** 산업별 일별 수익률은 산업 간 변동성 차이를 제외하면 특정 산업이 전체 기간 동안 구조적으로 다른 패턴을 보이지 않는다.

In [7]:
# [셀 4-2] KOSPI 수익률
fig,ax=plt.subplots(figsize=(13,3.5))
ax.plot(daily["date"],daily["kospi_return"],lw=0.7,color="navy"); ax.axhline(0,color="grey",lw=0.5)
ax.set_title("KOSPI 일별 수익률 (kospi_return, 단위: 소수)"); ax.set_ylabel("수익률(소수)")
fig.tight_layout(); fig.savefig(f"{FIGDIR}/kospi_return_timeseries.png",dpi=130,bbox_inches="tight"); plt.show()

**관찰.** KOSPI 수익률은 2020년 초와 같은 급락 구간에서 변동성이 확대된다.

In [8]:
# [셀 4-3] d_y3 & d_y10
fig,ax=plt.subplots(figsize=(13,3.5))
ax.plot(daily["date"],daily["d_y3"],lw=0.7,label="d_y3 (국고3년, %p)")
ax.plot(daily["date"],daily["d_y10"],lw=0.7,label="d_y10 (국고10년, %p)",alpha=0.8)
ax.axhline(0,color="grey",lw=0.5); ax.set_title("국고채 금리 일별 변화분 (%p)"); ax.set_ylabel("변화분(%p)"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIGDIR}/interest_rate_timeseries.png",dpi=130,bbox_inches="tight"); plt.show()

**관찰.** d_y3·d_y10은 2021–2022년 인상기와 2024년 이후 인하 재개 구간에서 변동폭이 커진다.

In [9]:
# [셀 4-4] term_spread & d_term_spread (레벨 vs 변화분)
fig,(a1,a2)=plt.subplots(2,1,figsize=(13,6),sharex=True)
a1.plot(daily["date"],daily["term_spread"],lw=0.8,color="darkgreen"); a1.axhline(0,color="grey",lw=0.5)
a1.set_title("장단기 금리차 term_spread = y10 - y3 (레벨, %p)"); a1.set_ylabel("%p")
a2.plot(daily["date"],daily["d_term_spread"],lw=0.7,color="darkorange"); a2.axhline(0,color="grey",lw=0.5)
a2.set_title("장단기 금리차 변화분 d_term_spread = d_y10 - d_y3 (%p)"); a2.set_ylabel("%p")
fig.tight_layout(); fig.savefig(f"{FIGDIR}/termspread_timeseries.png",dpi=130,bbox_inches="tight"); plt.show()

**관찰.** term_spread(레벨)는 표본기간 동안 완만하게 움직이는 반면 d_term_spread(변화분)는 레벨보다 훨씬 좁은 범위에서 진동한다.

### 1.4 상관관계

확인할 내용: 산업수익률·KOSPI·금리변수 간 Pearson 상관계수.

In [10]:
# [셀 5] Pearson 상관계수 + Heatmap
corr = df[VARS].corr(method="pearson"); corr.to_csv(f"{TABDIR}/correlation_matrix.csv", encoding="utf-8-sig")
fig,ax=plt.subplots(figsize=(7.5,6.2))
sns.heatmap(corr,annot=True,fmt=".2f",cmap="RdBu_r",vmin=-1,vmax=1,square=True,cbar_kws={"shrink":0.8},ax=ax)
ax.set_title("Pearson 상관계수 (STEP2 생성변수)")
fig.tight_layout(); fig.savefig(f"{FIGDIR}/correlation_heatmap.png",dpi=130,bbox_inches="tight"); plt.show()
display(corr.round(3))

,industry_return,kospi_return,d_y3,d_y10,term_spread,d_term_spread
industry_return,1.000,0.680,-0.028,-0.005,0.033,0.041
kospi_return,0.680,1.000,-0.054,-0.025,0.042,0.051
d_y3,-0.028,-0.054,1.000,0.861,0.029,-0.153
d_y10,-0.005,-0.025,0.861,1.000,0.053,0.372
term_spread,0.033,0.042,0.029,0.053,1.000,0.048
d_term_spread,0.041,0.051,-0.153,0.372,0.048,1.000


**관찰.** 산업수익률은 KOSPI 수익률과의 상관이 가장 크고, d_y3와 d_y10도 서로 강한 상관을 보인다. 상세 상관계수는 위 결과표에 제시한다. 이는 이후 H1에서 만기별 계수를 나란히 비교할 때 개별 변수가 서로 독립적이지 않다는 점을 함께 읽어야 함을 뜻한다.

### 1.5 데이터 검증

**관찰.** 1.1에서 실행한 검증 결과(균형 패널, 정렬, 항등식, 이상치 개수)는 이후 분석에서 별도의 표본 정제 없이 패널을 그대로 사용해도 되는 근거가 된다.

# 1. Baseline — 산업별 금리민감도 차이

## 연구질문과 검정

KOSPI 수익률을 통제한 뒤에도 산업별 3년 금리민감도가 서로 다른가?

$$
R_{i,t} = \alpha_i + \gamma_i R_{KOSPI,t} + \beta_{rate,i}\,\Delta y_{3,t} + \varepsilon_{i,t}
$$

$R_{i,t}$는 산업 $i$의 일별 수익률, $R_{KOSPI,t}$는 KOSPI 수익률, $\Delta y_{3,t}$는 국고채 3년물 금리의 일별 변화(%p)다. 산업별 계수는 HAC(Newey–West) 표준오차로 추정한다. 주 검정은 같은 날 산업들이 공유하는 충격을 고려한 날짜 클러스터 표준오차로 $H_0:\beta_1=\cdots=\beta_7$을 검정한다.


In [11]:
# 산업 · KOSPI · 금리 패널
panel = pd.read_csv(config.PROCESSED_DIR / "industry_rate_panel_daily.csv", encoding="utf-8-sig")
panel["date"] = pd.to_datetime(panel["date"])
industries = sorted(panel["industry"].unique())

print("표본 기간:", panel["date"].min().date(), "~", panel["date"].max().date())
print("산업 수:", len(industries), "|", industries)

# 산업별 개별 회귀(HAC) + 7개 산업에 대한 BH(FDR) 보정
beta_table = models.industry_rate_betas(panel)
baseline_table = (
    beta_table[["industry", "beta_rate", "p_value_bh"]]
    .rename(columns={"industry": "산업", "beta_rate": "3년 금리베타", "p_value_bh": "BH p-value"})
)
baseline_table["방향"] = np.where(
    beta_table["p_value_bh"] >= 0.05, "비유의",
    np.where(beta_table["beta_rate"] > 0, "유의한 +", "유의한 -"),
)
baseline_table["10bp 상승 시 예상 산업수익률 변화(%)"] = baseline_table["3년 금리베타"] * 10
baseline_effect = baseline_table.set_index("산업")["10bp 상승 시 예상 산업수익률 변화(%)"]
display(baseline_table.round(5))
print(
    f"3년물 금리가 10bp 오르면 의약품 지수는 평균 {abs(baseline_effect['의약품']):.2f}% 내리고, "
    f"금융업·운수장비·철강금속 지수는 각각 {baseline_effect['금융업']:+.2f}%, "
    f"{baseline_effect['운수장비']:+.2f}%, {baseline_effect['철강금속']:+.2f}% 변하는 것으로 추정된다."
)

표본 기간: 2016-01-04 ~ 2025-12-30
산업 수: 7 | ['건설업', '금융업', '운수장비', '의약품', '전기전자', '철강금속', '화학']


,산업,3년 금리베타,BH p-value,방향
0,건설업,0.00320,0.71155,비유의
1,금융업,0.01123,0.00822,유의한 +
2,운수장비,0.01723,0.00822,유의한 +
3,의약품,-0.02707,0.00384,유의한 -
4,전기전자,0.00159,0.71155,비유의
5,철강금속,0.01884,0.02862,유의한 +
6,화학,0.00256,0.71155,비유의


## 산업별 계수

- **의약품은 10bp 금리 상승에 유의한 음(-)의 수익률 반응을 보였다.**
- 금융업·운수장비·철강금속은 10bp 금리 상승에 유의한 양(+)의 수익률 반응을 보였다.
- 건설업·전기전자·화학은 BH 보정 후 유의수준에 도달하지 않았다.

아래 그림은 산업별 점추정과 95% 신뢰구간을 비교한다. 색은 BH 보정 후 방향과 유의성을 나타낸다.


In [12]:
# 산업별 금리베타 계수 플롯 (95% 신뢰구간, BH 유의성 기준 색상)
beta_plot = beta_table.assign(color=[
    (NEG_COLOR if b < 0 else POS_COLOR) if p < 0.05 else NS_COLOR
    for b, p in zip(beta_table["beta_rate"], beta_table["p_value_bh"])
]).sort_values("beta_rate")

fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(beta_plot))
for yi, row in zip(y, beta_plot.itertuples()):
    lower, upper = row.beta_rate - row.ci_lower, row.ci_upper - row.beta_rate
    ax.errorbar(row.beta_rate, yi, xerr=[[lower], [upper]], fmt="o", markersize=7,
                capsize=4, elinewidth=1.6, ecolor=row.color, color=row.color)
ax.axvline(0, linestyle="--", linewidth=1, color="black")
ax.set_yticks(y)
ax.set_yticklabels(beta_plot["industry"])
ax.set_xlabel("3년 금리베타")
ax.set_title("산업별 금리베타와 95% 신뢰구간 (HAC 표준오차)", loc="left", fontweight="bold")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

의약품은 음(-) 방향에 위치한 반면 금융업·운수장비·철강금속은 양(+) 방향에 위치한다. 이 그림은 계수의 방향과 불확실성을 보여주지만 산업 간 차이의 유의성은 다음 공동검정으로 판단한다.


In [13]:
# 산업 간 금리베타 동일성 공동검정 (H0: 7개 산업의 금리베타가 모두 같다)
# 날짜 클러스터 표준오차 — 같은 날 여러 산업이 함께 받는 충격(횡단면 잔차 상관)을 보정한다.
joint_test = models.cross_industry_Ftest(
    panel.rename(columns={"industry_return": "ret", "kospi_return": "mkt", "d_y3": "dY"})
    [["industry", "ret", "mkt", "dY", "date"]]
)
joint_test_table = pd.DataFrame([{
    "검정": "산업별 금리베타 동일성 공동검정",
    "표준오차": "날짜 클러스터",
    "F-statistic": float(joint_test.fvalue),
    "p-value": float(joint_test.pvalue),
}])
display(joint_test_table.round(5))

공동검정(날짜 클러스터 SE): F = 4.189, p = 0.00034


## 결과와 해석 범위

**산업별 금리베타의 동일성은 기각되었다.** 시장수익률을 통제한 뒤에도 산업별 3년 금리민감도에는 통계적으로 유의한 차이가 있다. 검정통계량과 p-value는 위 결과표에 제시한다.

양(+)의 계수를 금리상승의 인과효과로 읽지는 않는다. 관측된 금리변화에는 경기·물가 정보가 함께 포함될 수 있다. 정상국(2003)의 산업별 지수 사용, Sweeney & Warga(1986)의 시장·금리 2요인 모형, Bernanke & Kuttner(2005)의 시장요인 통제 논리는 이 모형의 설계 근거다.

Baseline은 산업 간 차이를 확인한다. 다음 H1에서는 이 차이가 단일 3년물에 한정되는지, 수익률곡선의 어느 요인에서 나타나는지 살펴본다.


In [14]:
# Baseline 시점의 panel·industries를 보존한다.
# 이후 절(06/07/08 등 원본 노트북 셀)이 전역 변수 panel/industries를
# 다른 내용으로 재정의하는 경우가 있어, H3·Appendix에서 FINAL 원문을
# 그대로 재현하려면 이 시점 값을 되돌려 쓸 수 있어야 한다.
# (새 계산이 아니라 상태 보존이다.)
_panel_baseline = panel.copy()
_industries_baseline = list(industries)

### 2.A 추가 보조분석 — Final Notebook 미포함

`04_rate_beta_estimation.ipynb`·`05_baseline_industry_difference.ipynb`의 보조 시각화 세 가지를 옮긴다.

**2.A.1 산업별 모형 설명력(R²).** 확인할 내용: Baseline 개별회귀의 R²가 산업마다 얼마나 다른지.

In [15]:
# 설정 · 데이터 로드



import pandas as pd
import matplotlib.pyplot as plt

# 저장소 루트를 찾아 config를 불러온다 (실행 위치/편집기와 무관하게 동작).
# 저장소 루트에서 `pip install -e .` 를 한 번 실행했다면 아래 탐색은 건너뛴다.
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더 (Jupyter/Lab/Colab)
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
# 마커를 하나만 보면 다른 폴더의 동명 config.py 를 잘못 집을 수 있다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
# 이미 sys.path에 있더라도 맨 앞으로 올린다. cwd(예: 바탕화면)가 앞에 있으면
# 같은 이름의 다른 config.py 가 먼저 잡히기 때문이다.
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(
        f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}\n"
        f"저장소 루트({_root})에서 `pip install -e .` 를 실행하세요."
    )

from src import models

# 결과물 저장 경로 (후속 가설 검정용 산출물)
OUT_DIR = config.PROJECT_ROOT / "results" / "Baseline_industry_beta"
TAB_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
for _d in (TAB_DIR, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(config.PROCESSED_DIR / "industry_rate_panel_daily.csv", encoding="utf-8-sig")
panel["date"] = pd.to_datetime(panel["date"])
print("패널:", panel.shape, "| 산업 수:", panel["industry"].nunique())
panel.head()

패널: (17164, 12) | 산업 수: 7


,date,industry,industry_index,industry_return,kospi_index,kospi_return,y3,d_y3,y10,d_y10,term_spread,d_term_spread
0,2016-01-04,건설업,104.43,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN
1,2016-01-04,금융업,408.22,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN
2,2016-01-04,운수장비,1731.95,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN
3,2016-01-04,의약품,8411.35,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN
4,2016-01-04,전기전자,9239.12,NaN,1918.76,NaN,1.634,NaN,2.039,NaN,0.405,NaN


In [16]:
# 산업별 금리 베타 추정(HAC) 및 결과표 저장
beta_table = models.industry_rate_betas(panel)
beta_table.to_csv(TAB_DIR / "rate_beta_by_industry_hac.csv", index=False, encoding="utf-8-sig")
beta_table.round(5)

,industry,beta_rate,std_error,t_value,p_value,p_value_bh,ci_lower,ci_upper,n_obs,r_squared
0,건설업,0.00320,0.00834,0.38421,0.70082,0.71155,-0.01314,0.01955,2451,0.39824
1,금융업,0.01123,0.00385,2.91797,0.00352,0.00822,0.00369,0.01878,2451,0.62286
2,운수장비,0.01723,0.00581,2.96585,0.00302,0.00822,0.00584,0.02862,2451,0.47196
3,의약품,-0.02707,0.00783,-3.45610,0.00055,0.00384,-0.04242,-0.01172,2451,0.24326
4,전기전자,0.00159,0.00429,0.36977,0.71155,0.71155,-0.00682,0.01000,2451,0.71928
5,철강금속,0.01884,0.00785,2.40088,0.01636,0.02862,0.00346,0.03421,2451,0.38342
6,화학,0.00256,0.00491,0.52072,0.60256,0.71155,-0.00707,0.01219,2451,0.64364


In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 입력 검증
required_columns = {
    "industry", "beta_rate", "std_error", "t_value", "p_value",
    "p_value_bh", "ci_lower", "ci_upper", "n_obs", "r_squared",
}
missing_columns = required_columns.difference(beta_table.columns)
if missing_columns:
    raise ValueError("beta_table 누락 컬럼: " + ", ".join(sorted(missing_columns)))

plot_table = beta_table.copy()
numeric_columns = [
    "beta_rate", "std_error", "t_value", "p_value",
    "p_value_bh", "ci_lower", "ci_upper", "n_obs", "r_squared",
]
plot_table[numeric_columns] = plot_table[numeric_columns].apply(pd.to_numeric, errors="raise")
if plot_table[numeric_columns].isna().any().any():
    raise ValueError("시각화 입력값에 결측치가 있습니다.")

# 한글 폰트
font_candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]
installed_fonts = {f.name for f in font_manager.fontManager.ttflist}
selected_font = next((n for n in font_candidates if n in installed_fonts), "DejaVu Sans")
plt.rcParams["font.family"] = selected_font
plt.rcParams["axes.unicode_minus"] = False

# 색·축 공통 설정
REPORT_BLUE, REPORT_RED, REPORT_GRAY = "#2563EB", "#E11D48", "#94A3B8"
REPORT_DARK, GRID_COLOR = "#334155", "#E2E8F0"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def significance_group(row):
    if row["p_value_bh"] < 0.05:
        return "positive" if row["beta_rate"] > 0 else "negative"
    return "neutral"

plot_table["significance_group"] = plot_table.apply(significance_group, axis=1)
color_map = {"positive": REPORT_BLUE, "negative": REPORT_RED, "neutral": REPORT_GRAY}

def finish_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRID_COLOR)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8, alpha=0.9)
    ax.set_axisbelow(True)


In [18]:
# 시각화 3: 산업별 모형 설명력 R²
r2_sorted = plot_table.sort_values("r_squared").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.5, 5.6))
bars = ax.barh(r2_sorted["industry"], r2_sorted["r_squared"],
               color=REPORT_DARK, height=0.60)
ax.set_title("산업별 모형 설명력 (R²)", loc="left",
             fontsize=16, fontweight="bold", pad=16)
ax.set_xlabel("R²")
ax.set_xlim(0, max(0.80, r2_sorted["r_squared"].max() * 1.12))
finish_axis(ax)
for bar, value in zip(bars, r2_sorted["r_squared"]):
    ax.text(value + 0.012, bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}", va="center", ha="left", fontsize=10, fontweight="bold")

fig.tight_layout()
fig.savefig(FIG_DIR / "03_industry_model_r_squared.png", dpi=200, bbox_inches="tight")
plt.show()


/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/2889841150.py:16: UserWarning: Glyph 51032 (\N{HANGUL SYLLABLE YI}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/2889841150.py:16: UserWarning: Glyph 50557 (\N{HANGUL SYLLABLE YAG}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/2889841150.py:16: UserWarning: Glyph 54408 (\N{HANGUL SYLLABLE PUM}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/2889841150.py:16: UserWarning: Glyph 52384 (\N{HANGUL SYLLABLE CEOL}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/2889841150.py:16: UserWarning: Glyph 44053 (\N{HANGUL SYLLABLE GANG}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/2889841150.py:16: UserWarning: Glyph 44552 (\N{HA

**관찰.** R²는 산업마다 차이가 있으며, 시장수익률 하나로 설명되는 부분이 대부분이고 금리베타가 더하는 설명력은 산업마다 다르다.

**2.A.2 원 p-value와 BH 보정 p-value 비교.** 확인할 내용: BH 보정이 어떤 산업의 유의성 판정을 바꾸는지.

In [19]:
# 시각화 4: 원 p-value와 BH 보정 p-value 비교
p_sorted = plot_table.sort_values("p_value_bh").reset_index(drop=True)
y_positions = np.arange(len(p_sorted))
bar_height = 0.34

fig, ax = plt.subplots(figsize=(9.5, 5.8))
ax.barh(y_positions - bar_height / 2, p_sorted["p_value"],
        height=bar_height, color="#93C5FD", label="원 p-value")
ax.barh(y_positions + bar_height / 2, p_sorted["p_value_bh"],
        height=bar_height, color=REPORT_BLUE, label="BH 보정 p-value")
ax.axvline(0.05, color=REPORT_RED, linewidth=1.3, linestyle="--", label="유의수준 0.05")
ax.set_yticks(y_positions)
ax.set_yticklabels(p_sorted["industry"])
ax.set_title("원 p-value와 BH 보정 p-value 비교", loc="left",
             fontsize=16, fontweight="bold", pad=16)
ax.set_xlabel("p-value")
ax.legend(frameon=False, loc="lower right")
finish_axis(ax)

fig.tight_layout()
fig.savefig(FIG_DIR / "04_raw_vs_bh_pvalue.png", dpi=200, bbox_inches="tight")
plt.show()


/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/214597458.py:20: UserWarning: Glyph 51032 (\N{HANGUL SYLLABLE YI}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/214597458.py:20: UserWarning: Glyph 50557 (\N{HANGUL SYLLABLE YAG}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/214597458.py:20: UserWarning: Glyph 54408 (\N{HANGUL SYLLABLE PUM}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/214597458.py:20: UserWarning: Glyph 44552 (\N{HANGUL SYLLABLE GEUM}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/214597458.py:20: UserWarning: Glyph 50997 (\N{HANGUL SYLLABLE YUNG}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/214597458.py:20: UserWarning: Glyph 50629 (\N{HANGUL S

**관찰.** BH 보정 후 p-value는 원 p-value보다 항상 크거나 같으며, 경계에 있던 산업의 판정이 바뀔 수 있다.

**2.A.3 산업 쌍별 금리베타 차이 Heatmap.** 확인할 내용: 21개 산업 쌍 전체에서 베타 차이의 크기·부호와 BH 유의 쌍의 분포.

In [20]:
# 설정 · 데이터 로드



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from matplotlib import font_manager
from matplotlib.lines import Line2D

# 저장소 루트를 찾아 config를 불러온다 (실행 위치/편집기와 무관하게 동작).
# 저장소 루트에서 `pip install -e .` 를 한 번 실행했다면 아래 탐색은 건너뛴다.
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더 (Jupyter/Lab/Colab)
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
# 마커를 하나만 보면 다른 폴더의 동명 config.py 를 잘못 집을 수 있다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
# 이미 sys.path에 있더라도 맨 앞으로 올린다. cwd(예: 바탕화면)가 앞에 있으면
# 같은 이름의 다른 config.py 가 먼저 잡히기 때문이다.
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(
        f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}\n"
        f"저장소 루트({_root})에서 `pip install -e .` 를 실행하세요."
    )

from src import models

# ---------------------------------------------------------------
# 한글 폰트 (OS별 후보를 순서대로 시도). 폰트가 없어도 분석은 진행한다.
# 특정 PC 경로(C:/Windows/...)만 보면 다른 팀원 환경에서 한글이 깨진다.
# ---------------------------------------------------------------
for _fp in ["C:/Windows/Fonts/malgun.ttf",
            "/Library/Fonts/AppleGothic.ttf",
            "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
            "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
    if Path(_fp).exists():
        font_manager.fontManager.addfont(_fp)
        plt.rcParams["font.family"] = font_manager.FontProperties(fname=_fp).get_name()
        break
else:
    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic",
                  "Noto Sans CJK KR", "Noto Sans CJK JP"]:
        if any(_name in f.name for f in font_manager.fontManager.ttflist):
            plt.rcParams["font.family"] = _name
            break
    else:
        print("한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")
plt.rcParams["axes.unicode_minus"] = False
print("그래프 폰트:", plt.rcParams["font.family"])

BASELINE_DIR = config.BASELINE_DIR
TAB_DIR = BASELINE_DIR / "tables"
FIG_DIR = BASELINE_DIR / "figures"
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(config.PROCESSED_DIR / "industry_rate_panel_daily.csv", encoding="utf-8-sig")
panel["date"] = pd.to_datetime(panel["date"])

beta_table = models.industry_rate_betas(panel)

print("panel:", panel.shape, "| 산업 수:", panel["industry"].nunique())

그래프 폰트: ['Noto Sans CJK JP']
panel: (17164, 12) | 산업 수: 7


In [21]:
# 분석 데이터 확인 및 정리
required_cols = ["date", "industry", "industry_return", "kospi_return", "d_y3"]
missing_cols = [col for col in required_cols if col not in panel.columns]

if missing_cols:
    raise ValueError(
        "panel에 STEP5 분석에 필요한 컬럼이 없습니다: " + ", ".join(missing_cols)
    )

h1_data = (
    panel[required_cols]
    .copy()
    .dropna()
    .sort_values(["date", "industry"])
    .reset_index(drop=True)
)

industries = sorted(h1_data["industry"].unique())
REFERENCE_INDUSTRY = "건설업" if "건설업" in industries else industries[0]

sample_check = (
    h1_data.groupby("industry")
    .agg(시작일=("date", "min"), 종료일=("date", "max"), 표본수=("date", "size"))
)

print("분석 행 수:", len(h1_data))
print("분석 산업 수:", len(industries))
print("산업 목록:", industries)
print("비교 설명에 사용할 기준 산업:", REFERENCE_INDUSTRY)
print("산업별 표본 수 동일 여부:", sample_check["표본수"].nunique() == 1)
display(sample_check)

분석 행 수: 17157
분석 산업 수: 7
산업 목록: ['건설업', '금융업', '운수장비', '의약품', '전기전자', '철강금속', '화학']
비교 설명에 사용할 기준 산업: 건설업
산업별 표본 수 동일 여부: True


,시작일,종료일,표본수
industry,,,
건설업,2016-01-05,2025-12-30,2451
금융업,2016-01-05,2025-12-30,2451
운수장비,2016-01-05,2025-12-30,2451
의약품,2016-01-05,2025-12-30,2451
전기전자,2016-01-05,2025-12-30,2451
철강금속,2016-01-05,2025-12-30,2451
화학,2016-01-05,2025-12-30,2451


In [22]:
# 통합 회귀모형 추정
HAC_LAGS = 5

h1_formula = (
    "industry_return ~ 0 + C(industry)"
    " + C(industry):kospi_return"
    " + C(industry):d_y3"
)

h1_model = smf.ols(
    formula=h1_formula,
    data=h1_data,
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": HAC_LAGS},
)

print("HAC maxlags:", HAC_LAGS)
print("관측치 수:", int(h1_model.nobs))
print("설명변수 수:", len(h1_model.params))

# 날짜 기준 Cluster Robust — 같은 날 7개 산업의 잔차 상관(횡단면 상관)을 보정한다.
# 통합회귀는 여러 산업을 한 식에 넣으므로 시계열 HAC만으로는 이 상관을 놓친다.
# 점추정(params)은 cov_type과 무관하므로 계수는 두 모형이 동일하다. SE·t·p·CI만 달라진다.
date_codes = pd.factorize(h1_data["date"])[0]

h1_model_cluster = smf.ols(formula=h1_formula, data=h1_data).fit(
    cov_type="cluster", cov_kwds={"groups": date_codes},
)

assert np.allclose(h1_model.params.values, h1_model_cluster.params.values), \
    "두 모형의 점추정이 다릅니다. 같은 식·같은 표본인지 확인하세요."

print("날짜 그룹 수:", len(np.unique(date_codes)))
print("점추정 일치 확인 — 최대 절대차:",
      float(np.max(np.abs(h1_model.params.values - h1_model_cluster.params.values))))

HAC maxlags: 5
관측치 수: 17157
설명변수 수: 21


날짜 그룹 수: 2451
점추정 일치 확인 — 최대 절대차: 0.0


In [23]:
# 통합 회귀에서 산업별 금리베타 추출
def find_rate_beta_param(model, industry):
    exact_name = f"C(industry)[{industry}]:d_y3"
    if exact_name in model.params.index:
        return exact_name

    candidates = [
        name for name in model.params.index
        if "C(industry)" in name
        and "d_y3" in name
        and str(industry) in name
    ]

    if len(candidates) != 1:
        raise KeyError(
            f"{industry}의 금리베타 계수명을 찾지 못했습니다. "
            f"후보: {candidates}"
        )
    return candidates[0]


beta_rows = []
h1_model_conf_int = h1_model.conf_int()
h1_model_cluster_conf_int = h1_model_cluster.conf_int()

for industry in industries:
    param_name = find_rate_beta_param(h1_model, industry)
    ci_low, ci_high = h1_model_conf_int.loc[param_name]

    beta_rows.append({
        "industry": industry,
        "beta_rate_pooled": h1_model.params[param_name],
        "std_error_pooled": h1_model.bse[param_name],
        "t_value_pooled": h1_model.tvalues[param_name],
        "p_value_pooled": h1_model.pvalues[param_name],
        "ci_lower_pooled": ci_low,
        "ci_upper_pooled": ci_high,
        "std_error_cluster": h1_model_cluster.bse[param_name],
        "t_value_cluster": h1_model_cluster.tvalues[param_name],
        "p_value_cluster": h1_model_cluster.pvalues[param_name],
        "ci_lower_cluster": h1_model_cluster_conf_int.loc[param_name][0],
        "ci_upper_cluster": h1_model_cluster_conf_int.loc[param_name][1],
        "parameter_name": param_name,
    })

pooled_beta_table = pd.DataFrame(beta_rows)

display(
    pooled_beta_table[
        [
            "industry",
            "beta_rate_pooled",
            "std_error_pooled",
            "t_value_pooled",
            "p_value_pooled",
            "ci_lower_pooled",
            "ci_upper_pooled",
            "std_error_cluster",
            "p_value_cluster",
        ]
    ].round(5)
)

,industry,beta_rate_pooled,std_error_pooled,t_value_pooled,p_value_pooled,ci_lower_pooled,ci_upper_pooled,std_error_cluster,p_value_cluster
0,건설업,0.00320,0.00728,0.44017,0.65982,-0.01107,0.01747,0.00729,0.66007
1,금융업,0.01123,0.00427,2.62983,0.00854,0.00286,0.01961,0.00427,0.00859
2,운수장비,0.01723,0.00580,2.96825,0.00299,0.00585,0.02861,0.00581,0.00302
3,의약품,-0.02707,0.00759,-3.56452,0.00036,-0.04195,-0.01218,0.00760,0.00037
4,전기전자,0.00159,0.00423,0.37513,0.70757,-0.00670,0.00988,0.00423,0.70779
5,철강금속,0.01884,0.00751,2.50925,0.01210,0.00412,0.03355,0.00751,0.01217
6,화학,0.00256,0.00477,0.53631,0.59174,-0.00679,0.01191,0.00477,0.59204


In [24]:
# 산업별 금리베타 동일성 공동검정 (Wald/F 검정)
param_names = list(h1_model.params.index)
reference_param = find_rate_beta_param(h1_model, REFERENCE_INDUSTRY)

R = []

for industry in industries:
    if industry == REFERENCE_INDUSTRY:
        continue

    industry_param = find_rate_beta_param(h1_model, industry)
    restriction = np.zeros(len(param_names))

    restriction[param_names.index(industry_param)] = 1
    restriction[param_names.index(reference_param)] = -1
    R.append(restriction)

R = np.vstack(R)

try:
    joint_test = h1_model.wald_test(R, use_f=True, scalar=True)
except TypeError:
    joint_test = h1_model.wald_test(R, use_f=True)

joint_stat = float(np.asarray(joint_test.statistic).squeeze())
joint_pvalue = float(np.asarray(joint_test.pvalue).squeeze())

# 날짜 클러스터 표준오차 기준 공동검정 — 같은 제약행렬 R로 h1_model_cluster에 대해 검정
try:
    joint_test_cluster = h1_model_cluster.wald_test(R, use_f=True, scalar=True)
except TypeError:
    joint_test_cluster = h1_model_cluster.wald_test(R, use_f=True)

joint_stat_cluster = float(np.asarray(joint_test_cluster.statistic).squeeze())
joint_pvalue_cluster = float(np.asarray(joint_test_cluster.pvalue).squeeze())

joint_result = pd.DataFrame([{
    "test": "산업별 금리베타 동일성 공동검정",
    "reference_industry": REFERENCE_INDUSTRY,
    "statistic": joint_stat,
    "df_num": getattr(joint_test, "df_num", len(industries) - 1),
    "df_denom": getattr(joint_test, "df_denom", h1_model.df_resid),
    "p_value": joint_pvalue,
    "decision_5pct": (
        "H0 기각: 산업별 금리베타 차이 존재"
        if joint_pvalue < 0.05
        else "H0 기각 못함: 산업별 차이 근거 부족"
    ),
    "statistic_cluster": joint_stat_cluster,
    "p_value_cluster": joint_pvalue_cluster,
    "decision_5pct_cluster": (
        "H0 기각: 산업별 금리베타 차이 존재"
        if joint_pvalue_cluster < 0.05
        else "H0 기각 못함: 산업별 차이 근거 부족"
    ),
}])

display(joint_result.round(6))

print("공동검정 — HAC 기준: F =", round(joint_stat, 4), ", p =", round(joint_pvalue, 6))
print("공동검정 — 클러스터 기준: F =", round(joint_stat_cluster, 4), ", p =", round(joint_pvalue_cluster, 6))


,test,reference_industry,statistic,df_num,df_denom,p_value,decision_5pct,statistic_cluster,p_value_cluster,decision_5pct_cluster
0,산업별 금리베타 동일성 공동검정,건설업,4.26942,6.0,17136.0,0.000264,H0 기각: 산업별 금리베타 차이 존재,4.189156,0.000338,H0 기각: 산업별 금리베타 차이 존재


공동검정 — HAC 기준: F = 4.2694 , p = 0.000264
공동검정 — 클러스터 기준: F = 4.1892 , p = 0.000338


In [25]:
# 산업 쌍별 금리베타 차이 검정과 BH 보정
pairwise_rows = []

for industry_a, industry_b in combinations(industries, 2):
    param_a = find_rate_beta_param(h1_model, industry_a)
    param_b = find_rate_beta_param(h1_model, industry_b)

    contrast = np.zeros(len(param_names))
    contrast[param_names.index(param_a)] = 1
    contrast[param_names.index(param_b)] = -1

    test = h1_model.t_test(contrast)
    test_cluster = h1_model_cluster.t_test(contrast)

    difference = float(np.asarray(test.effect).squeeze())
    std_error = float(np.asarray(test.sd).squeeze())
    t_value = float(np.asarray(test.tvalue).squeeze())
    p_value = float(np.asarray(test.pvalue).squeeze())
    ci_low, ci_high = np.asarray(test.conf_int()).reshape(-1, 2)[0]

    std_error_cluster = float(np.asarray(test_cluster.sd).squeeze())
    t_value_cluster = float(np.asarray(test_cluster.tvalue).squeeze())
    p_value_cluster = float(np.asarray(test_cluster.pvalue).squeeze())

    pairwise_rows.append({
        "industry_a": industry_a,
        "industry_b": industry_b,
        "beta_difference_a_minus_b": difference,
        "std_error": std_error,
        "t_value": t_value,
        "p_value": p_value,
        "ci_lower": ci_low,
        "ci_upper": ci_high,
        "std_error_cluster": std_error_cluster,
        "t_value_cluster": t_value_cluster,
        "p_value_cluster": p_value_cluster,
    })

pairwise_table = pd.DataFrame(pairwise_rows)

pairwise_table["p_value_bh"] = multipletests(
    pairwise_table["p_value"],
    method="fdr_bh",
)[1]

pairwise_table["significant_bh_5pct"] = (
    pairwise_table["p_value_bh"] < 0.05
)

pairwise_table["p_value_bh_cluster"] = multipletests(
    pairwise_table["p_value_cluster"], method="fdr_bh")[1]
pairwise_table["significant_bh_5pct_cluster"] = pairwise_table["p_value_bh_cluster"] < 0.05

pairwise_table = pairwise_table.sort_values(
    ["p_value_bh", "p_value"]
).reset_index(drop=True)

display(pairwise_table.round(5))

print(
    "BH 보정 후 유의한 산업 쌍 수:",
    int(pairwise_table["significant_bh_5pct"].sum()),
    "/",
    len(pairwise_table),
)

print("BH 보정 후 유의한 산업 쌍 — HAC 기준:",
      int(pairwise_table["significant_bh_5pct"].sum()), "/", len(pairwise_table))
print("BH 보정 후 유의한 산업 쌍 — 클러스터 기준:",
      int(pairwise_table["significant_bh_5pct_cluster"].sum()), "/", len(pairwise_table))
_flip = pairwise_table[
    pairwise_table["significant_bh_5pct"] != pairwise_table["significant_bh_5pct_cluster"]]
print("판정이 달라진 쌍:", len(_flip))
if len(_flip):
    display(_flip[["industry_a", "industry_b", "p_value_bh", "p_value_bh_cluster"]].round(5))

,industry_a,industry_b,beta_difference_a_minus_b,std_error,t_value,p_value,ci_lower,ci_upper,std_error_cluster,t_value_cluster,p_value_cluster,p_value_bh,significant_bh_5pct,p_value_bh_cluster,significant_bh_5pct_cluster
0,운수장비,의약품,0.04430,0.01009,4.39158,0.00001,0.02453,0.06406,0.01020,4.34400,0.00001,0.00018,True,0.00025,True
1,금융업,의약품,0.03830,0.00890,4.30263,0.00002,0.02085,0.05575,0.00906,4.22587,0.00002,0.00018,True,0.00025,True
2,의약품,철강금속,-0.04590,0.01122,-4.09234,0.00004,-0.06789,-0.02392,0.01149,-3.99637,0.00006,0.00030,True,0.00045,True
3,의약품,화학,-0.02962,0.00913,-3.24327,0.00118,-0.04753,-0.01172,0.00928,-3.19077,0.00142,0.00620,True,0.00745,True
4,의약품,전기전자,-0.02865,0.00922,-3.10855,0.00188,-0.04672,-0.01059,0.00933,-3.07227,0.00212,0.00790,True,0.00892,True
5,건설업,의약품,0.03027,0.01065,2.84160,0.00449,0.00939,0.05115,0.01082,2.79835,0.00514,0.01571,True,0.01798,True
6,철강금속,화학,0.01628,0.00765,2.12761,0.03337,0.00128,0.03127,0.00738,2.20484,0.02747,0.10011,False,0.08061,False
7,운수장비,화학,0.01467,0.00717,2.04554,0.04080,0.00061,0.02873,0.00679,2.16084,0.03071,0.10711,False,0.08061,False
8,운수장비,전기전자,0.01564,0.00822,1.90347,0.05698,-0.00046,0.03175,0.00869,1.80098,0.07171,0.13295,False,0.15006,False
9,전기전자,철강금속,-0.01725,0.00961,-1.79447,0.07274,-0.03609,0.00159,0.00981,-1.75884,0.07860,0.15275,False,0.15006,False


BH 보정 후 유의한 산업 쌍 수: 6 / 21
BH 보정 후 유의한 산업 쌍 — HAC 기준: 6 / 21
BH 보정 후 유의한 산업 쌍 — 클러스터 기준: 6 / 21
판정이 달라진 쌍: 0


In [26]:
# 산업 쌍별 베타 차이 Heatmap
n_industries = len(industries)
difference_matrix = pd.DataFrame(
    np.zeros((n_industries, n_industries)),
    index=industries,
    columns=industries,
)

significance_matrix = pd.DataFrame(
    False,
    index=industries,
    columns=industries,
)

for _, row in pairwise_table.iterrows():
    a = row["industry_a"]
    b = row["industry_b"]
    diff = row["beta_difference_a_minus_b"]
    sig = row["significant_bh_5pct_cluster"]

    difference_matrix.loc[a, b] = diff
    difference_matrix.loc[b, a] = -diff
    significance_matrix.loc[a, b] = sig
    significance_matrix.loc[b, a] = sig

fig, ax = plt.subplots(figsize=(8.5, 7.2))

max_abs = np.abs(difference_matrix.values).max()
image = ax.imshow(
    difference_matrix.values,
    vmin=-max_abs,
    vmax=max_abs,
    cmap="coolwarm_r",
)

ax.set_xticks(np.arange(n_industries))
ax.set_yticks(np.arange(n_industries))
ax.set_xticklabels(industries, rotation=45, ha="right")
ax.set_yticklabels(industries)

for i in range(n_industries):
    for j in range(n_industries):
        value = difference_matrix.iloc[i, j]
        star = "*" if significance_matrix.iloc[i, j] else ""
        ax.text(
            j,
            i,
            f"{value:+.3f}{star}",
            ha="center",
            va="center",
            fontsize=8.5,
        )

ax.set_title(
    "산업 쌍별 금리베타 차이\n(행 산업 - 열 산업, *: 날짜 클러스터 BH 보정 p<0.05)",
    loc="left",
    fontweight="bold",
)
fig.colorbar(image, ax=ax, label="금리베타 차이")
fig.tight_layout()

fig.savefig(FIG_DIR / "05_pairwise_beta_difference_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

**관찰.** Heatmap은 Baseline 공동검정이 기각되는 이유가 특정 한 쌍이 아니라 여러 쌍에 걸친 차이에서 온다는 것을 보여준다. 유의한 쌍은 대부분 의약품을 포함한다.

# 2. H1 — 만기와 수익률곡선 요인

## 3년·10년·장단기차 연결

Baseline의 3년물 결과에서 바로 수익률곡선 요인으로 넘어가기 전에 3년물, 10년물, 장단기차(10Y−3Y)의 산업별 HAC 회귀를 비교한다. 이 비교는 단일 만기에서 곡선의 수준·기울기·곡률로 확장하는 연결 단계다.


In [27]:
# 기존 만기별 HAC 회귀 결과를 불러와 연결표를 구성한다.
maturity_results = pd.read_csv(config.H1_TABDIR / "maturity_beta_comparison.csv")
maturity_order = ["3Y", "10Y", "Term_Spread"]
maturity_role = {
    "3Y": "Baseline",
    "10Y": "장기금리 비교",
    "Term_Spread": "기울기 대리",
}

bridge_rows = []
for model_name in maturity_order:
    sub = maturity_results[maturity_results["model"] == model_name]
    pharma = sub.loc[sub["industry"] == "의약품"].iloc[0]
    bridge_rows.append({
        "금리 변수": model_name,
        "개별 p<0.05 산업 수": int((sub["p_value"] < 0.05).sum()),
        "의약품 계수": pharma["coefficient"],
        "의약품 p-value": pharma["p_value"],
        "역할": maturity_role[model_name],
    })
maturity_bridge = pd.DataFrame(bridge_rows)
maturity_bridge.round(4)


,금리 변수,개별 p<0.05 산업 수,의약품 계수,의약품 p-value,역할
0,3Y,4,-0.0271,0.0005,Baseline
1,10Y,2,-0.0160,0.0697,장기금리 비교
2,Term_Spread,0,0.0296,0.0913,기울기 대리


3년물에서는 네 산업의 개별 계수가 5% 수준에 도달하지만 10년물에서는 패턴이 약해지고, 장단기차에서는 개별 유의 산업이 확인되지 않는다. 이 표는 산업별 개별 HAC 회귀의 비교이며, 산업 간 동일성에 대한 주 검정은 Baseline의 날짜 클러스터 공동검정이다.

만기별 결과만으로 곡선 움직임의 성격을 분리하기는 어렵다. 이에 국고채 1·3·5·10년을 Nelson–Siegel Level·Slope·Curvature로 분해한다.


## Nelson–Siegel 요인분해

H1의 질문은 개별 요인계수가 0인지보다 **같은 요인의 계수가 산업마다 같은지**이다. 감쇠계수는 요인 분리를 기준으로 $\lambda=0.5$를 사용하고, 곡선 재구성 오차가 가장 작은 $\lambda=0.8041$을 민감도 비교값으로 둔다.

$$
R_{i,t} = \alpha_i + \gamma_i R_{KOSPI,t} + \beta_{L,i}\Delta Level_t + \beta_{S,i}\Delta Slope_t + \beta_{C,i}\Delta Curvature_t + \varepsilon_{i,t}
$$


In [28]:
# 확장 산업 패널(NS 요인·신용스프레드 포함)
extended_panel = pd.read_csv(config.EXTENDED_PANEL_PATH, encoding="utf-8-sig")
extended_panel["date"] = pd.to_datetime(extended_panel["date"], errors="coerce")
print("확장 패널:", extended_panel.shape, "| NS lambda =", config.NS_LAMBDA)


def run_yield_curve_factor_model(data, hac_lags=HAC_LAGS, x_cols=None, factor_labels=None):
    """산업별 다요인 동시모형(HAC)."""
    if x_cols is None:
        x_cols = ["kospi_return", "d_ns_level", "d_ns_slope", "d_ns_curvature"]
    if factor_labels is None:
        factor_labels = {"d_ns_level": "Level", "d_ns_slope": "Slope", "d_ns_curvature": "Curvature"}

    rows = []
    for industry, group in data.groupby("industry"):
        reg_data = group[["industry_return"] + x_cols].dropna()
        X = sm.add_constant(reg_data[x_cols], has_constant="add")
        model = sm.OLS(reg_data["industry_return"], X).fit(
            cov_type="HAC", cov_kwds={"maxlags": hac_lags}
        )
        for variable, factor in factor_labels.items():
            rows.append({
                "industry": industry,
                "factor": factor,
                "coefficient": model.params[variable],
                "std_error_hac": model.bse[variable],
                "t_value": model.tvalues[variable],
                "p_value": model.pvalues[variable],
            })
    return pd.DataFrame(rows)


yield_curve_factor_betas = run_yield_curve_factor_model(extended_panel)
curvature_table = (
    yield_curve_factor_betas[yield_curve_factor_betas["factor"] == "Curvature"]
    .rename(columns={"industry": "산업", "coefficient": "Curvature 계수", "p_value": "p-value"})
    [["산업", "Curvature 계수", "p-value"]]
)
curvature_table["곡률 10bp 변화 시 예상 산업수익률 변화(%)"] = curvature_table["Curvature 계수"] * 10
curvature_effect = curvature_table.set_index("산업")["곡률 10bp 변화 시 예상 산업수익률 변화(%)"]
display(curvature_table.round(6))
print(
    f"곡률 요인이 10bp 커질 때 의약품 지수는 평균 {curvature_effect['의약품']:.2f}% 변하고, "
    f"운수장비 지수는 {curvature_effect['운수장비']:+.2f}% 변하는 것으로 추정된다."
)


확장 패널: (17164, 32) | NS lambda = 0.5


,산업,Curvature 계수,p-value
2,건설업,-0.002338,0.444568
5,금융업,0.002413,0.078067
8,운수장비,0.006207,0.007206
11,의약품,-0.008599,0.002363
14,전기전자,0.001441,0.362277
17,철강금속,0.004148,0.149483
20,화학,-0.000080,0.962436


10bp 곡률 변화에 대한 예상 수익률 변화는 의약품과 운수장비에서 반대 방향이다. 경제적 크기는 위 결과표에서 환산하고, 원계수와 추론통계는 표에 남겼다.


In [29]:
def run_industry_difference_test(data, test_vars, model_name,
                                  ret_col="industry_return", mkt_col="kospi_return",
                                  industry_col="industry", date_col="date"):
    """H0: 모든 산업의 계수가 같다(요인별 F검정). 날짜 클러스터 표준오차."""
    test_vars = list(test_vars)
    d = data.dropna(subset=[ret_col, mkt_col, industry_col, date_col] + test_vars).copy()
    date_codes = pd.factorize(d[date_col])[0]

    dummies = pd.get_dummies(d[industry_col], prefix="ind", dtype=float)
    blocks = [dummies, dummies.mul(d[mkt_col].values, axis=0).add_prefix("mkt_")]
    for v in test_vars:
        blocks.append(dummies.mul(d[v].values, axis=0).add_prefix(f"{v}__"))
    X = pd.concat(blocks, axis=1)
    names = list(X.columns)

    model = sm.OLS(d[ret_col].values, X.values).fit(cov_type="cluster", cov_kwds={"groups": date_codes})

    rows = []
    for v in test_vars:
        idx = [i for i, c in enumerate(names) if c.startswith(f"{v}__")]
        R = np.zeros((len(idx) - 1, len(names)))
        for r, j in enumerate(idx[1:]):
            R[r, idx[0]] = 1.0
            R[r, j] = -1.0
        t = model.f_test(R)
        rows.append({"model": model_name, "variable": v, "f_stat": float(t.fvalue), "p_value": float(t.pvalue)})
    return pd.DataFrame(rows)


industry_difference_test = pd.concat([
    run_industry_difference_test(extended_panel, ["d_ns_level", "d_ns_slope", "d_ns_curvature"], "B_ThreeFactor"),
    run_industry_difference_test(extended_panel, ["d_ns_level", "d_ns_slope", "d_ns_curvature", "d_credit_spread"], "C_FourFactor"),
], ignore_index=True)

FACTOR_NAME = {"d_ns_level": "Level", "d_ns_slope": "Slope", "d_ns_curvature": "Curvature", "d_credit_spread": "CreditSpread"}
industry_difference_test.insert(2, "factor", industry_difference_test["variable"].map(FACTOR_NAME))
industry_difference_test = add_bh_correction(industry_difference_test, group_cols=("model",))

# 본문 표: Level·Slope·Curvature는 3요인 모형, Credit Spread는 4요인 모형 기준
h1_headline = pd.concat([
    industry_difference_test[(industry_difference_test["model"] == "B_ThreeFactor")],
    industry_difference_test[(industry_difference_test["model"] == "C_FourFactor") & (industry_difference_test["factor"] == "CreditSpread")],
])[["factor", "f_stat", "p_value_bh"]].rename(columns={"factor": "요인", "f_stat": "산업 간 F", "p_value_bh": "BH p-value"})
h1_headline.round(4)

,요인,산업 간 F,BH p-value
0,Level,1.4724,0.2755
1,Slope,1.2363,0.2843
2,Curvature,3.5800,0.0047
6,CreditSpread,1.0833,0.3699


## 산업 간 동일성 검정

| 요인 | 판단 |
|---|---|
| Level (F=1.472, BH p=0.2755) | 산업 간 차이 확인 안 됨 |
| Slope (F=1.236, BH p=0.2843) | 산업 간 차이 확인 안 됨 |
| **Curvature (F=3.580, BH p=0.0047)** | **$\lambda=0.5$에서 산업 간 차이 관찰** |
| Credit Spread (F=1.083, BH p=0.3699) | 산업 간 차이 확인 안 됨 |

아래 그림은 네 요인의 산업 간 동일성 검정 BH p-value를 같은 축에서 비교한다.


In [30]:
# 요인별 산업 간 차이 검정 p-value 비교 (로그축, 붉은 점선 = 0.05)
plot_h1 = h1_headline.sort_values("BH p-value", ascending=False)
colors = [POS_COLOR if p < 0.05 else NS_COLOR for p in plot_h1["BH p-value"]]

fig, ax = plt.subplots(figsize=(7.5, 3.5))
y = np.arange(len(plot_h1))
ax.barh(y, plot_h1["BH p-value"], color=colors)
ax.axvline(0.05, linestyle="--", linewidth=1.2, color="crimson")
ax.set_xscale("log")
ax.set_xticks([0.005, 0.01, 0.05, 0.1])
ax.set_xticklabels(["0.005", "0.01", "0.05", "0.1"])
ax.set_yticks(y)
ax.set_yticklabels(plot_h1["요인"])
ax.set_xlabel("산업 간 차이 검정 BH p-value (로그축)")
ax.set_title("요인별 산업 간 차이 — Curvature만 0.05 아래", loc="left", fontweight="bold")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

Curvature만 5% 기준 아래에 있다. Level·Slope·Credit Spread의 비유의 결과는 모든 금리요인에서 산업 차이가 나타나는 것은 아님을 보여준다.


In [31]:
# 국고채 1·3·5·10년 '변화분'의 주성분분석 — 수익률곡선이 실질적으로 몇 차원인지 확인
YIELD_CHANGE_COLS = ["d_y1", "d_y3_new", "d_y5", "d_y10_new"]
yield_changes = (
    extended_panel[["date"] + YIELD_CHANGE_COLS]
    .drop_duplicates(subset="date").dropna().sort_values("date")
)
Z = yield_changes[YIELD_CHANGE_COLS].to_numpy(dtype=float)
Z_centered = Z - Z.mean(axis=0)
eigenvalues, _ = np.linalg.eigh(np.cov(Z_centered, rowvar=False))
eigenvalues = eigenvalues[::-1]
explained = eigenvalues / eigenvalues.sum()

print(f"PC1 설명비중: {explained[0]*100:.2f}%   PC1+PC2: {explained[:2].sum()*100:.2f}%")

PC1 설명비중: 92.09%   PC1+PC2: 96.88%


## 해석의 범위

PC1은 금리곡선 변화의 92.09%를 설명한다. 표본기간의 국고채 금리는 대부분 함께 움직였으므로 Level·Slope·Curvature를 완전히 독립적인 충격으로 해석하기 어렵다.

**대체 $\lambda=0.8041$에서는 Curvature의 산업 간 차이가 5% 수준에서 유지되지 않았다.** 따라서 Curvature 결과는 $\lambda=0.5$ 설정에서 관찰된 조건부 결과다. 곡률이 산업 차이의 확정적 원인이라고 보지 않는다.

직교화와 FWL 검증은 공유변동과 계수 구조를 확인하는 기술적 진단이다. Curvature와 Credit Spread가 일부 변동을 공유한다는 사실만으로 어느 변수에 경제적 효과가 귀속되는지는 식별되지 않는다. 세부 민감도는 Appendix B에 제시한다.

H1은 산업 차이가 곡률에서 관찰될 수 있음을 보이지만 설정 의존성도 함께 드러낸다. H2는 요인 분해와 별개로 금리 상승과 하락의 반응이 같은지 직접 검정한다.


### 3.A H1 보조 — Final Notebook 미포함

`06_yield_curve_exposure.ipynb`의 λ 선택·나비 스프레드·Credit Spread·PCA·직교화 등 강건성·보조분석을 옮긴다. 06은 국고채 원자료에서 `extended_panel`을 직접 재구성하지만, 여기서는 FINAL[13]이 이미 읽어들인 `extended_panel`을 그대로 쓰고 날짜별 만기 자료(`yield_wide`)만 그 안에서 다시 추출한다(재수집 없음).

In [32]:
# 06의 원자료 재구성 대신, 이미 읽은 extended_panel에서 날짜별 만기 자료를 추출한다.
# (재수집·재계산 없음 — extended_panel의 y1·y3_new·y5·y10_new 열을 날짜 단위로 펴는 것뿐이다.)
yield_wide = (
    extended_panel[["date", "y1", "y3_new", "y5", "y10_new"]]
    .drop_duplicates(subset="date")
    .sort_values("date")
    .reset_index(drop=True)
)
print("yield_wide:", yield_wide.shape)

yield_wide: (2452, 5)


**3.A.1 λ 선택 근거와 민감도.** 확인할 내용: λ를 고르는 네 가지 기준(재구성오차·최대VIF·상관 등)이 서로 다른 값을 주는지, λ=0.2~2.0 전 구간에서 결론(BH 유의 산업 수)이 흔들리는지.

In [33]:
MATURITIES = np.array([1.0, 3.0, 5.0, 10.0])

def nelson_siegel_loadings(maturities, lambda_value):
    maturities = np.asarray(maturities, dtype=float)

    slope_loading = (
        1.0 - np.exp(-lambda_value * maturities)
    ) / (
        lambda_value * maturities
    )

    curvature_loading = (
        slope_loading - np.exp(-lambda_value * maturities)
    )

    return np.column_stack([
        np.ones_like(maturities),
        slope_loading,
        curvature_loading,
    ])

yield_matrix = yield_wide[
    ["y1", "y3_new", "y5", "y10_new"]
].to_numpy(dtype=float)


def total_reconstruction_sse(lambda_value):
    """주어진 lambda 로 곡선을 재구성했을 때의 전체 제곱오차."""
    if lambda_value <= 0:
        return np.inf

    X = nelson_siegel_loadings(MATURITIES, lambda_value)
    betas = yield_matrix @ np.linalg.pinv(X).T
    return float(np.sum((yield_matrix - betas @ X.T) ** 2))


def factor_separation(lambda_value):
    """요인 간 최대 VIF. 값이 작을수록 요인이 잘 분리된다."""
    X = nelson_siegel_loadings(MATURITIES, lambda_value)
    betas = yield_matrix @ np.linalg.pinv(X).T
    d = pd.DataFrame(betas, columns=["L", "S", "C"]).diff().dropna()
    design = sm.add_constant(d, has_constant="add").to_numpy(dtype=float)
    return max(variance_inflation_factor(design, i + 1) for i in range(3))


# 기본값은 config 에서 가져온다(팀 합의값).
OPTIMAL_LAMBDA = config.NS_LAMBDA
COMPARE_LAMBDA = config.NS_LAMBDA_COMPARE

print(f"{'':10}{'lambda':>9}{'곡률정점':>10}{'재구성SSE':>12}{'최대VIF':>10}")
for lam, tag in [(OPTIMAL_LAMBDA, "기본값"), (COMPARE_LAMBDA, "비교값")]:
    print(f"{tag:<10}{lam:>9.4f}{1.7932/lam:>9.2f}년"
          f"{total_reconstruction_sse(lam):>12.6f}{factor_separation(lam):>10.2f}")

print()
print(f"이후 분석은 기본값 lambda = {OPTIMAL_LAMBDA} 로 진행한다.")
print("선택 기준: 산업별 요인 분리 (config.NS_LAMBDA)")

             lambda      곡률정점      재구성SSE     최대VIF
기본값          0.5000     3.59년    2.011858      7.17
비교값          0.8041     2.23년    1.148787      9.45

이후 분석은 기본값 lambda = 0.5 로 진행한다.
선택 기준: 산업별 요인 분리 (config.NS_LAMBDA)


In [34]:
from scipy.optimize import minimize_scalar


def lambda_diagnostics(lam):
    """λ 하나의 적합도·분리도. 주가는 쓰지 않는다."""
    X = nelson_siegel_loadings(MATURITIES, lam)
    B = yield_matrix @ np.linalg.pinv(X).T
    rmse = float(np.sqrt(np.mean((yield_matrix - B @ X.T) ** 2)))

    f = pd.DataFrame(B, columns=["Level", "Slope", "Curvature"]).diff().dropna()
    corr = f.corr()
    design = sm.add_constant(f, has_constant="add").to_numpy(dtype=float)
    vifs = [variance_inflation_factor(design, i + 1) for i in range(3)]

    return {"rmse": rmse, "max_vif": max(vifs),
            "abs_corr_SC": abs(float(corr.loc["Slope", "Curvature"])),
            "mean_abs_corr": float(np.mean([abs(corr.loc["Level", "Slope"]),
                                            abs(corr.loc["Slope", "Curvature"]),
                                            abs(corr.loc["Level", "Curvature"])]))}


rows = []
for name, key in [("곡선 재구성 오차 최소", "rmse"),
                  ("최대 VIF 최소 (채택 기준)", "max_vif"),
                  ("|corr(Slope,Curv)| 최소", "abs_corr_SC"),
                  ("요인 간 평균 |상관| 최소", "mean_abs_corr")]:
    r = minimize_scalar(lambda x: lambda_diagnostics(x)[key],
                        bounds=(0.05, 3.0), method="bounded", options={"xatol": 1e-6})
    st = lambda_diagnostics(r.x)
    rows.append({"기준": name, "최적 λ": round(r.x, 4),
                 "곡률정점(년)": round(1.7932 / r.x, 2),
                 "RMSE": round(st["rmse"], 5),
                 "최대VIF": round(st["max_vif"], 2),
                 "|corr(S,C)|": round(st["abs_corr_SC"], 3)})

lambda_criteria = pd.DataFrame(rows)
lambda_criteria.to_csv(config.H1_TABDIR / "lambda_selection_criteria.csv",
                       index=False, encoding="utf-8-sig")
display(lambda_criteria)

print(f"채택값 λ = {OPTIMAL_LAMBDA} (요인 분리 기준 0.52를 반올림)")
print(f"비교값 λ = {COMPARE_LAMBDA} (곡선 적합도 기준)")
print(f"참고: Diebold-Li(2006)는 곡률정점 30개월 고정 → λ = {1.7932/2.5:.4f}")

,기준,최적 λ,곡률정점(년),RMSE,최대VIF,"|corr(S,C)|"
0,곡선 재구성 오차 최소,0.8032,2.23,0.01082,9.44,0.558
1,최대 VIF 최소 (채택 기준),0.5202,3.45,0.01390,7.06,0.214
2,"|corr(Slope,Curv)| 최소",0.4334,4.14,0.01584,7.79,0.000
3,요인 간 평균 |상관| 최소,2.3741,0.76,0.02132,193.07,0.983


채택값 λ = 0.5 (요인 분리 기준 0.52를 반올림)
비교값 λ = 0.8041 (곡선 적합도 기준)
참고: Diebold-Li(2006)는 곡률정점 30개월 고정 → λ = 0.7173


In [35]:
LAMBDA_GRID = [0.2, 0.3, 0.4, 0.5, 0.6, 0.8041, 1.0, 1.5, 2.0]


def lambda_case(lam):
    """λ 하나에 대해 겹침 지표와 회귀 결론을 함께 계산한다."""
    X = nelson_siegel_loadings(MATURITIES, lam)
    B = yield_matrix @ np.linalg.pinv(X).T
    f = pd.DataFrame(B, index=yield_wide["date"],
                     columns=["L", "S", "C"]).diff().dropna()
    f.columns = ["d_L", "d_S", "d_C"]
    f = f.reset_index()

    dd = extended_panel[["date", "industry", "industry_return", "kospi_return"]]         .merge(f, on="date", how="inner")
    fac = ["d_L", "d_S", "d_C"]

    pvals, joint = [], {}
    for ind, g in dd.groupby("industry"):
        z = g[["industry_return", "kospi_return"] + fac].dropna()
        m = sm.OLS(z["industry_return"],
                   sm.add_constant(z[["kospi_return"] + fac], has_constant="add")
                   ).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
        for v in fac:
            pvals.append({"industry": ind, "var": v, "p": m.pvalues[v]})
        joint[ind] = float(m.f_test("d_L = 0, d_S = 0, d_C = 0").pvalue)

    P = pd.DataFrame(pvals)
    P["p_bh"] = P.groupby("var")["p"].transform(
        lambda s: multipletests(s, method="fdr_bh")[1])

    st = lambda_diagnostics(lam)
    return {"lambda": lam, "곡률정점": round(1.7932 / lam, 2),
            "RMSE": round(st["rmse"], 5), "최대VIF": round(st["max_vif"], 2),
            "개별_원p유의": int((P["p"] < 0.05).sum()),
            "개별_BH유의": int((P["p_bh"] < 0.05).sum()),
            "동시검정_유의산업": int(sum(v < 0.05 for v in joint.values())),
            "의약품_동시p": round(joint.get("의약품", np.nan), 5),
            "운수장비_동시p": round(joint.get("운수장비", np.nan), 5)}


lambda_sensitivity = pd.DataFrame([lambda_case(l) for l in LAMBDA_GRID])
lambda_sensitivity.to_csv(config.H1_TABDIR / "lambda_sensitivity.csv",
                          index=False, encoding="utf-8-sig")
display(lambda_sensitivity)

print(f"동시검정 유의 산업 수 범위: "
      f"{lambda_sensitivity['동시검정_유의산업'].min()} ~ "
      f"{lambda_sensitivity['동시검정_유의산업'].max()}")
print(f"최대 VIF 최솟값: {lambda_sensitivity['최대VIF'].min():.2f}")

,lambda,곡률정점,RMSE,최대VIF,개별_원p유의,개별_BH유의,동시검정_유의산업,의약품_동시p,운수장비_동시p
0,0.2000,8.97,0.02213,52.05,3,0,3,0.00446,0.01356
1,0.3000,5.98,0.01928,13.68,5,4,3,0.00377,0.01348
2,0.4000,4.48,0.01665,8.37,4,3,3,0.00320,0.01346
3,0.5000,3.59,0.01432,7.17,3,2,3,0.00273,0.01350
4,0.6000,2.99,0.01245,7.53,3,0,3,0.00237,0.01358
5,0.8041,2.23,0.01082,9.45,1,0,3,0.00185,0.01383
6,1.0000,1.79,0.01195,12.43,1,0,3,0.00156,0.01411
7,1.5000,1.20,0.01721,30.97,1,0,3,0.00121,0.01468
8,2.0000,0.90,0.02024,88.73,1,0,3,0.00110,0.01496


동시검정 유의 산업 수 범위: 3 ~ 3
최대 VIF 최솟값: 7.17


In [36]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

ax = axes[0]
ax.plot(lambda_sensitivity["lambda"], lambda_sensitivity["개별_BH유의"],
        marker="o", linewidth=1.8, label="개별 BH 유의 (/21)")
ax.plot(lambda_sensitivity["lambda"], lambda_sensitivity["동시검정_유의산업"],
        marker="s", linewidth=1.8, label="동시검정 유의 산업 (/7)")
ax.axvline(OPTIMAL_LAMBDA, linestyle="--", color="crimson", linewidth=1.3)
ax.text(OPTIMAL_LAMBDA, ax.get_ylim()[1] * 0.95, " 채택 λ",
        color="crimson", fontsize=9, va="top")
ax.set_xlabel("λ"); ax.set_ylabel("개수")
ax.set_title("결론이 λ에 흔들리는가", loc="left", fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.25)

ax = axes[1]
ax.plot(lambda_sensitivity["lambda"], lambda_sensitivity["최대VIF"],
        marker="o", linewidth=1.8, color="tab:orange")
ax.axhline(10, linestyle=":", color="crimson", linewidth=1.3)
ax.axvline(OPTIMAL_LAMBDA, linestyle="--", color="crimson", linewidth=1.3)
ax.set_xlabel("λ"); ax.set_ylabel("최대 VIF")
ax.set_title("요인 겹침 (낮을수록 분리 잘 됨)", loc="left", fontweight="bold")
ax.grid(alpha=0.25)

ax = axes[2]
ax.plot(lambda_sensitivity["lambda"], lambda_sensitivity["RMSE"],
        marker="o", linewidth=1.8, color="tab:green")
ax.axvline(OPTIMAL_LAMBDA, linestyle="--", color="crimson", linewidth=1.3)
ax.axvline(COMPARE_LAMBDA, linestyle=":", color="gray", linewidth=1.3)
ax.text(COMPARE_LAMBDA, ax.get_ylim()[1] * 0.95, " 비교 λ",
        color="gray", fontsize=9, va="top")
ax.set_xlabel("λ"); ax.set_ylabel("곡선 재구성 RMSE")
ax.set_title("곡선 적합도 (낮을수록 좋음)", loc="left", fontweight="bold")
ax.grid(alpha=0.25)

fig.suptitle("λ 민감도 — 붉은 점선이 채택값, 회색 점선이 비교값",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig(config.H1_FIGDIR / "11_lambda_sensitivity.png", dpi=200, bbox_inches="tight")
plt.show()

**관찰.** 네 기준이 제시하는 최적 λ는 서로 다르며, 채택값(λ=0.5)은 요인 분리(최대 VIF) 기준에 가장 가깝다. λ를 0.2~2.0으로 바꿔도 동시검정 유의 산업 수는 크게 흔들리지 않지만, 개별 계수의 유의성 판정은 λ에 따라 달라지는 산업이 있다.

**3.A.2 관측금리 나비 스프레드로 본 곡률 강건성.** 확인할 내용: λ가 필요 없는 나비 스프레드(2·Δy3−Δy1−Δy5) 곡률이 Nelson–Siegel 곡률과 얼마나 일치하는지.

In [37]:
# 관측금리 나비 스프레드 프록시 (λ 불필요) — 대체가 아니라 NS와 병기해 비교 (교수 §4.2)
extended_panel["bfly_level"] = (
    extended_panel["d_y1"] + extended_panel["d_y3_new"]
    + extended_panel["d_y5"] + extended_panel["d_y10_new"]
) / 4                                                              # 4개 만기 평균 = 수준
extended_panel["bfly_slope"] = extended_panel["d_y10_new"] - extended_panel["d_y1"]        # Δy10 − Δy1 = 기울기
extended_panel["bfly_curvature"] = (
    2 * extended_panel["d_y3_new"] - extended_panel["d_y1"] - extended_panel["d_y5"]        # 2Δy3 − Δy1 − Δy5
)                                                                  # 곡률 (Level·Slope에 정확히 중립)

_fac = extended_panel.drop_duplicates("date")
ns_bfly_corr = pd.DataFrame([
    {"factor": lab, "corr_NS_butterfly": _fac[ns].corr(_fac[bf])}
    for lab, ns, bf in [("Level", "d_ns_level", "bfly_level"),
                        ("Slope", "d_ns_slope", "bfly_slope"),
                        ("Curvature", "d_ns_curvature", "bfly_curvature")]
])
display(ns_bfly_corr.round(4))
ns_bfly_corr.to_csv(config.H1_TABDIR / "ns_vs_butterfly_factor_corr.csv",
                    index=False, encoding="utf-8-sig")

_c = _fac[["d_ns_curvature", "bfly_curvature"]].dropna()
_r = _c["d_ns_curvature"].corr(_c["bfly_curvature"])
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(_c["d_ns_curvature"], _c["bfly_curvature"], s=8, alpha=0.3, color="#334155")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlabel("NS 곡률 변화 (d_ns_curvature)")
ax.set_ylabel("나비 곡률 (2·Δy3 − Δy1 − Δy5)")
ax.set_title(f"NS vs 나비 곡률  (corr = {_r:.3f})")
fig.tight_layout()
fig.savefig(config.H1_FIGDIR / "ns_vs_butterfly_curvature.png", dpi=120, bbox_inches="tight")
plt.show()

,factor,corr_NS_butterfly
0,Level,0.5933
1,Slope,-0.9935
2,Curvature,0.8330


In [38]:
def run_yield_curve_factor_model(data, hac_lags=5, x_cols=None, factor_labels=None):
    """산업별 다요인 동시모형(HAC). x_cols / factor_labels 로 요인 구성을 바꾼다."""
    if x_cols is None:
        x_cols = [
            "kospi_return",
            "d_ns_level",
            "d_ns_slope",
            "d_ns_curvature",
        ]

    if factor_labels is None:
        factor_labels = {
            "d_ns_level": "Level",
            "d_ns_slope": "Slope",
            "d_ns_curvature": "Curvature",
        }

    rows = []

    for industry, group in data.groupby("industry"):
        reg_data = group[
            ["industry_return"] + x_cols
        ].dropna()

        X = sm.add_constant(
            reg_data[x_cols],
            has_constant="add",
        )
        y = reg_data["industry_return"]

        model = sm.OLS(y, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags},
        )

        ci = model.conf_int()

        for variable, factor in factor_labels.items():
            rows.append({
                "industry": industry,
                "factor": factor,
                "variable": variable,
                "coefficient": model.params[variable],
                "std_error_hac": model.bse[variable],
                "t_value": model.tvalues[variable],
                "p_value": model.pvalues[variable],
                "ci_lower_95": ci.loc[variable, 0],
                "ci_upper_95": ci.loc[variable, 1],
                "significant_5pct": model.pvalues[variable] < 0.05,
                "market_beta": model.params["kospi_return"],
                "r_squared": model.rsquared,
                "adj_r_squared": model.rsquared_adj,
                "n_obs": int(model.nobs),
                "hac_lags": hac_lags,
                "optimal_lambda": OPTIMAL_LAMBDA,
            })

    return pd.DataFrame(rows)

yield_curve_factor_betas = run_yield_curve_factor_model(
    extended_panel,
    hac_lags=HAC_LAGS,
)

display(yield_curve_factor_betas.round(6))

,industry,factor,variable,coefficient,std_error_hac,t_value,p_value,ci_lower_95,ci_upper_95,significant_5pct,market_beta,r_squared,adj_r_squared,n_obs,hac_lags,optimal_lambda
0,건설업,Level,d_ns_level,-0.021038,0.019810,-1.061952,0.288258,-0.059866,0.017790,False,1.013279,0.399432,0.398449,2451,5,0.5
1,건설업,Slope,d_ns_slope,-0.027805,0.018091,-1.536940,0.124308,-0.063263,0.007653,False,1.013279,0.399432,0.398449,2451,5,0.5
2,건설업,Curvature,d_ns_curvature,-0.002338,0.003058,-0.764503,0.444568,-0.008330,0.003655,False,1.013279,0.399432,0.398449,2451,5,0.5
3,금융업,Level,d_ns_level,0.017979,0.008254,2.178290,0.029384,0.001802,0.034156,True,0.925754,0.623471,0.622855,2451,5,0.5
4,금융업,Slope,d_ns_slope,0.007189,0.007423,0.968519,0.332785,-0.007359,0.021737,False,0.925754,0.623471,0.622855,2451,5,0.5
5,금융업,Curvature,d_ns_curvature,0.002413,0.001370,1.762013,0.078067,-0.000271,0.005097,False,0.925754,0.623471,0.622855,2451,5,0.5
6,운수장비,Level,d_ns_level,0.013579,0.012917,1.051196,0.293168,-0.011739,0.038896,False,1.006309,0.472321,0.471458,2451,5,0.5
7,운수장비,Slope,d_ns_slope,0.009915,0.012103,0.819256,0.412641,-0.013806,0.033636,False,1.006309,0.472321,0.471458,2451,5,0.5
8,운수장비,Curvature,d_ns_curvature,0.006207,0.002310,2.687189,0.007206,0.001680,0.010735,True,1.006309,0.472321,0.471458,2451,5,0.5
9,의약품,Level,d_ns_level,-0.013817,0.015855,-0.871495,0.383484,-0.044891,0.017257,False,0.789172,0.243525,0.242288,2451,5,0.5


In [39]:
# 산업 수준 병기: 같은 산업 모형을 나비 곡률로 다시 추정해 NS 곡률 결과와 비교
_bf_fit = run_yield_curve_factor_model(
    extended_panel, hac_lags=HAC_LAGS,
    x_cols=["kospi_return", "bfly_level", "bfly_slope", "bfly_curvature"],
    factor_labels={"bfly_level": "Level", "bfly_slope": "Slope", "bfly_curvature": "Curvature"},
)
_ns_curv = (yield_curve_factor_betas[yield_curve_factor_betas["factor"] == "Curvature"]
            [["industry", "coefficient", "p_value"]]
            .rename(columns={"coefficient": "NS_곡률β", "p_value": "NS_p"}))
_bf_curv = (_bf_fit[_bf_fit["factor"] == "Curvature"]
            [["industry", "coefficient", "p_value"]]
            .rename(columns={"coefficient": "나비_곡률β", "p_value": "나비_p"}))
curvature_ns_vs_butterfly = _ns_curv.merge(_bf_curv, on="industry")
curvature_ns_vs_butterfly["부호일치"] = (
    np.sign(curvature_ns_vs_butterfly["NS_곡률β"]) == np.sign(curvature_ns_vs_butterfly["나비_곡률β"])
)
display(curvature_ns_vs_butterfly.round(5))
curvature_ns_vs_butterfly.to_csv(
    config.H1_TABDIR / "curvature_ns_vs_butterfly_by_industry.csv",
    index=False, encoding="utf-8-sig")

,industry,NS_곡률β,NS_p,나비_곡률β,나비_p,부호일치
0,건설업,-0.00234,0.44457,0.00279,0.82073,False
1,금융업,0.00241,0.07807,-0.00078,0.91779,False
2,운수장비,0.00621,0.00721,0.01042,0.30881,True
3,의약품,-0.00860,0.00236,-0.04014,0.00480,True
4,전기전자,0.00144,0.36228,0.00015,0.98287,True
5,철강금속,0.00415,0.14948,0.01624,0.18079,True
6,화학,-0.00008,0.96244,0.00258,0.76441,False


**관찰.** NS 곡률과 나비 곡률의 상관은 매우 높고, 산업별 곡률계수의 부호는 두 방법에서 대부분 일치한다. λ 선택이 곡률의 방향성 결론을 뒤집지는 않는다.

**3.A.3 Credit Spread 단독모형과 확장모형.** 확인할 내용: 신용스프레드를 단독으로 넣었을 때와 3요인에 더했을 때 산업별 계수·유의성이 어떻게 달라지는지.

In [40]:
HAC_LAGS = config.HAC_MAXLAGS   # 팀 공통 설정

def run_single_rate_models(data, hac_lags=5, model_specs=None):
    """산업 x 금리변수별 2지수 개별 회귀(HAC). model_specs 로 변수 조합을 바꾼다."""
    if model_specs is None:
        model_specs = {
            "3Y": "d_y3",
            "10Y": "d_y10",
            "Term_Spread": "d_term_spread",
        }

    rows = []

    for industry, group in data.groupby("industry"):
        for model_name, rate_var in model_specs.items():
            reg_data = group[
                ["industry_return", "kospi_return", rate_var]
            ].dropna()

            X = sm.add_constant(
                reg_data[["kospi_return", rate_var]],
                has_constant="add",
            )
            y = reg_data["industry_return"]

            model = sm.OLS(y, X).fit(
                cov_type="HAC",
                cov_kwds={"maxlags": hac_lags},
            )

            ci = model.conf_int().loc[rate_var]

            rows.append({
                "industry": industry,
                "model": model_name,
                "rate_variable": rate_var,
                "coefficient": model.params[rate_var],
                "std_error_hac": model.bse[rate_var],
                "t_value": model.tvalues[rate_var],
                "p_value": model.pvalues[rate_var],
                "ci_lower_95": ci.iloc[0],
                "ci_upper_95": ci.iloc[1],
                "significant_5pct": model.pvalues[rate_var] < 0.05,
                "market_beta": model.params["kospi_return"],
                "r_squared": model.rsquared,
                "n_obs": int(model.nobs),
                "hac_lags": hac_lags,
            })

    return pd.DataFrame(rows)

maturity_beta_comparison = run_single_rate_models(
    extended_panel,
    hac_lags=HAC_LAGS,
)

display(maturity_beta_comparison.round(6))

,industry,model,rate_variable,coefficient,std_error_hac,t_value,p_value,ci_lower_95,ci_upper_95,significant_5pct,market_beta,r_squared,n_obs,hac_lags
0,건설업,3Y,d_y3,0.003205,0.008341,0.384214,0.700820,-0.013143,0.019553,False,1.015509,0.398237,2451,5
1,건설업,10Y,d_y10,0.006581,0.007948,0.828072,0.407630,-0.008996,0.022159,False,1.015502,0.398420,2451,5
2,건설업,Term_Spread,d_term_spread,0.014193,0.013236,1.072342,0.283566,-0.011748,0.040134,False,1.013508,0.398473,2451,5
3,금융업,3Y,d_y3,0.011234,0.003850,2.917974,0.003523,0.003688,0.018779,True,0.926918,0.622864,2451,5
4,금융업,10Y,d_y10,0.012810,0.004496,2.848881,0.004387,0.003997,0.021623,True,0.925958,0.623396,2451,5
5,금융업,Term_Spread,d_term_spread,0.011024,0.009366,1.177055,0.239174,-0.007333,0.029382,False,0.923707,0.622072,2451,5
6,운수장비,3Y,d_y3,0.017229,0.005809,2.965850,0.003018,0.005843,0.028615,True,1.004995,0.471961,2451,5
7,운수장비,10Y,d_y10,0.010800,0.006045,1.786679,0.073989,-0.001047,0.022647,False,1.002716,0.471024,2451,5
8,운수장비,Term_Spread,d_term_spread,-0.016487,0.012233,-1.347761,0.177735,-0.040462,0.007489,False,1.003351,0.470735,2451,5
9,의약품,3Y,d_y3,-0.027066,0.007831,-3.456104,0.000548,-0.042416,-0.011717,True,0.790549,0.243256,2451,5


In [41]:
# Credit Spread 단독모형.
# model_specs 만 갈아끼우고 함수는 5단계 것을 그대로 쓴다.
credit_spread_single = run_single_rate_models(
    extended_panel,
    hac_lags=HAC_LAGS,
    model_specs={"Credit_Spread": "d_credit_spread"},
)

display(credit_spread_single.round(6))

,industry,model,rate_variable,coefficient,std_error_hac,t_value,p_value,ci_lower_95,ci_upper_95,significant_5pct,market_beta,r_squared,n_obs,hac_lags
0,건설업,Credit_Spread,d_credit_spread,-0.006627,0.033401,-0.198415,0.842721,-0.072092,0.058838,False,1.015018,0.398200,2451,5
1,금융업,Credit_Spread,d_credit_spread,-0.035308,0.020979,-1.682981,0.092379,-0.076427,0.005811,False,0.925408,0.622364,2451,5
2,운수장비,Credit_Spread,d_credit_spread,-0.057150,0.030019,-1.903785,0.056938,-0.115986,0.001686,False,1.002731,0.471308,2451,5
3,의약품,Credit_Spread,d_credit_spread,0.110007,0.041327,2.661862,0.007771,0.029007,0.191007,True,0.793752,0.242992,2451,5
4,전기전자,Credit_Spread,d_credit_spread,-0.001354,0.018067,-0.074965,0.940242,-0.036765,0.034056,False,1.205398,0.719268,2451,5
5,철강금속,Credit_Spread,d_credit_spread,-0.064747,0.028716,-2.254782,0.024147,-0.121029,-0.008466,True,1.023287,0.382886,2451,5
6,화학,Credit_Spread,d_credit_spread,0.018430,0.019754,0.932976,0.350832,-0.020287,0.057147,False,1.053827,0.643730,2451,5


In [42]:
# 확장모형: 기존 3요인 + Credit Spread
CREDIT_X_COLS = [
    "kospi_return",
    "d_ns_level",
    "d_ns_slope",
    "d_ns_curvature",
    "d_credit_spread",
]

CREDIT_FACTOR_LABELS = {
    "d_ns_level": "Level",
    "d_ns_slope": "Slope",
    "d_ns_curvature": "Curvature",
    "d_credit_spread": "CreditSpread",
}

credit_spread_extended = run_yield_curve_factor_model(
    extended_panel,
    hac_lags=HAC_LAGS,
    x_cols=CREDIT_X_COLS,
    factor_labels=CREDIT_FACTOR_LABELS,
)

display(credit_spread_extended.round(6))

,industry,factor,variable,coefficient,std_error_hac,t_value,p_value,ci_lower_95,ci_upper_95,significant_5pct,market_beta,r_squared,adj_r_squared,n_obs,hac_lags,optimal_lambda
0,건설업,Level,d_ns_level,-0.021001,0.020021,-1.048935,0.294208,-0.060241,0.018240,False,1.013283,0.399432,0.398204,2451,5,0.5
1,건설업,Slope,d_ns_slope,-0.027787,0.018219,-1.525172,0.127216,-0.063495,0.007921,False,1.013283,0.399432,0.398204,2451,5,0.5
2,건설업,Curvature,d_ns_curvature,-0.002313,0.003354,-0.689571,0.490464,-0.008888,0.004261,False,1.013283,0.399432,0.398204,2451,5,0.5
3,건설업,CreditSpread,d_credit_spread,0.000816,0.036832,0.022153,0.982326,-0.071374,0.073006,False,1.013283,0.399432,0.398204,2451,5,0.5
4,금융업,Level,d_ns_level,0.016651,0.008259,2.016157,0.043784,0.000464,0.032838,True,0.925611,0.623810,0.623041,2451,5,0.5
5,금융업,Slope,d_ns_slope,0.006531,0.007489,0.872113,0.383147,-0.008147,0.021209,False,0.925611,0.623810,0.623041,2451,5,0.5
6,금융업,Curvature,d_ns_curvature,0.001543,0.001584,0.974326,0.329895,-0.001561,0.004648,False,0.925611,0.623810,0.623041,2451,5,0.5
7,금융업,CreditSpread,d_credit_spread,-0.029060,0.024413,-1.190334,0.233915,-0.076908,0.018789,False,0.925611,0.623810,0.623041,2451,5,0.5
8,운수장비,Level,d_ns_level,0.012290,0.012796,0.960495,0.336806,-0.012789,0.037370,False,1.006170,0.472527,0.471448,2451,5,0.5
9,운수장비,Slope,d_ns_slope,0.009277,0.012030,0.771116,0.440638,-0.014302,0.032855,False,1.006170,0.472527,0.471448,2451,5,0.5


In [43]:
# BH(FDR) 보정 공통 함수 — 14단계뿐 아니라 10단계 핵심 결과표에서도 쓰므로 여기서 한 번만 정의한다.
def add_bh_correction(result, group_cols=("model", "factor"),
                      p_col="p_value", alpha=0.05):
    """검정족별 BH(FDR) 보정 p값을 붙인다. 원 p_value 는 그대로 둔다."""
    out = result.copy()
    group_cols = list(group_cols)

    out["p_value_bh"] = np.nan
    out["bh_n_tests"] = 0

    for keys, idx in out.groupby(group_cols).groups.items():
        pvals = out.loc[idx, p_col].to_numpy(dtype=float)
        out.loc[idx, "p_value_bh"] = multipletests(pvals, method="fdr_bh")[1]
        out.loc[idx, "bh_n_tests"] = len(pvals)

    # 어떤 단위로 보정했는지 표에 남겨 두어야 나중에 재검토가 가능하다.
    out["bh_family"] = out[group_cols].astype(str).agg(" | ".join, axis=1)
    out["significant_5pct_bh"] = out["p_value_bh"] < alpha

    return out

In [44]:
# 단독모형과 확장모형의 컬럼 구성을 맞춘 뒤 하나의 표로 합친다.
# (단독모형은 rate_variable, 확장모형은 variable/factor 를 쓴다)
credit_single_norm = (
    credit_spread_single
    .rename(columns={"rate_variable": "variable"})
    .assign(
        model="A_Credit_Single",
        factor="CreditSpread",
        adj_r_squared=np.nan,
        optimal_lambda=np.nan,
    )
)

credit_extended_norm = credit_spread_extended.assign(model="B_Extended_4Factor")

CREDIT_TABLE_COLS = [
    "model",
    "industry",
    "factor",
    "variable",
    "coefficient",
    "std_error_hac",
    "t_value",
    "p_value",
    "p_value_bh",
    "bh_family",
    "bh_n_tests",
    "ci_lower_95",
    "ci_upper_95",
    "significant_5pct",
    "significant_5pct_bh",
    "market_beta",
    "r_squared",
    "adj_r_squared",
    "n_obs",
    "hac_lags",
    "optimal_lambda",
]

credit_spread_betas = (
    pd.concat([credit_single_norm, credit_extended_norm], ignore_index=True)
    .pipe(add_bh_correction, group_cols=("model", "factor"))
    .sort_values(["model", "factor", "industry"])
    .reset_index(drop=True)
    .reindex(columns=CREDIT_TABLE_COLS)
)

credit_path = config.H1_TABDIR / "credit_spread_betas.csv"

credit_spread_betas.to_csv(
    credit_path,
    index=False,
    encoding="utf-8-sig",
)

display(credit_spread_betas.round(6))

,model,industry,factor,variable,coefficient,std_error_hac,t_value,p_value,p_value_bh,bh_family,bh_n_tests,ci_lower_95,ci_upper_95,significant_5pct,significant_5pct_bh,market_beta,r_squared,adj_r_squared,n_obs,hac_lags,optimal_lambda
0,A_Credit_Single,건설업,CreditSpread,d_credit_spread,-0.006627,0.033401,-0.198415,0.842721,0.940242,A_Credit_Single | CreditSpread,7,-0.072092,0.058838,False,False,1.015018,0.398200,NaN,2451,5,NaN
1,A_Credit_Single,금융업,CreditSpread,d_credit_spread,-0.035308,0.020979,-1.682981,0.092379,0.161663,A_Credit_Single | CreditSpread,7,-0.076427,0.005811,False,False,0.925408,0.622364,NaN,2451,5,NaN
2,A_Credit_Single,운수장비,CreditSpread,d_credit_spread,-0.057150,0.030019,-1.903785,0.056938,0.132856,A_Credit_Single | CreditSpread,7,-0.115986,0.001686,False,False,1.002731,0.471308,NaN,2451,5,NaN
3,A_Credit_Single,의약품,CreditSpread,d_credit_spread,0.110007,0.041327,2.661862,0.007771,0.054397,A_Credit_Single | CreditSpread,7,0.029007,0.191007,True,False,0.793752,0.242992,NaN,2451,5,NaN
4,A_Credit_Single,전기전자,CreditSpread,d_credit_spread,-0.001354,0.018067,-0.074965,0.940242,0.940242,A_Credit_Single | CreditSpread,7,-0.036765,0.034056,False,False,1.205398,0.719268,NaN,2451,5,NaN
5,A_Credit_Single,철강금속,CreditSpread,d_credit_spread,-0.064747,0.028716,-2.254782,0.024147,0.084515,A_Credit_Single | CreditSpread,7,-0.121029,-0.008466,True,False,1.023287,0.382886,NaN,2451,5,NaN
6,A_Credit_Single,화학,CreditSpread,d_credit_spread,0.018430,0.019754,0.932976,0.350832,0.491165,A_Credit_Single | CreditSpread,7,-0.020287,0.057147,False,False,1.053827,0.643730,NaN,2451,5,NaN
7,B_Extended_4Factor,건설업,CreditSpread,d_credit_spread,0.000816,0.036832,0.022153,0.982326,0.982326,B_Extended_4Factor | CreditSpread,7,-0.071374,0.073006,False,False,1.013283,0.399432,0.398204,2451,5,0.5
8,B_Extended_4Factor,금융업,CreditSpread,d_credit_spread,-0.029060,0.024413,-1.190334,0.233915,0.409352,B_Extended_4Factor | CreditSpread,7,-0.076908,0.018789,False,False,0.925611,0.623810,0.623041,2451,5,0.5
9,B_Extended_4Factor,운수장비,CreditSpread,d_credit_spread,-0.028199,0.032764,-0.860672,0.389419,0.545186,B_Extended_4Factor | CreditSpread,7,-0.092415,0.036017,False,False,1.006170,0.472527,0.471448,2451,5,0.5


In [45]:
# 확장모형(4요인) 계수와 95% 신뢰구간을 2x2 패널로 그린다.
# 스타일(errorbar, capsize, 0선, 정렬)은 12단계 그림과 동일하게 유지한다.
credit_factor_order = ["Level", "Slope", "Curvature", "CreditSpread"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# BH 보정 컬럼이 붙은 credit_spread_betas 를 쓴다(credit_spread_extended 는 보정 전 원본).
_extended_bh = credit_spread_betas[credit_spread_betas["model"] == "B_Extended_4Factor"]

for ax, factor in zip(axes.ravel(), credit_factor_order):
    plot_data = (
        _extended_bh[_extended_bh["factor"] == factor]
        .sort_values("coefficient")
        .reset_index(drop=True)
    )

    y_pos = np.arange(len(plot_data))

    lower_error = plot_data["coefficient"] - plot_data["ci_lower_95"]
    upper_error = plot_data["ci_upper_95"] - plot_data["coefficient"]

    # 색상은 계수 부호를 나타낸다. 유의 판정은 BH 보정 후 q < 0.05 기준이다.
    # 오차막대는 보정하지 않은 95% 신뢰구간이므로, 구간이 0을 벗어나도 회색일 수 있다.
    point_colors = [
        config.sign_color(coef, sig)
        for coef, sig in zip(plot_data["coefficient"], plot_data["significant_5pct_bh"])
    ]

    for y, coef, lo, hi, color in zip(
        y_pos, plot_data["coefficient"], lower_error, upper_error, point_colors,
    ):
        ax.errorbar(
            coef, y,
            xerr=[[lo], [hi]],
            fmt="o",
            capsize=4,
            linewidth=1.8,
            markersize=7,
            color=color,
            ecolor=color,
        )

    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_data["industry"])
    ax.set_xlabel(f"{factor} 변화에 대한 산업수익률 반응계수")
    ax.set_title(f"{factor}", loc="left", fontweight="bold")
    ax.grid(axis="x", alpha=0.25)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           color=config.SIGN_COLORS["neg"], label="유의한 음(\u2212)"),
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           color=config.SIGN_COLORS["pos"], label="유의한 양(+)"),
    Line2D([0], [0], marker="o", linestyle="None", markersize=8,
           color=config.SIGN_COLORS["ns"], label="비유의"),
]
fig.legend(handles=legend_handles, loc="upper right", frameon=True, fontsize=9,
           bbox_to_anchor=(0.99, 0.99))

fig.suptitle(
    "Credit Spread 포함 확장모형 — 산업별 요인 민감도\n"
    "(색 = BH 보정 후 5% 판정, family = 요인별 7개 산업 / 막대 = 보정 전 95% CI, HAC)",
    x=0.01,
    ha="left",
    fontweight="bold",
)
fig.tight_layout()

credit_fig_path = config.H1_FIGDIR / "04_yield_curve_factor_ci.png"
fig.savefig(credit_fig_path, dpi=200, bbox_inches="tight")
plt.show()


**관찰.** Credit Spread를 추가해도 Level·Slope·Curvature 계수의 부호와 유의성 판정은 크게 바뀌지 않는다. Credit Spread 자체의 계수는 일부 산업에서만 BH 5%를 충족한다.

**3.A.4 수익률곡선의 차원과 모형 절약성(parsimony).** 확인할 내용: 국고채 1·3·5·10년 변화분의 PCA로 곡선이 몇 차원인지, 3요인 모형이 단일금리 모형보다 조정 R²를 실제로 높이는지.

In [46]:
# 국고채 1·3·5·10년 '변화분'의 주성분분석.
# 수준이 아니라 변화분을 쓰는 이유는 회귀에서 쓰는 변수가 변화분이기 때문이다.
YIELD_CHANGE_COLS = ["d_y1", "d_y3_new", "d_y5", "d_y10_new"]
MATURITY_LABELS = ["1년", "3년", "5년", "10년"]

# 설명변수는 모든 산업이 공유하므로 날짜 단위로 한 번씩만 쓴다.
yield_changes = (
    extended_panel[["date"] + YIELD_CHANGE_COLS]
    .drop_duplicates(subset="date")
    .dropna()
    .sort_values("date")
    .reset_index(drop=True)
)

Z = yield_changes[YIELD_CHANGE_COLS].to_numpy(dtype=float)
Z_centered = Z - Z.mean(axis=0)

# 공분산행렬 고유분해. eigh 는 오름차순이므로 뒤집어 내림차순으로 만든다.
eigenvalues, eigenvectors = np.linalg.eigh(np.cov(Z_centered, rowvar=False))
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

explained = eigenvalues / eigenvalues.sum()

yield_pca = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(eigenvalues))],
    "eigenvalue": eigenvalues,
    "explained_ratio": explained,
    "cumulative_ratio": np.cumsum(explained),
})

# 각 만기가 성분에 얼마나 실리는지(적재값)를 같은 표에 붙인다.
for j, label in enumerate(MATURITY_LABELS):
    yield_pca[f"loading_{label}"] = eigenvectors[j, :]

# 부호는 고유벡터에서 임의로 결정되므로, PC1 이 양(+)이 되도록 통일한다.
for i in range(len(eigenvalues)):
    load = eigenvectors[:, i]
    if load.sum() < 0 if i == 0 else load[0] < 0:
        for j, label in enumerate(MATURITY_LABELS):
            yield_pca.loc[i, f"loading_{label}"] *= -1

yield_pca["n_obs"] = len(yield_changes)

pca_path = config.H1_TABDIR / "exploratory_yield_pca.csv"
yield_pca.to_csv(pca_path, index=False, encoding="utf-8-sig")

print(f"PC1 설명비중: {explained[0]*100:.2f}%   PC1+PC2: {explained[:2].sum()*100:.2f}%")
display(yield_pca.round(5))

PC1 설명비중: 92.09%   PC1+PC2: 96.88%


,component,eigenvalue,explained_ratio,cumulative_ratio,loading_1년,loading_3년,loading_5년,loading_10년,n_obs
0,PC1,0.00492,0.92086,0.92086,0.23634,0.53339,0.58489,0.56352,2451
1,PC2,0.00026,0.04795,0.96881,0.45814,0.53572,0.00965,-0.70923,2451
2,PC3,0.00012,0.02239,0.99120,0.85263,-0.38260,-0.24462,0.25845,2451
3,PC4,0.00005,0.00880,1.00000,0.08525,-0.53115,0.77329,-0.33562,2451


In [47]:
def compare_model_parsimony(data, base_vars, full_vars, control="kospi_return"):
    """단순모형과 확장모형의 조정 결정계수를 산업별로 비교한다."""
    rows = []

    for industry, group in data.groupby("industry"):
        need = ["industry_return", control] + list(base_vars) + list(full_vars)
        d = group[list(dict.fromkeys(need))].dropna()
        y = d["industry_return"]

        m_base = sm.OLS(y, sm.add_constant(d[[control] + list(base_vars)],
                                           has_constant="add")).fit()
        m_full = sm.OLS(y, sm.add_constant(d[[control] + list(full_vars)],
                                           has_constant="add")).fit()

        rows.append({
            "industry": industry,
            "adj_r2_single_rate": m_base.rsquared_adj,
            "adj_r2_three_factor": m_full.rsquared_adj,
            "delta_adj_r2": m_full.rsquared_adj - m_base.rsquared_adj,
            "improved": bool(m_full.rsquared_adj > m_base.rsquared_adj),
            "n_obs": int(m_base.nobs),
        })

    return pd.DataFrame(rows).sort_values("delta_adj_r2", ascending=False).reset_index(drop=True)


model_parsimony = compare_model_parsimony(
    extended_panel,
    base_vars=["d_y3"],
    full_vars=["d_ns_level", "d_ns_slope", "d_ns_curvature"],
)

parsimony_path = config.H1_TABDIR / "exploratory_model_parsimony.csv"
model_parsimony.to_csv(parsimony_path, index=False, encoding="utf-8-sig")

print("설명력이 개선된 산업:", int(model_parsimony["improved"].sum()), "/ 7")
print("평균 조정 R² 증가폭:", round(model_parsimony["delta_adj_r2"].mean(), 6))
display(model_parsimony.round(6))

설명력이 개선된 산업: 2 / 7
평균 조정 R² 증가폭: -2.7e-05


,industry,adj_r2_single_rate,adj_r2_three_factor,delta_adj_r2,improved,n_obs
0,건설업,0.397745,0.398449,0.000704,True,2451
1,금융업,0.622556,0.622855,0.000299,True,2451
2,운수장비,0.471529,0.471458,-0.000071,False,2451
3,전기전자,0.719054,0.718931,-0.000122,False,2451
4,화학,0.643351,0.643162,-0.000189,False,2451
5,의약품,0.242638,0.242288,-0.000349,False,2451
6,철강금속,0.382919,0.382460,-0.000459,False,2451


In [48]:
# 왼쪽: 각 주성분의 설명비중과 누적선
# 오른쪽: 성분별 적재값. PC1 이 네 만기에서 같은 부호면 '평행이동'이다.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x_pos = np.arange(len(yield_pca))
ax.bar(x_pos, yield_pca["explained_ratio"] * 100)
for i, v in enumerate(yield_pca["explained_ratio"] * 100):
    ax.text(i, v + 1.5, f"{v:.1f}%", ha="center", fontsize=10, fontweight="bold")

ax_cum = ax.twinx()
ax_cum.plot(x_pos, yield_pca["cumulative_ratio"] * 100,
            marker="o", linewidth=1.5, color="crimson")
ax_cum.set_ylim(0, 105)
ax_cum.set_ylabel("누적 설명비중 (%)", color="crimson")

ax.set_xticks(x_pos)
ax.set_xticklabels(yield_pca["component"])
ax.set_ylim(0, 105)
ax.set_ylabel("설명비중 (%)")
ax.set_title("국고채 일별 변화의 주성분 설명비중", loc="left", fontweight="bold")

ax = axes[1]
for i in range(3):
    ax.plot(MATURITY_LABELS,
            [yield_pca.loc[i, f"loading_{m}"] for m in MATURITY_LABELS],
            marker="o", linewidth=1.8,
            label=f"PC{i+1} ({yield_pca.loc[i, 'explained_ratio']*100:.1f}%)")

ax.axhline(0, linestyle="--", linewidth=1, color="gray")
ax.set_xlabel("만기")
ax.set_ylabel("적재값")
ax.set_title("성분별 만기 적재값\n(PC1이 전 만기에서 같은 부호 → 평행이동)",
             loc="left", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

fig.tight_layout()

pca_fig_path = config.H1_FIGDIR / "08_yield_curve_pca.png"
fig.savefig(pca_fig_path, dpi=200, bbox_inches="tight")
plt.show()


In [49]:
# 3년물이 곡선의 지배적 변동(PC1)을 얼마나 잘 대표하는지 확인한다.
# 상관이 높다면 '3년물 하나만 쓴 것'은 임의 선택이 아니라 효율적 선택이 된다.
pc1_score = Z_centered @ eigenvectors[:, 0]

proxy_check = pd.DataFrame({
    "date": yield_changes["date"],
    "pc1": pc1_score,
}).merge(
    extended_panel[["date", "d_y3", "d_ns_level", "d_ns_slope", "d_ns_curvature"]]
    .drop_duplicates(subset="date"),
    on="date",
    how="inner",
)

pc1_corr = (
    proxy_check[["pc1", "d_y3", "d_ns_level", "d_ns_slope", "d_ns_curvature"]]
    .corr()["pc1"]
    .drop("pc1")
    .abs()
    .sort_values(ascending=False)
    .rename("abs_corr_with_pc1")
    .reset_index()
    .rename(columns={"index": "variable"})
)

print("PC1(지배적 곡선변동)과의 절대상관 — 높을수록 좋은 대리변수")
display(pc1_corr.round(4))

PC1(지배적 곡선변동)과의 절대상관 — 높을수록 좋은 대리변수


,variable,abs_corr_with_pc1
0,d_y3,0.9647
1,d_ns_slope,0.7197
2,d_ns_level,0.6036
3,d_ns_curvature,0.6030


**관찰.** PC1이 설명비중 대부분을 차지해 곡선 변동은 사실상 1차원(평행이동)에 가깝다. 3요인을 추가하면 조정 R²는 대부분 산업에서 개선되지만 폭은 크지 않다.

**3.A.5 Rolling PCA와 정책국면별 곡선구조.** 확인할 내용: 곡선의 1차원성이 표본 전체에서 안정적인 구조인지, 아니면 정책금리 사이클에 따라 달라지는지(504영업일 이동창).

In [50]:
VIF_VARS = [
    "kospi_return",
    "d_ns_level",
    "d_ns_slope",
    "d_ns_curvature",
    "d_credit_spread",
]

# 설명변수는 모든 산업이 공유하므로 날짜 기준으로 한 번씩만 남긴다.
factor_daily = (
    extended_panel[["date"] + VIF_VARS]
    .drop_duplicates(subset="date")
    .dropna()
    .sort_values("date")
    .reset_index(drop=True)
)

corr_matrix = factor_daily[VIF_VARS].corr()

# VIF 는 상수항을 포함한 설계행렬에서 계산해야 값이 왜곡되지 않는다.
vif_design = sm.add_constant(factor_daily[VIF_VARS], has_constant="add")

vif_values = {
    var: variance_inflation_factor(vif_design.to_numpy(dtype=float), i)
    for i, var in enumerate(vif_design.columns)
    if var != "const"
}

factor_correlation_vif = (
    corr_matrix
    .rename(columns=lambda c: f"corr_{c}")
    .reset_index()
    .rename(columns={"index": "variable"})
    .assign(
        vif=lambda d: d["variable"].map(vif_values),
        n_obs=len(factor_daily),
    )
)

# variable / vif / n_obs 를 앞으로 빼서 읽기 쉽게 한다.
factor_correlation_vif = factor_correlation_vif[
    ["variable", "vif", "n_obs"]
    + [c for c in factor_correlation_vif.columns if c.startswith("corr_")]
]

vif_path = config.H1_TABDIR / "factor_correlation_vif.csv"

factor_correlation_vif.to_csv(
    vif_path,
    index=False,
    encoding="utf-8-sig",
)

print("VIF 5 초과 변수:", [v for v, x in vif_values.items() if x > 5])
display(factor_correlation_vif.round(4))

VIF 5 초과 변수: ['d_ns_level', 'd_ns_slope']


,variable,vif,n_obs,corr_kospi_return,corr_d_ns_level,corr_d_ns_slope,corr_d_ns_curvature,corr_d_credit_spread
0,kospi_return,1.0061,2451,1.0000,0.0175,-0.0101,-0.0615,0.0212
1,d_ns_level,7.2545,2451,0.0175,1.0000,-0.8429,-0.2360,-0.0202
2,d_ns_slope,7.0136,2451,-0.0101,-0.8429,1.0000,-0.1727,0.1627
3,d_ns_curvature,2.4474,2451,-0.0615,-0.2360,-0.1727,1.0000,-0.4112
4,d_credit_spread,1.2284,2451,0.0212,-0.0202,0.1627,-0.4112,1.0000


In [51]:
# ---------------------------------------------------------------
# D-0-1. Rolling PCA / 정책금리 사이클 진단 공통 설정
# ---------------------------------------------------------------

ROLLING_WINDOW = 504  # 약 2년 영업일

# 정책국면은 분석 결과가 아니라 한국은행 기준금리 방향 전환일을 기준으로 정의한다.
POLICY_CYCLES = pd.DataFrame([
    {
        "cycle": 1,
        "policy_cycle": "저금리·완화 국면",
        "start": "2016-01-04",
        "end": "2017-11-29",
    },
    {
        "cycle": 2,
        "policy_cycle": "인상·정상화 사이클",
        "start": "2017-11-30",
        "end": "2019-07-17",
    },
    {
        "cycle": 3,
        "policy_cycle": "인하·초저금리 사이클",
        "start": "2019-07-18",
        "end": "2021-08-25",
    },
    {
        "cycle": 4,
        "policy_cycle": "인상·고금리 유지 사이클",
        "start": "2021-08-26",
        "end": "2024-10-10",
    },
    {
        "cycle": 5,
        "policy_cycle": "인하 재개 사이클",
        "start": "2024-10-11",
        "end": "2025-12-30",
    },
])

POLICY_CYCLES["start"] = pd.to_datetime(
    POLICY_CYCLES["start"],
    format="%Y-%m-%d",
    errors="raise",
)

POLICY_CYCLES["end"] = pd.to_datetime(
    POLICY_CYCLES["end"],
    format="%Y-%m-%d",
    errors="raise",
)

# 기존 15단계 VIF 계산과 동일한 사양을 유지한다.
# 그래야 전체 표본 VIF 7.25 / 7.01과 직접 비교할 수 있다.
ROLLING_VIF_VARS = VIF_VARS.copy()

# 관심 대상은 수익률곡선 세 요인의 VIF.
CURVE_VIF_TARGETS = [
    "d_ns_level",
    "d_ns_slope",
    "d_ns_curvature",
]


def classify_policy_cycle(date):
    """날짜를 사전 정의된 정책금리 사이클에 배정한다."""
    for row in POLICY_CYCLES.itertuples(index=False):
        if row.start <= date <= row.end:
            return row.cycle, row.policy_cycle

    return pd.NA, pd.NA


def compute_pca_diagnostics(data):
    """
    국고채 1·3·5·10년 변화분의 PCA 진단값.

    전체 표본 D-0과 동일하게:
    - 금리 수준이 아니라 변화분 사용
    - 표준화하지 않고 공분산행렬 사용
    """
    z = data[YIELD_CHANGE_COLS].to_numpy(dtype=float)
    z_centered = z - z.mean(axis=0)

    eigenvalues, eigenvectors = np.linalg.eigh(
        np.cov(z_centered, rowvar=False)
    )

    # 내림차순 정렬
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    explained = eigenvalues / eigenvalues.sum()

    # PC1 부호는 임의적이므로 기존 D-0과 동일하게 양(+) 방향으로 통일
    pc1_loading = eigenvectors[:, 0].copy()

    if pc1_loading.sum() < 0:
        pc1_loading *= -1

    return {
        "pc1_share": float(explained[0]),
        "pc1_pc2_share": float(explained[:2].sum()),
        "pc1_loading_1y": float(pc1_loading[0]),
        "pc1_loading_3y": float(pc1_loading[1]),
        "pc1_loading_5y": float(pc1_loading[2]),
        "pc1_loading_10y": float(pc1_loading[3]),
        "pc1_same_sign": bool(
            np.all(pc1_loading > 0)
            or np.all(pc1_loading < 0)
        ),
    }


def compute_vif_diagnostics(data):
    """
    기존 15단계와 같은 설명변수 사양으로 VIF를 계산한다.
    """
    design = sm.add_constant(
        data[ROLLING_VIF_VARS],
        has_constant="add",
    )

    values = {
        var: variance_inflation_factor(
            design.to_numpy(dtype=float),
            i,
        )
        for i, var in enumerate(design.columns)
        if var != "const"
    }

    return {
        "vif_level": float(values["d_ns_level"]),
        "vif_slope": float(values["d_ns_slope"]),
        "vif_curvature": float(values["d_ns_curvature"]),
        "max_curve_vif": float(
            max(values[var] for var in CURVE_VIF_TARGETS)
        ),
    }


# 설명변수는 산업마다 반복되므로 날짜별 한 번만 사용한다.
DIAG_COLS = list(dict.fromkeys(
    ["date"]
    + YIELD_CHANGE_COLS
    + ROLLING_VIF_VARS
))

diagnostic_daily = (
    extended_panel[DIAG_COLS]
    .drop_duplicates(subset="date")
    .dropna()
    .sort_values("date")
    .reset_index(drop=True)
)

print("진단용 일별 관측치:", len(diagnostic_daily))
print("Rolling window:", ROLLING_WINDOW, "영업일")

진단용 일별 관측치: 2451
Rolling window: 504 영업일


In [52]:
# ---------------------------------------------------------------
# 504영업일 Rolling PCA / L-S 상관 / VIF
# ---------------------------------------------------------------

rolling_rows = []

for end_idx in range(
    ROLLING_WINDOW,
    len(diagnostic_daily) + 1,
):
    window = diagnostic_daily.iloc[
        end_idx - ROLLING_WINDOW:end_idx
    ].copy()

    pca_stats = compute_pca_diagnostics(window)
    vif_stats = compute_vif_diagnostics(window)

    # 정책국면별 rolling 평균을 만들 때는 종료일보다
    # window 중앙일을 기준으로 국면을 부여한다.
    mid_date = window["date"].iloc[len(window) // 2]

    cycle_no, cycle_name = classify_policy_cycle(mid_date)

    rolling_rows.append({
        "window_start": window["date"].iloc[0],
        "window_mid": mid_date,
        "window_end": window["date"].iloc[-1],

        "cycle_mid": cycle_no,
        "policy_cycle_mid": cycle_name,

        "n_obs": len(window),

        # Level-Slope 상관
        "ls_corr": float(
            window["d_ns_level"].corr(
                window["d_ns_slope"]
            )
        ),

        **pca_stats,
        **vif_stats,
    })


rolling_curve_structure = pd.DataFrame(rolling_rows)


# config 규정: 결과 폴더를 직접 만들지 않고 H1_TABDIR 사용
rolling_path = (
    config.H1_TABDIR
    / "rolling_curve_structure.csv"
)

rolling_curve_structure.to_csv(
    rolling_path,
    index=False,
    encoding="utf-8-sig",
)

print("Rolling window 수:", len(rolling_curve_structure))
display(rolling_curve_structure.head().round(4))

Rolling window 수: 1948


,window_start,window_mid,window_end,cycle_mid,policy_cycle_mid,n_obs,ls_corr,pc1_share,pc1_pc2_share,pc1_loading_1y,pc1_loading_3y,pc1_loading_5y,pc1_loading_10y,pc1_same_sign,vif_level,vif_slope,vif_curvature,max_curve_vif
0,2016-01-05,2017-01-11,2018-01-23,1,저금리·완화 국면,504,-0.9327,0.9140,0.9738,0.2466,0.4569,0.5744,0.6328,True,13.5355,11.6337,2.7047,13.5355
1,2016-01-06,2017-01-12,2018-01-24,1,저금리·완화 국면,504,-0.9328,0.9140,0.9738,0.2468,0.4569,0.5744,0.6328,True,13.5693,11.6615,2.7044,13.5693
2,2016-01-07,2017-01-13,2018-01-25,1,저금리·완화 국면,504,-0.9325,0.9141,0.9738,0.2467,0.4571,0.5746,0.6325,True,13.5202,11.6236,2.7032,13.5202
3,2016-01-08,2017-01-16,2018-01-26,1,저금리·완화 국면,504,-0.9325,0.9142,0.9738,0.2466,0.4572,0.5746,0.6324,True,13.4959,11.6106,2.6980,13.4959
4,2016-01-11,2017-01-17,2018-01-29,1,저금리·완화 국면,504,-0.9325,0.9148,0.9740,0.2472,0.4581,0.5749,0.6313,True,13.4578,11.6057,2.6924,13.4578


In [53]:
# ---------------------------------------------------------------
# Rolling 결과의 정책국면별 평균 — 보조 진단
# ---------------------------------------------------------------

rolling_policy_summary = (
    rolling_curve_structure
    .dropna(subset=["cycle_mid"])
    .groupby(
        ["cycle_mid", "policy_cycle_mid"],
        as_index=False,
    )
    .agg(
        n_windows=("pc1_share", "size"),

        mean_pc1_share=("pc1_share", "mean"),
        mean_pc1_pc2_share=("pc1_pc2_share", "mean"),

        mean_ls_corr=("ls_corr", "mean"),

        mean_max_curve_vif=(
            "max_curve_vif",
            "mean",
        ),
    )
)


rolling_policy_summary_path = (
    config.H1_TABDIR
    / "rolling_policy_cycle_summary.csv"
)

rolling_policy_summary.to_csv(
    rolling_policy_summary_path,
    index=False,
    encoding="utf-8-sig",
)

display(rolling_policy_summary.round(4))

,cycle_mid,policy_cycle_mid,n_windows,mean_pc1_share,mean_pc1_pc2_share,mean_ls_corr,mean_max_curve_vif
0,1,저금리·완화 국면,216,0.9110,0.9732,-0.9300,15.1769
1,2,인상·정상화 사이클,398,0.9227,0.9742,-0.9101,11.0101
2,3,인하·초저금리 사이클,522,0.9018,0.9655,-0.8858,12.5796
3,4,인상·고금리 유지 사이클,765,0.9335,0.9722,-0.8066,7.8734
4,5,인하 재개 사이클,47,0.9392,0.9774,-0.8987,12.6844


In [54]:
# ---------------------------------------------------------------
# 정책금리 사이클별 직접 진단
# ---------------------------------------------------------------

policy_curve_rows = []
policy_model_rows = []


for row in POLICY_CYCLES.itertuples(index=False):

    # --------------------------------
    # 1. 금리곡선 구조 진단
    # --------------------------------
    daily_sub = diagnostic_daily.loc[
        diagnostic_daily["date"].between(
            row.start,
            row.end,
        )
    ].copy()

    if len(daily_sub) < 10:
        continue

    pca_stats = compute_pca_diagnostics(daily_sub)
    vif_stats = compute_vif_diagnostics(daily_sub)

    policy_curve_rows.append({
        "cycle": row.cycle,
        "policy_cycle": row.policy_cycle,
        "start": row.start,
        "end": row.end,
        "n_dates": len(daily_sub),

        "ls_corr": float(
            daily_sub["d_ns_level"].corr(
                daily_sub["d_ns_slope"]
            )
        ),

        **pca_stats,
        **vif_stats,
    })


    # --------------------------------
    # 2. 산업수익률 설명력 비교
    # --------------------------------
    panel_sub = extended_panel.loc[
        extended_panel["date"].between(
            row.start,
            row.end,
        )
    ].copy()

    # 기존 D-0에서 만든 함수를 그대로 재사용
    parsimony_sub = compare_model_parsimony(
        panel_sub,
        base_vars=["d_y3"],
        full_vars=[
            "d_ns_level",
            "d_ns_slope",
            "d_ns_curvature",
        ],
    )

    policy_model_rows.append({
        "cycle": row.cycle,
        "policy_cycle": row.policy_cycle,

        "n_industries": int(
            parsimony_sub["industry"].nunique()
        ),

        "mean_adj_r2_single_rate": float(
            parsimony_sub[
                "adj_r2_single_rate"
            ].mean()
        ),

        "mean_adj_r2_three_factor": float(
            parsimony_sub[
                "adj_r2_three_factor"
            ].mean()
        ),

        "mean_delta_adj_r2": float(
            parsimony_sub[
                "delta_adj_r2"
            ].mean()
        ),

        "improved_industries": int(
            parsimony_sub["improved"].sum()
        ),
    })


policy_curve_structure = pd.DataFrame(
    policy_curve_rows
)

policy_model_parsimony = pd.DataFrame(
    policy_model_rows
)


# 두 결과를 하나의 정책국면 표로 결합
policy_cycle_diagnostics = (
    policy_curve_structure
    .merge(
        policy_model_parsimony,
        on=["cycle", "policy_cycle"],
        how="left",
        validate="one_to_one",
    )
    .sort_values("cycle")
    .reset_index(drop=True)
)


policy_cycle_path = (
    config.H1_TABDIR
    / "policy_cycle_curve_structure.csv"
)

policy_cycle_diagnostics.to_csv(
    policy_cycle_path,
    index=False,
    encoding="utf-8-sig",
)


display_cols = [
    "cycle",
    "policy_cycle",
    "n_dates",

    "pc1_share",
    "pc1_pc2_share",

    "ls_corr",

    "vif_level",
    "vif_slope",
    "max_curve_vif",

    "mean_adj_r2_single_rate",
    "mean_adj_r2_three_factor",
    "mean_delta_adj_r2",

    "improved_industries",
]

display(
    policy_cycle_diagnostics[
        display_cols
    ].round(4)
)

,cycle,policy_cycle,n_dates,pc1_share,pc1_pc2_share,ls_corr,vif_level,vif_slope,max_curve_vif,mean_adj_r2_single_rate,mean_adj_r2_three_factor,mean_delta_adj_r2,improved_industries
0,1,저금리·완화 국면,468,0.9201,0.9751,-0.9336,13.5744,11.7597,13.5744,0.3278,0.3284,0.0006,3
1,2,인상·정상화 사이클,398,0.9187,0.9739,-0.9192,11.8108,10.7106,11.8108,0.4098,0.4094,-0.0003,3
2,3,인하·초저금리 사이클,522,0.8939,0.9647,-0.9088,13.9964,11.1960,13.9964,0.6336,0.6333,-0.0002,2
3,4,인상·고금리 유지 사이클,765,0.9316,0.9708,-0.7833,5.0977,5.5515,5.5515,0.5208,0.5203,-0.0005,1
4,5,인하 재개 사이클,298,0.9107,0.9696,-0.8995,15.2033,12.9959,15.2033,0.4546,0.4529,-0.0017,2


In [55]:
# ---------------------------------------------------------------
# 504영업일 Rolling PC1 시계열
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 5.5))


ax.plot(
    rolling_curve_structure["window_end"],
    rolling_curve_structure["pc1_share"] * 100,
    linewidth=1.6,
    label=f"{ROLLING_WINDOW}영업일 Rolling PC1",
)


# 전체 표본의 PC1 = 기존 D-0 결과
full_sample_pc1 = float(
    yield_pca.loc[0, "explained_ratio"] * 100
)

ax.axhline(
    full_sample_pc1,
    linestyle="--",
    linewidth=1.2,
    label=(
        f"전체 표본 PC1 = "
        f"{full_sample_pc1:.1f}%"
    ),
)


# 정책금리 사이클 시작점 표시
for row in POLICY_CYCLES.itertuples(index=False):
    ax.axvline(
        row.start,
        linestyle=":",
        linewidth=1,
        alpha=0.55,
    )


# 정책국면 명칭 표시
y_top = ax.get_ylim()[1]

for row in POLICY_CYCLES.itertuples(index=False):

    mid = (
        row.start
        + (row.end - row.start) / 2
    )

    ax.text(
        mid,
        y_top,
        f"{row.cycle}. {row.policy_cycle}",
        ha="center",
        va="top",
        fontsize=8.5,
    )


ax.set_title(
    "504영업일 Rolling PCA — "
    "PC1 설명비중과 한국은행 정책금리 사이클",
    loc="left",
    fontweight="bold",
)

ax.set_xlabel("Rolling window 종료일")
ax.set_ylabel("PC1 설명비중 (%)")

ax.set_ylim(0, 105)

ax.grid(alpha=0.25)
ax.legend(loc="lower right")

fig.tight_layout()


# config 경로 규정 준수
rolling_pc1_fig_path = (
    config.H1_FIGDIR
    / "rolling_pc1_policy_cycles.png"
)
fig.savefig(
    rolling_pc1_fig_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

**관찰.** PC1 설명비중은 이동창에 따라 변동하며 일부 정책국면에서 상대적으로 낮아진다. 전체 표본의 1차원성 결론을 모든 시기에 동일하게 적용하기는 어렵다.

**3.A.6 산업 자신의 3요인 동시유의성(own-joint) 검정.** 확인할 내용: 산업별로 Level·Slope·Curvature 세 계수가 함께 유의한지, 동시검정과 개별검정 결과가 갈리는 산업이 있는지(다중공선성 신호).

In [56]:
def run_joint_significance_test(data, hac_lags=5,
                                test_vars=("d_ns_level", "d_ns_slope", "d_ns_curvature"),
                                control_cols=("kospi_return",)):
    """산업별로 H0: 지정한 요인 계수가 모두 0 인지 동시검정(F-test)한다."""
    test_vars = list(test_vars)
    x_cols = list(control_cols) + test_vars

    rows = []

    for industry, group in data.groupby("industry"):
        reg_data = group[["industry_return"] + x_cols].dropna()

        X = sm.add_constant(reg_data[x_cols], has_constant="add")
        y = reg_data["industry_return"]

        model = sm.OLS(y, X).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": hac_lags},
        )

        # "d_ns_level = 0, d_ns_slope = 0, ..." 형태의 제약식을 만든다.
        hypothesis = ", ".join(f"{v} = 0" for v in test_vars)
        f_test = model.f_test(hypothesis)

        individual_p = model.pvalues[test_vars]

        rows.append({
            "industry": industry,
            "model": "Exploratory_Joint_3Factor",
            "joint_f_stat": float(f_test.fvalue),
            "joint_p_value": float(f_test.pvalue),
            "df_num": int(len(test_vars)),
            "df_denom": int(model.df_resid),
            "min_individual_p": float(individual_p.min()),
            "n_individual_sig_5pct": int((individual_p < 0.05).sum()),
            "joint_sig_5pct": bool(f_test.pvalue < 0.05),
            "r_squared": model.rsquared,
            "adj_r_squared": model.rsquared_adj,
            "n_obs": int(model.nobs),
            "hac_lags": hac_lags,
        })

    return pd.DataFrame(rows)


joint_ftest = (
    run_joint_significance_test(extended_panel, hac_lags=HAC_LAGS)
    .pipe(add_bh_correction, group_cols=("model",), p_col="joint_p_value")
    .rename(columns={"p_value_bh": "joint_p_value_bh"})
    .sort_values("joint_p_value")
    .reset_index(drop=True)
)

joint_path = config.H1_TABDIR / "exploratory_joint_ftest.csv"
joint_ftest.to_csv(joint_path, index=False, encoding="utf-8-sig")

print("동시검정 유의 산업:", int(joint_ftest["joint_sig_5pct"].sum()), "/ 7")
print("개별검정 유의 조합:", int(joint_ftest["n_individual_sig_5pct"].sum()), "/ 21")
display(joint_ftest.round(5))

동시검정 유의 산업: 3 / 7
개별검정 유의 조합: 3 / 21


,industry,model,joint_f_stat,joint_p_value,df_num,df_denom,min_individual_p,n_individual_sig_5pct,joint_sig_5pct,r_squared,adj_r_squared,n_obs,hac_lags,joint_p_value_bh,bh_n_tests,bh_family,significant_5pct_bh
0,의약품,Exploratory_Joint_3Factor,4.73722,0.00268,3,2446,0.00236,1,True,0.24353,0.24229,2451,5,0.01875,7,Exploratory_Joint_3Factor,True
1,운수장비,Exploratory_Joint_3Factor,3.59193,0.01313,3,2446,0.00721,1,True,0.47232,0.47146,2451,5,0.04596,7,Exploratory_Joint_3Factor,True
2,금융업,Exploratory_Joint_3Factor,2.97331,0.03057,3,2446,0.02938,1,True,0.62347,0.62286,2451,5,0.07134,7,Exploratory_Joint_3Factor,False
3,철강금속,Exploratory_Joint_3Factor,2.03121,0.10744,3,2446,0.14948,0,False,0.38347,0.38246,2451,5,0.18802,7,Exploratory_Joint_3Factor,False
4,건설업,Exploratory_Joint_3Factor,1.25128,0.28955,3,2446,0.12431,0,False,0.39943,0.39845,2451,5,0.40537,7,Exploratory_Joint_3Factor,False
5,전기전자,Exploratory_Joint_3Factor,0.38846,0.76133,3,2446,0.36228,0,False,0.71939,0.71893,2451,5,0.79026,7,Exploratory_Joint_3Factor,False
6,화학,Exploratory_Joint_3Factor,0.34849,0.79026,3,2446,0.51630,0,False,0.64374,0.64316,2451,5,0.79026,7,Exploratory_Joint_3Factor,False


In [57]:
# 동시검정 p값과 개별검정 최소 p값을 나란히 놓아
# "함께는 유의한데 따로는 아니다"가 눈에 보이게 한다.
plot_data = joint_ftest.sort_values("joint_p_value", ascending=False).reset_index(drop=True)

y_pos = np.arange(len(plot_data))
bar_height = 0.38

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(y_pos + bar_height / 2, plot_data["joint_p_value"],
        height=bar_height, label="동시검정 p (Level·Slope·Curvature 공동)")
ax.barh(y_pos - bar_height / 2, plot_data["min_individual_p"],
        height=bar_height, label="개별검정 최소 p")

ax.axvline(0.05, linestyle="--", linewidth=1.2, color="crimson")
ax.text(0.05, len(plot_data) - 0.35, " p = 0.05", color="crimson", fontsize=9, va="top")

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_data["industry"])
ax.set_xlabel("p-value (작을수록 유의)")
ax.set_title(
    "산업별 수익률곡선 요인 동시검정 vs 개별검정\n"
    "왼쪽 막대가 0.05선을 넘고 오른쪽이 못 넘으면 다중공선성 신호",
    loc="left",
    fontweight="bold",
)
ax.legend(loc="lower right", fontsize=9)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()

joint_fig_path = config.H1_FIGDIR / "05_joint_ftest.png"
fig.savefig(joint_fig_path, dpi=200, bbox_inches="tight")
plt.show()


**관찰.** 동시검정에서 유의한 산업 수와 개별검정 최소 p-value 기준 유의 산업 수는 정확히 같지 않다. 함께는 유의하지만 개별로는 어느 계수도 5%를 넘지 못하는 산업이 있다.

**3.A.7 2요인(Level+Slope) 모형.** 확인할 내용: Curvature를 빼도 Level·Slope 계수와 VIF가 3요인 모형과 크게 달라지는지.

In [58]:
# Level + Slope 2요인 모형. 함수는 9단계 것을 그대로 쓴다.
TWO_FACTOR_X_COLS = ["kospi_return", "d_ns_level", "d_ns_slope"]
TWO_FACTOR_LABELS = {"d_ns_level": "Level", "d_ns_slope": "Slope"}

two_factor_betas = (
    run_yield_curve_factor_model(
        extended_panel,
        hac_lags=HAC_LAGS,
        x_cols=TWO_FACTOR_X_COLS,
        factor_labels=TWO_FACTOR_LABELS,
    )
    .assign(model="Exploratory_TwoFactor")
    .pipe(add_bh_correction, group_cols=("model", "factor"))
)

# 2요인 모형의 VIF 를 다시 계산한다(설명변수가 줄면 VIF 도 줄어야 한다).
two_factor_daily = (
    extended_panel[["date"] + TWO_FACTOR_X_COLS]
    .drop_duplicates(subset="date")
    .dropna()
)
two_factor_design = sm.add_constant(two_factor_daily[TWO_FACTOR_X_COLS], has_constant="add")

two_factor_vif = {
    var: variance_inflation_factor(two_factor_design.to_numpy(dtype=float), i)
    for i, var in enumerate(two_factor_design.columns)
    if var != "const"
}

two_factor_betas["vif"] = two_factor_betas["variable"].map(two_factor_vif)

two_factor_path = config.H1_TABDIR / "exploratory_two_factor_betas.csv"
two_factor_betas.to_csv(two_factor_path, index=False, encoding="utf-8-sig")

print("2요인 VIF:", {k: round(v, 3) for k, v in two_factor_vif.items()})
print("3요인 대비 유의 조합:",
      int(two_factor_betas["significant_5pct"].sum()), "/", len(two_factor_betas))
display(two_factor_betas.round(5))

2요인 VIF: {'kospi_return': np.float64(1.0), 'd_ns_level': np.float64(3.455), 'd_ns_slope': np.float64(3.454)}
3요인 대비 유의 조합: 1 / 14


,industry,factor,variable,coefficient,std_error_hac,t_value,p_value,ci_lower_95,ci_upper_95,significant_5pct,market_beta,r_squared,adj_r_squared,n_obs,hac_lags,optimal_lambda,model,p_value_bh,bh_n_tests,bh_family,significant_5pct_bh,vif
0,건설업,Level,d_ns_level,-0.01153,0.01257,-0.91767,0.35879,-0.03616,0.01310,False,1.01490,0.39925,0.39852,2451,5,0.5,Exploratory_TwoFactor,0.54582,7,Exploratory_TwoFactor | Level,False,3.45470
1,건설업,Slope,d_ns_slope,-0.01900,0.01190,-1.59641,0.11040,-0.04233,0.00433,False,1.01490,0.39925,0.39852,2451,5,0.5,Exploratory_TwoFactor,0.19319,7,Exploratory_TwoFactor | Slope,False,3.45399
2,금융업,Level,d_ns_level,0.00817,0.00637,1.28217,0.19978,-0.00432,0.02065,False,0.92408,0.62311,0.62265,2451,5,0.5,Exploratory_TwoFactor,0.46616,7,Exploratory_TwoFactor | Level,False,3.45470
3,금융업,Slope,d_ns_slope,-0.00190,0.00532,-0.35701,0.72109,-0.01232,0.00853,False,0.92408,0.62311,0.62265,2451,5,0.5,Exploratory_TwoFactor,0.78542,7,Exploratory_TwoFactor | Slope,False,3.45399
4,운수장비,Level,d_ns_level,-0.01166,0.00901,-1.29442,0.19552,-0.02932,0.00600,False,1.00200,0.47079,0.47014,2451,5,0.5,Exploratory_TwoFactor,0.46616,7,Exploratory_TwoFactor | Level,False,3.45470
5,운수장비,Slope,d_ns_slope,-0.01346,0.00812,-1.65867,0.09718,-0.02937,0.00245,False,1.00200,0.47079,0.47014,2451,5,0.5,Exploratory_TwoFactor,0.19319,7,Exploratory_TwoFactor | Slope,False,3.45399
6,의약품,Level,d_ns_level,0.02115,0.01156,1.83028,0.06721,-0.00150,0.04380,False,0.79514,0.24115,0.24022,2451,5,0.5,Exploratory_TwoFactor,0.46616,7,Exploratory_TwoFactor | Level,False,3.45470
7,의약품,Slope,d_ns_slope,0.02338,0.00983,2.37760,0.01743,0.00411,0.04265,True,0.79514,0.24115,0.24022,2451,5,0.5,Exploratory_TwoFactor,0.12198,7,Exploratory_TwoFactor | Slope,False,3.45399
8,전기전자,Level,d_ns_level,-0.00037,0.00638,-0.05864,0.95324,-0.01288,0.01213,False,1.20547,0.71930,0.71896,2451,5,0.5,Exploratory_TwoFactor,0.95324,7,Exploratory_TwoFactor | Level,False,3.45470
9,전기전자,Slope,d_ns_slope,0.00153,0.00561,0.27226,0.78542,-0.00947,0.01252,False,1.20547,0.71930,0.71896,2451,5,0.5,Exploratory_TwoFactor,0.78542,7,Exploratory_TwoFactor | Slope,False,3.45399


**관찰.** 2요인 모형의 VIF는 3요인 모형보다 낮고, Level·Slope 계수의 유의성 판정은 대체로 유지된다.

**3.A.8 3요인·직교화·2요인 모형 비교.** 확인할 내용: Gram–Schmidt로 직교화한 Curvature 계수가 원 3요인 모형과 같은지(FWL 확인), 세 모형의 유의 조합 수를 나란히 비교했을 때 어느 모형이 우월하다고 말할 수 있는지.

In [59]:
def orthogonalize_factors(daily, factor_cols):
    """Gram-Schmidt 직교화. 앞 요인과 겹치는 부분을 잔차로 떼어낸다."""
    out = daily.copy()
    built = []

    for col in factor_cols:
        target = f"o_{col}"
        if not built:
            out[target] = out[col]                      # 첫 요인은 그대로
        else:
            X = sm.add_constant(out[built], has_constant="add")
            out[target] = sm.OLS(out[col], X).fit().resid   # 앞 요인 성분 제거
        built.append(target)

    return out, built


BASE_FACTORS = ["d_ns_level", "d_ns_slope", "d_ns_curvature"]

# 설명변수는 모든 산업이 공유하므로 날짜 단위에서 한 번만 직교화한다.
orth_daily, ORTH_COLS = orthogonalize_factors(
    extended_panel[["date"] + BASE_FACTORS].drop_duplicates(subset="date").dropna(),
    BASE_FACTORS,
)

# 직교화 확인: 상관 0, VIF 1
print("직교화 후 상관행렬:")
display(orth_daily[ORTH_COLS].corr().round(6))

# 기존 패널은 건드리지 않고 사본에만 직교 변수를 붙인다.
exploratory_panel = extended_panel.merge(
    orth_daily[["date"] + ORTH_COLS],
    on="date",
    how="left",
    validate="many_to_one",
)

ORTH_X_COLS = ["kospi_return"] + ORTH_COLS
ORTH_LABELS = {
    "o_d_ns_level": "Level_orth",
    "o_d_ns_slope": "Slope_orth",
    "o_d_ns_curvature": "Curvature_orth",
}

orthogonalized_betas = (
    run_yield_curve_factor_model(
        exploratory_panel,
        hac_lags=HAC_LAGS,
        x_cols=ORTH_X_COLS,
        factor_labels=ORTH_LABELS,
    )
    .assign(model="Exploratory_Orthogonalized")
    .pipe(add_bh_correction, group_cols=("model", "factor"))
)

orth_path = config.H1_TABDIR / "exploratory_orthogonalized_betas.csv"
orthogonalized_betas.to_csv(orth_path, index=False, encoding="utf-8-sig")

display(orthogonalized_betas.round(5))

직교화 후 상관행렬:


,o_d_ns_level,o_d_ns_slope,o_d_ns_curvature
o_d_ns_level,1.0,-0.0,0.0
o_d_ns_slope,-0.0,1.0,-0.0
o_d_ns_curvature,0.0,-0.0,1.0


,industry,factor,variable,coefficient,std_error_hac,t_value,p_value,ci_lower_95,ci_upper_95,significant_5pct,market_beta,r_squared,adj_r_squared,n_obs,hac_lags,optimal_lambda,model,p_value_bh,bh_n_tests,bh_family,significant_5pct_bh
0,건설업,Level_orth,o_d_ns_level,0.00533,0.00637,0.83580,0.40327,-0.00717,0.01782,False,1.01328,0.39943,0.39845,2451,5,0.5,Exploratory_Orthogonalized,0.91886,7,Exploratory_Orthogonalized | Level_orth,False
1,건설업,Slope_orth,o_d_ns_slope,-0.01900,0.01198,-1.58630,0.11267,-0.04247,0.00447,False,1.01328,0.39943,0.39845,2451,5,0.5,Exploratory_Orthogonalized,0.19717,7,Exploratory_Orthogonalized | Slope_orth,False
2,건설업,Curvature_orth,o_d_ns_curvature,-0.00234,0.00306,-0.76450,0.44457,-0.00833,0.00366,False,1.01328,0.39943,0.39845,2451,5,0.5,Exploratory_Orthogonalized,0.51866,7,Exploratory_Orthogonalized | Curvature_orth,False
3,금융업,Level_orth,o_d_ns_level,0.00984,0.00436,2.25534,0.02411,0.00129,0.01840,True,0.92575,0.62347,0.62286,2451,5,0.5,Exploratory_Orthogonalized,0.16878,7,Exploratory_Orthogonalized | Level_orth,False
4,금융업,Slope_orth,o_d_ns_slope,-0.00190,0.00533,-0.35725,0.72091,-0.01236,0.00855,False,0.92575,0.62347,0.62286,2451,5,0.5,Exploratory_Orthogonalized,0.78731,7,Exploratory_Orthogonalized | Slope_orth,False
5,금융업,Curvature_orth,o_d_ns_curvature,0.00241,0.00137,1.76201,0.07807,-0.00027,0.00510,False,0.92575,0.62347,0.62286,2451,5,0.5,Exploratory_Orthogonalized,0.18216,7,Exploratory_Orthogonalized | Curvature_orth,False
6,운수장비,Level_orth,o_d_ns_level,0.00026,0.00557,0.04651,0.96290,-0.01066,0.01118,False,1.00631,0.47232,0.47146,2451,5,0.5,Exploratory_Orthogonalized,0.96290,7,Exploratory_Orthogonalized | Level_orth,False
7,운수장비,Slope_orth,o_d_ns_slope,-0.01348,0.00789,-1.70782,0.08767,-0.02895,0.00199,False,1.00631,0.47232,0.47146,2451,5,0.5,Exploratory_Orthogonalized,0.19717,7,Exploratory_Orthogonalized | Slope_orth,False
8,운수장비,Curvature_orth,o_d_ns_curvature,0.00621,0.00231,2.68719,0.00721,0.00168,0.01074,True,1.00631,0.47232,0.47146,2451,5,0.5,Exploratory_Orthogonalized,0.02522,7,Exploratory_Orthogonalized | Curvature_orth,True
9,의약품,Level_orth,o_d_ns_level,0.00044,0.00826,0.05345,0.95737,-0.01574,0.01662,False,0.78917,0.24353,0.24229,2451,5,0.5,Exploratory_Orthogonalized,0.96290,7,Exploratory_Orthogonalized | Level_orth,False


In [60]:
# 2요인·직교화 모형의 계수 신뢰구간 그림 (12단계 errorbar 스타일 유지)
def plot_factor_ci(result, factors, title, out_path, ncols=2):
    """요인별 산업 계수와 95% CI 를 패널로 그린다. 스타일은 12단계와 동일.

    색상은 계수 부호를 나타낸다. 탐색적 분석이라 BH를 적용하지 않고
    보정 전 p_value < 0.05(significant_5pct) 기준으로 판정한다.
    """
    nrows = int(np.ceil(len(factors) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 4.5 * nrows),
                             squeeze=False)
    flat = axes.ravel()

    for ax, factor in zip(flat, factors):
        d = (result[result["factor"] == factor]
             .sort_values("coefficient")
             .reset_index(drop=True))
        y_pos = np.arange(len(d))

        point_colors = [
            config.sign_color(coef, sig)
            for coef, sig in zip(d["coefficient"], d["significant_5pct"])
        ]

        for y, coef, lo, hi, color in zip(
            y_pos,
            d["coefficient"],
            d["coefficient"] - d["ci_lower_95"],
            d["ci_upper_95"] - d["coefficient"],
            point_colors,
        ):
            ax.errorbar(
                coef, y,
                xerr=[[lo], [hi]],
                fmt="o", capsize=4, linewidth=1.8, markersize=7,
                color=color, ecolor=color,
            )
        ax.axvline(0, linestyle="--", linewidth=1)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(d["industry"])
        ax.set_xlabel(f"{factor} 반응계수")
        ax.set_title(factor, loc="left", fontweight="bold")
        ax.grid(axis="x", alpha=0.25)

    for ax in flat[len(factors):]:
        ax.set_visible(False)

    legend_handles = [
        Line2D([0], [0], marker="o", linestyle="None", markersize=8,
               color=config.SIGN_COLORS["neg"], label="유의한 음(\u2212)"),
        Line2D([0], [0], marker="o", linestyle="None", markersize=8,
               color=config.SIGN_COLORS["pos"], label="유의한 양(+)"),
        Line2D([0], [0], marker="o", linestyle="None", markersize=8,
               color=config.SIGN_COLORS["ns"], label="비유의"),
    ]
    fig.legend(handles=legend_handles, loc="upper right", frameon=True, fontsize=9,
               bbox_to_anchor=(0.99, 0.99))

    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()


plot_factor_ci(
    two_factor_betas, ["Level", "Slope"],
    "[탐색적] 2요인 모형 — 산업별 계수와 95% 신뢰구간 (HAC, 보정 전 p 기준)",
    config.H1_FIGDIR / "06_two_factor_ci.png",
)

plot_factor_ci(
    orthogonalized_betas, ["Level_orth", "Slope_orth", "Curvature_orth"],
    "[탐색적] 직교화 모형 — 산업별 계수와 95% 신뢰구간 (HAC, 순서 의존)",
    config.H1_FIGDIR / "07_orthogonalized_ci.png",
    ncols=3,
)

In [61]:
# 모형별 유의 조합 수 비교
comparison = pd.DataFrame([
    {"모형": "3요인 (B-1 기준)",
     "설명변수": 3,
     "유의_조합": int(yield_curve_factor_betas["significant_5pct"].sum()),
     "전체_조합": len(yield_curve_factor_betas),
     "최대_VIF": round(max(v for k, v in vif_values.items() if k != "d_credit_spread"), 2)},
    {"모형": "4요인 (B-2 확장)",
     "설명변수": 4,
     "유의_조합": int(credit_spread_extended["significant_5pct"].sum()),
     "전체_조합": len(credit_spread_extended),
     "최대_VIF": round(max(vif_values.values()), 2)},
    {"모형": "2요인 (탐색적)",
     "설명변수": 2,
     "유의_조합": int(two_factor_betas["significant_5pct"].sum()),
     "전체_조합": len(two_factor_betas),
     "최대_VIF": round(max(two_factor_vif.values()), 2)},
    {"모형": "직교화 3요인 (탐색적)",
     "설명변수": 3,
     "유의_조합": int(orthogonalized_betas["significant_5pct"].sum()),
     "전체_조합": len(orthogonalized_betas),
     "최대_VIF": 1.0},
])

print("동시검정(F) 유의 산업:", int(joint_ftest["joint_sig_5pct"].sum()), "/ 7")
print()
display(comparison)
print()
print("주의: 유의 개수가 많은 모형이 더 옳은 모형이 아니다.")
print("      모형을 바꿔가며 유의한 결과를 고르는 것은 p-hacking 이다.")

동시검정(F) 유의 산업: 3 / 7



,모형,설명변수,유의_조합,전체_조합,최대_VIF
0,3요인 (B-1 기준),3,3,21,7.25
1,4요인 (B-2 확장),4,3,28,7.25
2,2요인 (탐색적),2,1,14,3.45
3,직교화 3요인 (탐색적),3,4,21,1.00



주의: 유의 개수가 많은 모형이 더 옳은 모형이 아니다.
      모형을 바꿔가며 유의한 결과를 고르는 것은 p-hacking 이다.


**관찰.** 직교화 순서상 마지막에 남은 Curvature의 계수·표준오차는 원 3요인 모형과 수치오차 범위에서 같다(FWL). 유의 조합 수는 모형마다 다르지만, 이 차이를 모형 선택의 근거로 쓰지 않는다 — 유의한 결과가 나오는 모형을 고르는 것은 p-hacking이다.

**3.A.9 곡률 부호와 시장베타의 관계.** 확인할 내용: 산업별 곡률계수 부호가 그 산업의 시장베타(경기민감도)와 체계적으로 연결되는지.

In [62]:
from scipy.stats import spearmanr

beta_profile = (
    yield_curve_factor_betas[yield_curve_factor_betas["factor"] == "Curvature"]
    [["industry", "market_beta", "coefficient", "p_value"]]
    .rename(columns={"coefficient": "curvature_beta", "p_value": "curvature_p"})
    .sort_values("market_beta").reset_index(drop=True)
)

rho, pval = spearmanr(beta_profile["market_beta"], beta_profile["curvature_beta"])
beta_profile["spearman_rho"] = round(rho, 4)
beta_profile["spearman_p"] = round(pval, 4)

profile_path = config.H1_TABDIR / "curvature_market_beta_profile.csv"
beta_profile.to_csv(profile_path, index=False, encoding="utf-8-sig")

display(beta_profile.round(5))
print(f"\n시장베타 vs 곡률계수  Spearman rho = {rho:+.3f}  (p = {pval:.4f}, n=7)")
print()
if pval < 0.10:
    print("→ 경기민감도와 곡률 노출 사이에 관련이 있다.")
else:
    print("→ 관련이 확인되지 않는다.")
    print("   '방어주 대 경기민감주' 해석은 이 자료로 뒷받침되지 않는다.")
    print("   부호 차이는 관찰 사실로만 보고하고 해석을 붙이지 않는다.")

,industry,market_beta,curvature_beta,curvature_p,spearman_rho,spearman_p
0,의약품,0.78917,-0.00860,0.00236,0.1429,0.7599
1,금융업,0.92575,0.00241,0.07807,0.1429,0.7599
2,운수장비,1.00631,0.00621,0.00721,0.1429,0.7599
3,건설업,1.01328,-0.00234,0.44457,0.1429,0.7599
4,철강금속,1.02485,0.00415,0.14948,0.1429,0.7599
5,화학,1.05406,-0.00008,0.96244,0.1429,0.7599
6,전기전자,1.20647,0.00144,0.36228,0.1429,0.7599



시장베타 vs 곡률계수  Spearman rho = +0.143  (p = 0.7599, n=7)

→ 관련이 확인되지 않는다.
   '방어주 대 경기민감주' 해석은 이 자료로 뒷받침되지 않는다.
   부호 차이는 관찰 사실로만 보고하고 해석을 붙이지 않는다.


**관찰.** 시장베타와 곡률계수의 Spearman 순위상관은 7개 산업 표본에서 통계적으로 유의한 수준에 이르지 못한다. '방어주 대 경기민감주' 해석은 이 자료만으로는 뒷받침되지 않는다.

# 3. H2 — 금리 상승·하락 비대칭

## 연구질문과 검정

금리 상승과 하락에 대한 산업별 반응은 서로 다른가?

$$
R_{i,t} = \alpha_i + \gamma_i R_{KOSPI,t} + \beta_{up,i}\Delta y_{3,t}^{up} + \beta_{down,i}\Delta y_{3,t}^{down} + \varepsilon_{i,t}
$$

$\Delta y_{3,t}^{up}=\max(\Delta y_{3,t},0)$, $\Delta y_{3,t}^{down}=\min(\Delta y_{3,t},0)$이다. H2의 질문은 두 계수가 각각 0인지가 아니라 서로 같은지이다. 따라서 $H_0:\beta_{up}=\beta_{down}$을 Wald 검정하고 산업별 p-value에 BH 보정을 적용한다.


In [63]:
# 상승분·하락분 변수 생성 (3년물·Curvature)
extended_panel["d_y3_up"] = extended_panel["d_y3"].clip(lower=0)
extended_panel["d_y3_down"] = extended_panel["d_y3"].clip(upper=0)
extended_panel["d_curv_up"] = extended_panel["d_ns_curvature"].clip(lower=0)
extended_panel["d_curv_down"] = extended_panel["d_ns_curvature"].clip(upper=0)

n_up = int((extended_panel.drop_duplicates("date")["d_y3"] > 0).sum())
n_down = int((extended_panel.drop_duplicates("date")["d_y3"] < 0).sum())
n_zero = int((extended_panel.drop_duplicates("date")["d_y3"] == 0).sum())
print(f"표본 구성: 상승 {n_up}일 · 하락 {n_down}일 · 무변화 {n_zero}일")


def fit_asymmetric(data, up, down, hac_lags=HAC_LAGS):
    use_cols = ["industry_return", "kospi_return", up, down]
    sample = data[use_cols].replace([np.inf, -np.inf], np.nan).dropna()
    X = sm.add_constant(sample[["kospi_return", up, down]], has_constant="add")
    return sm.OLS(sample["industry_return"], X).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})


rows_wald = []
for ind in industries:
    g = extended_panel[extended_panel["industry"] == ind]
    m = fit_asymmetric(g, "d_y3_up", "d_y3_down")
    w = m.f_test("d_y3_up = d_y3_down")
    rows_wald.append({
        "industry": ind, "beta_up": m.params["d_y3_up"], "beta_down": m.params["d_y3_down"],
        "f_stat": float(w.fvalue), "p_value": float(w.pvalue),
    })
asymmetry_wald_tests = pd.DataFrame(rows_wald)
asymmetry_wald_tests["p_value_bh"] = multipletests(asymmetry_wald_tests["p_value"], method="fdr_bh")[1]
asymmetry_wald_tests["10bp 상승 시 예상 수익률 변화(%)"] = asymmetry_wald_tests["beta_up"] * 10
asymmetry_wald_tests["10bp 하락 시 예상 수익률 변화(%)"] = asymmetry_wald_tests["beta_down"] * -10
display(asymmetry_wald_tests.round(5))

표본 구성: 상승 1203일 · 하락 1197일 · 무변화 51일


,industry,beta_up,beta_down,f_stat,p_value,p_value_bh
0,건설업,-0.02161,0.03103,10.41780,0.00126,0.00885
1,금융업,0.00566,0.01748,1.11376,0.29137,0.42965
2,운수장비,0.01745,0.01699,0.00102,0.97451,0.97451
3,의약품,-0.01716,-0.03818,1.11250,0.29164,0.42965
4,전기전자,0.00784,-0.00542,1.65486,0.19842,0.42965
5,철강금속,0.02813,0.00841,1.04444,0.30689,0.42965
6,화학,0.00094,0.00437,0.07558,0.78341,0.91398


아래 그림은 각 산업의 상승·하락 계수를 연결해 두 계수의 간격과 부호를 보여준다. 별표는 Wald 검정의 BH p-value가 0.05 미만인 경우다.


In [64]:
# 산업별 상승·하락 계수 비교 (별표 = Wald BH p<0.05)
plot_w = asymmetry_wald_tests.sort_values("f_stat")
fig, ax = plt.subplots(figsize=(8, 4.3))
y = np.arange(len(plot_w))
ax.scatter(plot_w["beta_up"], y, color=NEG_COLOR, label="상승 β", zorder=3)
ax.scatter(plot_w["beta_down"], y, color=POS_COLOR, label="하락 β", zorder=3)
for yi, (_, row) in zip(y, plot_w.iterrows()):
    ax.plot([row["beta_up"], row["beta_down"]], [yi, yi], color="lightgray", zorder=1)
    if row["p_value_bh"] < 0.05:
        ax.text(max(row["beta_up"], row["beta_down"]) + 0.003, yi, "*", fontsize=14, va="center", color="black")
ax.axvline(0, color="gray", lw=0.8, ls="--")
ax.set_yticks(y)
ax.set_yticklabels(plot_w["industry"])
ax.set_xlabel("금리 계수 (3년물)")
ax.set_title("산업별 상승·하락 계수 (* = Wald BH p<0.05)", loc="left", fontweight="bold")
ax.legend(loc="best", frameon=True, fontsize=9)
fig.tight_layout()
plt.show()

## 핵심 결과

**건설업에서만 BH 보정 후 상승·하락 반응의 차이가 확인되었다.** 3년물 금리가 10bp 오를 때와 10bp 내릴 때의 예상 수익률 변화는 아래 대칭모형 비교 셀에서 결과표로부터 조립한다. 나머지 여섯 산업에서는 비대칭이 확인되지 않았다.


In [65]:
# 대칭모형과의 비교 — 건설업 대칭 β는 방향 분리 전에는 유의하지 않았다
sym_rows = []
for ind in industries:
    g = extended_panel[extended_panel["industry"] == ind][["industry_return", "kospi_return", "d_y3"]].dropna()
    X = sm.add_constant(g[["kospi_return", "d_y3"]], has_constant="add")
    m = sm.OLS(g["industry_return"], X).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
    sym_rows.append({"industry": ind, "beta_symmetric": m.params["d_y3"], "p_symmetric": m.pvalues["d_y3"]})
symmetric_betas = pd.DataFrame(sym_rows)

con_sym = symmetric_betas[symmetric_betas["industry"] == "건설업"].iloc[0]
con_wald = asymmetry_wald_tests[asymmetry_wald_tests["industry"] == "건설업"].iloc[0]
con_up_10bp = con_wald["10bp 상승 시 예상 수익률 변화(%)"]
con_down_10bp = con_wald["10bp 하락 시 예상 수익률 변화(%)"]
con_sym_10bp = con_sym["beta_symmetric"] * 10
comparison_table = pd.DataFrame([{
    "산업": "건설업",
    "대칭모형 원계수": con_sym["beta_symmetric"],
    "대칭모형 p-value": con_sym["p_symmetric"],
    "상승 원계수": con_wald["beta_up"],
    "하락 원계수": con_wald["beta_down"],
    "Wald F": con_wald["f_stat"],
    "BH p-value": con_wald["p_value_bh"],
}])
display(comparison_table.round(5))
print(
    f"건설업 지수의 예상 수익률 변화는 대칭모형에서 10bp 상승당 {con_sym_10bp:+.2f}%이며 "
    f"통계적으로 유의하지 않다. 방향을 나누면 10bp 상승 시 {con_up_10bp:+.2f}%, "
    f"10bp 하락 시 {con_down_10bp:+.2f}%로 추정된다."
)

건설업 대칭모형: beta=+0.0032, p=0.7008
건설업 방향분리: up=-0.0216, down=+0.0310, Wald F=10.4178, BH p=0.0089


방향을 분리하면 10bp 상승과 10bp 하락 모두 건설업 수익률에 음(-)의 기여를 하지만 크기는 서로 다르다. 하락분 자체가 0 이하이므로 양(+)의 하락 원계수도 실제 예상 수익률 변화에는 음(-)으로 환산된다. 이 패턴은 대칭모형에서 상쇄되던 방향별 반응이 드러난 경우다.


In [66]:
def joint_symmetry_test(data, up, down, date_col="date"):
    """H0: 모든 산업 i에 대해 beta_up_i = beta_down_i (7개 제약). 날짜 클러스터 SE."""
    d = data.dropna(subset=["industry", "industry_return", "kospi_return", date_col, up, down]).copy()
    date_codes = pd.factorize(d[date_col])[0]
    dummies = pd.get_dummies(d["industry"], prefix="ind", dtype=float)
    X = pd.concat([
        dummies,
        dummies.mul(d["kospi_return"].values, axis=0).add_prefix("mkt_"),
        dummies.mul(d[up].values, axis=0).add_prefix(f"{up}__"),
        dummies.mul(d[down].values, axis=0).add_prefix(f"{down}__"),
    ], axis=1)
    res = sm.OLS(d["industry_return"].values, X.values).fit(cov_type="cluster", cov_kwds={"groups": date_codes})
    names = list(X.columns)
    iu = [i for i, c in enumerate(names) if c.startswith(f"{up}__")]
    idn = [i for i, c in enumerate(names) if c.startswith(f"{down}__")]
    R = np.zeros((len(iu), len(names)))
    for k, (a, b) in enumerate(zip(iu, idn)):
        R[k, a], R[k, b] = 1.0, -1.0
    return res.f_test(R)


joint_y3 = joint_symmetry_test(extended_panel, "d_y3_up", "d_y3_down")
joint_curv = joint_symmetry_test(extended_panel, "d_curv_up", "d_curv_down")
print(f"전 산업 동시검정 — 3년물:   F(7)={float(joint_y3.fvalue):.4f}, p={float(joint_y3.pvalue):.4f}")
print(f"전 산업 동시검정 — Curvature: F(7)={float(joint_curv.fvalue):.4f}, p={float(joint_curv.pvalue):.4f}")

# Curvature 방향분해 — 산업별 Wald
curv_wald_rows = []
for ind in industries:
    g = extended_panel[extended_panel["industry"] == ind]
    m = fit_asymmetric(g, "d_curv_up", "d_curv_down")
    w = m.f_test("d_curv_up = d_curv_down")
    curv_wald_rows.append({"industry": ind, "f_stat": float(w.fvalue), "p_value": float(w.pvalue)})
curvature_wald_tests = pd.DataFrame(curv_wald_rows)
curvature_wald_tests["p_value_bh"] = multipletests(curvature_wald_tests["p_value"], method="fdr_bh")[1]
curvature_wald_tests.round(5)

전 산업 동시검정 — 3년물:   F(7)=2.1039, p=0.0401
전 산업 동시검정 — Curvature: F(7)=0.7644, p=0.6173


,industry,f_stat,p_value,p_value_bh
0,건설업,0.52478,0.46888,0.78409
1,금융업,0.01180,0.91351,0.91351
2,운수장비,1.11124,0.29192,0.78409
3,의약품,0.07584,0.78304,0.91351
4,전기전자,0.33969,0.56006,0.78409
5,철강금속,2.49889,0.11406,0.78409
6,화학,0.47997,0.48850,0.78409


전 산업 동시검정에서 3년물 비대칭은 5% 기준에서 기각되지만 경계에 가깝다. Curvature 방향분해는 산업별 Wald 검정과 전 산업 동시검정 모두 비대칭을 확인하지 못했다. 검정통계량과 p-value는 위 결과표에 제시한다. **H1의 Curvature 산업 차이는 H2의 방향 비대칭으로 이어지지 않았다.**

Basistha & Kurov(2008)는 주가반응이 상황에 따라 달라질 수 있음을 보인다. 변수와 분석단위가 다르므로 직접 재현으로 보지 않고, 건설업의 조건부 반응을 논의하는 참고문헌으로만 연결한다.


## 정책국면별 보조분석

상승·하락 방향과 별개로, 완화·인하 국면과 인상·긴축 국면의 평균 금리베타가 다른지 보조적으로 확인한다. H2의 주 검정은 상승·하락 계수의 Wald 검정이며, 정책국면 결과는 같은 비중으로 해석하지 않는다.


In [67]:
# 완화·인하 3개 국면과 인상·긴축 2개 국면의 금리베타 비교
REGIME_SPECS = [
    (1, "2016-01-04", "2017-11-29", "완화·인하"),
    (2, "2017-11-30", "2019-07-17", "인상·긴축"),
    (3, "2019-07-18", "2021-08-25", "완화·인하"),
    (4, "2021-08-26", "2024-10-10", "인상·긴축"),
    (5, "2024-10-11", "2025-12-30", "완화·인하"),
]
regime_map = pd.concat([
    pd.DataFrame({"date": pd.date_range(s, e, freq="D"), "regime": r, "direction": d})
    for r, s, e, d in REGIME_SPECS
], ignore_index=True)
regime_panel = extended_panel.merge(regime_map, on="date", how="inner")

direction_rows, all_regime_rows = [], []
for ind in industries:
    sub = regime_panel[regime_panel["industry"] == ind].dropna(
        subset=["industry_return", "kospi_return", "d_y3", "regime"]
    )
    X = pd.concat([
        pd.get_dummies(sub["regime"], prefix="const", dtype=float),
        pd.get_dummies(sub["regime"], prefix="mkt", dtype=float).mul(sub["kospi_return"].values, axis=0),
        pd.get_dummies(sub["regime"], prefix="rate", dtype=float).mul(sub["d_y3"].values, axis=0),
    ], axis=1)
    model = sm.OLS(sub["industry_return"], X).fit(
        cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS}
    )

    beta_tight = (model.params["rate_2"] + model.params["rate_4"]) / 2
    beta_ease = (model.params["rate_1"] + model.params["rate_3"] + model.params["rate_5"]) / 3
    R_direction = np.zeros((1, len(model.params)))
    for regime, weight_value in {1: -1/3, 2: 0.5, 3: -1/3, 4: 0.5, 5: -1/3}.items():
        R_direction[0, list(model.params.index).index(f"rate_{regime}")] = weight_value
    direction_rows.append({
        "industry": ind,
        "beta_tight": beta_tight,
        "beta_ease": beta_ease,
        "p_value": float(model.f_test(R_direction).pvalue),
    })

    R_all = np.zeros((4, len(model.params)))
    rate_idx = [list(model.params.index).index(f"rate_{r}") for r in [1, 2, 3, 4, 5]]
    for k in range(4):
        R_all[k, rate_idx[0]], R_all[k, rate_idx[k + 1]] = 1.0, -1.0
    all_regime_rows.append({"industry": ind, "p_value": float(model.f_test(R_all).pvalue)})

regime_direction_tests = pd.DataFrame(direction_rows)
regime_direction_tests["p_value_bh"] = multipletests(
    regime_direction_tests["p_value"], method="fdr_bh"
)[1]
regime_joint_tests = pd.DataFrame(all_regime_rows)
regime_joint_tests["p_value_bh"] = multipletests(
    regime_joint_tests["p_value"], method="fdr_bh"
)[1]

pd.DataFrame([
    {
        "검정": "인상·긴축 vs 완화·인하 평균차",
        "최소 BH p-value": regime_direction_tests["p_value_bh"].min(),
        "유의 산업 수": int((regime_direction_tests["p_value_bh"] < 0.05).sum()),
    },
    {
        "검정": "5개 정책국면 금리베타 동일성",
        "최소 BH p-value": regime_joint_tests["p_value_bh"].min(),
        "유의 산업 수": int((regime_joint_tests["p_value_bh"] < 0.05).sum()),
    },
]).round(4)


,검정,최소 BH p-value,유의 산업 수
0,인상·긴축 vs 완화·인하 평균차,0.2956,0
1,5개 정책국면 금리베타 동일성,0.0650,0


BH 보정 후 어느 산업에서도 정책국면 간 차이가 확인되지 않았다. 이 결과는 상승·하락 방향 비대칭과 정책국면 구분이 서로 다른 질문임을 보여주는 보조결과다.

H2의 주 결과는 건설업 한 산업에 집중되고 전 산업 검정도 경계적이다. 극단값·기간·Rolling Window·표준오차·다중검정 범위·변동크기 분석은 Appendix C에서 반론별로 점검한다. H3는 이 방향 비대칭을 설명하는 단계가 아니라, Baseline 산업베타와 재무특성의 순위관계를 별도로 탐색한다.


### 4.A 추가 분석 — Final Notebook 미포함

`07_asymmetric_rate_response.ipynb`·`08_regime_interaction.ipynb`의 보조분석을 옮긴다. 07의 원본 셀은 전역 변수 `panel`이 확장 패널을 가리킨다고 가정하므로, 이 블록 동안만 `panel`을 `extended_panel`의 별칭으로 쓰고 08[3]이 Baseline 패널을 다시 읽으면서 자연히 원래대로 돌아온다.

In [68]:
# 07 원본 셀은 `panel`이 확장 패널을 가리킨다고 가정한다.
# 이미 로드된 extended_panel을 그대로 별칭으로 쓴다(재수집·재계산 없음).
panel = extended_panel

**4.A.1 표본 균형과 금리변화 기술통계.** 확인할 내용: 3년물 금리의 상승·하락·무변화일 구성과 상승분·하락분의 기술통계.

In [69]:
# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
# 저장소 루트에서 `pip install -e .` 를 한 번 실행했다면 아래 탐색은 건너뛴다.
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더 (Jupyter/Lab/Colab)
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
# 마커를 하나만 보면 다른 폴더의 동명 config.py 를 잘못 집을 수 있다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
# 이미 sys.path에 있더라도 맨 앞으로 올린다. cwd(예: 바탕화면)가 앞에 있으면
# 같은 이름의 다른 config.py 가 먼저 잡히기 때문이다.
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(
        f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}\n"
        f"저장소 루트({_root})에서 `pip install -e .` 를 실행하세요."
    )
# --- 부트스트랩 끝 ---

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

# 한글 폰트 설정 (config.py 5-c절 공통 설정).
config.setup_korean_font()

# 경로·상수는 반드시 config에서 가져온다. 로컬 별칭으로 값을 다시 정의하지 않는다.
TABDIR = config.H2_TABDIR
FIGDIR = config.H2_FIGDIR
HAC_LAGS = config.HAC_MAXLAGS
BH_ALPHA = config.BH_ALPHA

if not config.EXTENDED_PANEL_PATH.exists():
    raise FileNotFoundError(
        f"확장 패널을 찾지 못했습니다: {config.EXTENDED_PANEL_PATH}\n"
        "06_yield_curve_exposure 를 먼저 실행하세요."
    )


그래프 폰트: Noto Sans CJK JP


In [70]:
daily_rate = (
    panel[["date", "d_y3", "d_y3_up", "d_y3_down"]]
    .drop_duplicates(subset="date")
    .sort_values("date")
    .reset_index(drop=True)
)

up_down_sample_diagnostics = pd.DataFrame({
    "category": ["up", "down", "zero", "missing", "total"],
    "label": ["금리 상승일", "금리 하락일", "변화 없음", "결측", "전체"],
    "n_days": [
        int((daily_rate["d_y3"] > 0).sum()),
        int((daily_rate["d_y3"] < 0).sum()),
        int((daily_rate["d_y3"] == 0).sum()),
        int(daily_rate["d_y3"].isna().sum()),
        int(len(daily_rate)),
    ],
})

diag_path = TABDIR / "up_down_sample_diagnostics.csv"
up_down_sample_diagnostics.to_csv(diag_path, index=False, encoding="utf-8-sig")
display(up_down_sample_diagnostics)

,category,label,n_days
0,up,금리 상승일,1203
1,down,금리 하락일,1197
2,zero,변화 없음,51
3,missing,결측,1
4,total,전체,2452


In [71]:
up_down_rate_summary = daily_rate[["d_y3", "d_y3_up", "d_y3_down"]].describe().T

rate_summary_path = TABDIR / "up_down_rate_summary.csv"
up_down_rate_summary.to_csv(rate_summary_path, encoding="utf-8-sig")
display(up_down_rate_summary)

print("금리 상승폭이 가장 큰 날짜(상위 5일):")
display(daily_rate.nlargest(5, "d_y3"))

print("금리 하락폭이 가장 큰 날짜(하위 5일):")
display(daily_rate.nsmallest(5, "d_y3"))

,count,mean,std,min,25%,50%,75%,max
d_y3,2451.0,0.000538,0.038799,-0.268,-0.017,0.0,0.016,0.349
d_y3_up,2451.0,0.012881,0.025294,0.000,0.000,0.0,0.016,0.349
d_y3_down,2451.0,-0.012344,0.023398,-0.268,-0.017,0.0,0.000,0.000


금리 상승폭이 가장 큰 날짜(상위 5일):


,date,d_y3,d_y3_up,d_y3_down
1654,2022-09-26,0.349,0.349,0.0
1652,2022-09-22,0.257,0.257,0.0
1530,2022-03-28,0.242,0.242,0.0
1582,2022-06-13,0.239,0.239,0.0
1906,2023-10-04,0.224,0.224,0.0


금리 하락폭이 가장 큰 날짜(하위 5일):


,date,d_y3,d_y3_up,d_y3_down
1768,2023-03-13,-0.268,0.0,-0.268
1655,2022-09-27,-0.244,0.0,-0.244
1664,2022-10-12,-0.235,0.0,-0.235
1956,2023-12-14,-0.207,0.0,-0.207
1686,2022-11-11,-0.199,0.0,-0.199


In [72]:
diag_plot = up_down_sample_diagnostics[
    up_down_sample_diagnostics["category"].isin(["up", "down", "zero"])
].copy()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(diag_plot["label"], diag_plot["n_days"], color="#4c72b0")
ax.bar_label(bars, padding=3, fmt="%d")
ax.set_xlabel("금리변화 구분")
ax.set_ylabel("거래일 수")
ax.set_title("국고채 3년 금리 상승·하락 표본 구성")
ax.grid(axis="y", linestyle=":", alpha=0.5)
fig.tight_layout()

fig01_path = FIGDIR / "01_up_down_sample_balance.png"
fig.savefig(fig01_path, dpi=200, bbox_inches="tight")
plt.show()

**관찰.** 상승일과 하락일의 표본 수는 거의 균등하게 나뉜다. 상승분과 하락분의 평균 절대크기는 비슷하지만 완전히 같지는 않다.

**4.A.2 10bp 환산.** 확인할 내용: 산업별 상승·하락 계수를 10bp 금리 변화에 대한 예상 수익률 변화(%)로 환산했을 때 크기가 얼마나 되는지.

In [73]:
# add_bh_correction -- nb06(06_yield_curve_exposure.ipynb) cell 42 원문 그대로.
# PART A(5단계)에서 곧바로 BH 보정이 필요하므로 여기서 먼저 정의한다.
# (검정 함수 자체에 대한 설명은 7단계에서 run_industry_difference_test와 함께 다룬다.)
def add_bh_correction(result, group_cols=("model", "factor"),
                      p_col="p_value", alpha=0.05):
    out = result.copy()
    group_cols = list(group_cols)

    out["p_value_bh"] = np.nan
    out["bh_n_tests"] = 0

    # 길이 1 리스트로 그룹화하면 pandas 4에서 키가 스칼라 대신 튜플이 된다. 단일 열이면 스칼라로 넘긴다.
    _by = group_cols[0] if len(group_cols) == 1 else group_cols
    for _, _sub in out.groupby(_by):
        idx = _sub.index
        pvals = out.loc[idx, p_col].to_numpy(dtype=float)
        out.loc[idx, "p_value_bh"] = multipletests(pvals, method="fdr_bh")[1]
        out.loc[idx, "bh_n_tests"] = len(pvals)

    out["bh_family"] = out[group_cols].astype(str).agg(" | ".join, axis=1)
    out["significant_5pct_bh"] = out["p_value_bh"] < alpha

    return out


def fit_asymmetric(data, up, down, hac_lags=HAC_LAGS):
    # R = a + g*kospi_return + b_up*X_up + b_down*X_down (HAC)
    use_cols = ["industry_return", "kospi_return", up, down]
    sample = data[use_cols].replace([np.inf, -np.inf], np.nan).dropna()
    X = sm.add_constant(sample[["kospi_return", up, down]], has_constant="add")
    return sm.OLS(sample["industry_return"], X).fit(
        cov_type="HAC", cov_kwds={"maxlags": hac_lags}
    )


industries = sorted(panel["industry"].unique())

rows_y3 = []
for ind in industries:
    g = panel[panel["industry"] == ind]
    m = fit_asymmetric(g, "d_y3_up", "d_y3_down")
    ci = m.conf_int(alpha=0.05)
    for direction, var in [("up", "d_y3_up"), ("down", "d_y3_down")]:
        rows_y3.append({
            "model": "Y3_Asymmetry",
            "industry": ind,
            "direction": direction,
            "beta": m.params[var],
            "se_hac": m.bse[var],
            "t_value": m.tvalues[var],
            "p_value": m.pvalues[var],
            "ci_lower_95": ci.loc[var, 0],
            "ci_upper_95": ci.loc[var, 1],
            "nobs": int(m.nobs),
            "r_squared": m.rsquared,
            "hac_lags": HAC_LAGS,
        })

up_down_beta_long = add_bh_correction(
    pd.DataFrame(rows_y3), group_cols=("model", "direction")
)
display(up_down_beta_long.round(6))

,model,industry,direction,beta,se_hac,t_value,p_value,ci_lower_95,ci_upper_95,nobs,r_squared,hac_lags,p_value_bh,bh_n_tests,bh_family,significant_5pct_bh
0,Y3_Asymmetry,건설업,up,-0.021607,0.011762,-1.837099,0.066195,-0.044659,0.001445,2451,0.400090,5,0.154456,7,Y3_Asymmetry | up,False
1,Y3_Asymmetry,건설업,down,0.031027,0.011120,2.790127,0.005269,0.009232,0.052822,2451,0.400090,5,0.016153,7,Y3_Asymmetry | down,True
2,Y3_Asymmetry,금융업,up,0.005664,0.007039,0.804570,0.421068,-0.008133,0.019461,2451,0.623040,5,0.491246,7,Y3_Asymmetry | up,False
3,Y3_Asymmetry,금융업,down,0.017479,0.006473,2.700536,0.006923,0.004793,0.030165,2451,0.623040,5,0.016153,7,Y3_Asymmetry | down,True
4,Y3_Asymmetry,운수장비,up,0.017446,0.009371,1.861647,0.062653,-0.000921,0.035814,2451,0.471961,5,0.154456,7,Y3_Asymmetry | up,False
5,Y3_Asymmetry,운수장비,down,0.016986,0.009103,1.866056,0.062034,-0.000855,0.034827,2451,0.471961,5,0.108559,7,Y3_Asymmetry | down,False
6,Y3_Asymmetry,의약품,up,-0.017156,0.011598,-1.479163,0.139097,-0.039888,0.005576,2451,0.243546,5,0.243419,7,Y3_Asymmetry | up,False
7,Y3_Asymmetry,의약품,down,-0.038179,0.013737,-2.779356,0.005447,-0.065102,-0.011256,2451,0.243546,5,0.016153,7,Y3_Asymmetry | down,True
8,Y3_Asymmetry,전기전자,up,0.007840,0.007236,1.083492,0.278590,-0.006342,0.022022,2451,0.719434,5,0.390026,7,Y3_Asymmetry | up,False
9,Y3_Asymmetry,전기전자,down,-0.005425,0.005985,-0.906372,0.364739,-0.017155,0.006306,2451,0.719434,5,0.510635,7,Y3_Asymmetry | down,False


In [74]:
effect_10bp = up_down_beta_long.copy()

effect_10bp["rate_change_10bp"] = np.where(effect_10bp["direction"] == "up", 0.10, -0.10)
effect_10bp["expected_return_change_pct"] = (
    effect_10bp["beta"] * effect_10bp["rate_change_10bp"] * 100
)
effect_10bp["scenario"] = effect_10bp["direction"].replace(
    {"up": "금리 10bp 상승", "down": "금리 10bp 하락"}
)

up_down_effect_10bp = effect_10bp[[
    "industry", "scenario", "beta", "expected_return_change_pct", "p_value", "p_value_bh",
]].rename(columns={
    "industry": "산업",
    "scenario": "금리 시나리오",
    "beta": "추정 계수",
    "expected_return_change_pct": "예상 산업수익률 변화(%)",
    "p_value": "원 p-value",
    "p_value_bh": "BH p-value",
})

effect_path = TABDIR / "up_down_effect_10bp.csv"
up_down_effect_10bp.to_csv(effect_path, index=False, encoding="utf-8-sig")
display(up_down_effect_10bp)

,산업,금리 시나리오,추정 계수,예상 산업수익률 변화(%),원 p-value,BH p-value
0,건설업,금리 10bp 상승,-0.021607,-0.216071,0.066195,0.154456
1,건설업,금리 10bp 하락,0.031027,-0.310266,0.005269,0.016153
2,금융업,금리 10bp 상승,0.005664,0.056637,0.421068,0.491246
3,금융업,금리 10bp 하락,0.017479,-0.174793,0.006923,0.016153
4,운수장비,금리 10bp 상승,0.017446,0.174464,0.062653,0.154456
5,운수장비,금리 10bp 하락,0.016986,-0.169860,0.062034,0.108559
6,의약품,금리 10bp 상승,-0.017156,-0.171558,0.139097,0.243419
7,의약품,금리 10bp 하락,-0.038179,0.381791,0.005447,0.016153
8,전기전자,금리 10bp 상승,0.007840,0.078400,0.278590,0.390026
9,전기전자,금리 10bp 하락,-0.005425,0.054246,0.364739,0.510635


**관찰.** 10bp 환산 크기는 산업마다 차이가 있으며 BH 유의 산업에서도 절대적인 크기는 크지 않다.

**4.A.3 곡률 상승·하락 계수.** 확인할 내용: 3년물뿐 아니라 곡률에서도 상승·하락 계수가 산업별로 어떻게 나타나는지.

In [75]:
rows_curv_beta = []
for ind in industries:
    g = panel[panel["industry"] == ind]
    m = fit_asymmetric(g, "d_curv_up", "d_curv_down")
    ci = m.conf_int(alpha=0.05)
    for direction, var in [("up", "d_curv_up"), ("down", "d_curv_down")]:
        rows_curv_beta.append({
            "model": "Curvature_Coef",
            "direction": direction,
            "industry": ind,
            "beta": m.params[var],
            "se_hac": m.bse[var],
            "p_value": m.pvalues[var],
            "ci_lower_95": ci.loc[var, 0],
            "ci_upper_95": ci.loc[var, 1],
            "nobs": int(m.nobs),
            "r_squared": m.rsquared,
        })

key_factor_asymmetry_tests = add_bh_correction(
    pd.DataFrame(rows_curv_beta), group_cols=("model", "direction")
)

curv_path = TABDIR / "key_factor_asymmetry_tests.csv"
key_factor_asymmetry_tests.to_csv(curv_path, index=False, encoding="utf-8-sig")
display(key_factor_asymmetry_tests.round(6))

,model,direction,industry,beta,se_hac,p_value,ci_lower_95,ci_upper_95,nobs,r_squared,p_value_bh,bh_n_tests,bh_family,significant_5pct_bh
0,Curvature_Coef,up,건설업,-0.000732,0.003106,0.813582,-0.006819,0.005355,2451,0.398371,0.949179,7,Curvature_Coef | up,False
1,Curvature_Coef,down,건설업,0.002717,0.003055,0.373812,-0.003271,0.008706,2451,0.398371,0.654171,7,Curvature_Coef | down,False
2,Curvature_Coef,up,금융업,0.000469,0.001753,0.789130,-0.002967,0.003905,2451,0.621801,0.949179,7,Curvature_Coef | up,False
3,Curvature_Coef,down,금융업,0.000775,0.001675,0.643743,-0.002509,0.004058,2451,0.621801,0.896478,7,Curvature_Coef | down,False
4,Curvature_Coef,up,운수장비,0.006420,0.002316,0.005579,0.001880,0.010959,2451,0.472245,0.039051,7,Curvature_Coef | up,True
5,Curvature_Coef,down,운수장비,0.002577,0.002409,0.284874,-0.002146,0.007299,2451,0.472245,0.654171,7,Curvature_Coef | down,False
6,Curvature_Coef,up,의약품,-0.006412,0.002777,0.020943,-0.011855,-0.000969,2451,0.243281,0.048867,7,Curvature_Coef | up,True
7,Curvature_Coef,down,의약품,-0.007665,0.003220,0.017303,-0.013977,-0.001353,2451,0.243281,0.121119,7,Curvature_Coef | down,False
8,Curvature_Coef,up,전기전자,-0.000074,0.001674,0.964871,-0.003354,0.003207,2451,0.719329,0.964871,7,Curvature_Coef | up,False
9,Curvature_Coef,down,전기전자,0.001358,0.001526,0.373306,-0.001632,0.004348,2451,0.719329,0.654171,7,Curvature_Coef | down,False


**관찰.** 곡률의 상승·하락 계수 부호와 크기는 산업마다 다르며, H2 본문(FINAL)의 곡률 방향분해 결과와 같은 방향을 가리킨다.

**4.A.4 3년물·곡률 산업 간 차이 비교.** 확인할 내용: 방향 분리 후 산업 간 계수 이질성(F검정)이 3년물과 곡률에서 각각 어떻게 나타나는지.

In [76]:
# run_industry_difference_test -- nb06 cell 68 원문 그대로
def run_industry_difference_test(data, test_vars, model_name,
                                 ret_col="industry_return",
                                 mkt_col="kospi_return",
                                 industry_col="industry",
                                 date_col="date"):
    """H0: 모든 산업의 계수가 같다. 요인별로 F-검정한다."""
    test_vars = list(test_vars)
    d = data.dropna(subset=[ret_col, mkt_col, industry_col, date_col] + test_vars).copy()

    date_codes = pd.factorize(d[date_col])[0]
    if len(np.unique(date_codes)) < 2:
        raise ValueError("날짜 그룹이 2개 이상이어야 클러스터 SE를 쓸 수 있습니다.")

    dummies = pd.get_dummies(d[industry_col], prefix="ind", dtype=float)
    blocks = [dummies, dummies.mul(d[mkt_col].values, axis=0).add_prefix("mkt_")]

    for v in test_vars:
        blocks.append(dummies.mul(d[v].values, axis=0).add_prefix(f"{v}__"))

    X = pd.concat(blocks, axis=1)
    names = list(X.columns)

    model = sm.OLS(d[ret_col].values, X.values).fit(
        cov_type="cluster", cov_kwds={"groups": date_codes})

    rows = []
    for v in test_vars:
        idx = [i for i, c in enumerate(names) if c.startswith(f"{v}__")]

        # "첫 산업 계수 = 나머지 산업 계수" 제약행렬
        R = np.zeros((len(idx) - 1, len(names)))
        for r, j in enumerate(idx[1:]):
            R[r, idx[0]] = 1.0
            R[r, j] = -1.0

        t = model.f_test(R)

        rows.append({
            "model": model_name,
            "variable": v,
            "f_stat": float(t.fvalue),
            "p_value": float(t.pvalue),
            "df_num": int(len(idx) - 1),
            "n_industries": int(dummies.shape[1]),
            "n_obs": int(model.nobs),
            "n_dates": int(len(np.unique(date_codes))),
            "significant_5pct": bool(t.pvalue < 0.05),
        })

    return pd.DataFrame(rows)


# 검증: 3년물 하나만 넣어 src/models.py 의 기존 함수와 결과가 같은지 확인한다.
from src.models import cross_industry_Ftest

_check_new = run_industry_difference_test(panel, ["d_y3"], "check")
f_new = float(_check_new.loc[0, "f_stat"])

_long_df = panel.rename(columns={
    "industry_return": "ret", "kospi_return": "mkt", "d_y3": "dY",
})
_ft_old = cross_industry_Ftest(_long_df, date_col="date")
f_old = float(_ft_old.fvalue)

f_diff = abs(f_new - f_old)
print(f"run_industry_difference_test F = {f_new:.10f}")
print(f"cross_industry_Ftest F         = {f_old:.10f}")
print(f"차이 = {f_diff:.3e}")

assert f_diff < 1e-8, f"F값 불일치: {f_diff}"
print("-> 일치. run_industry_difference_test는 기존 사양을 그대로 따른다.")

run_industry_difference_test F = 4.1891559371
cross_industry_Ftest F         = 4.1891559371
차이 = 0.000e+00
-> 일치. run_industry_difference_test는 기존 사양을 그대로 따른다.


In [77]:
industry_difference_tests = pd.concat([
    run_industry_difference_test(panel, ["d_y3_up", "d_y3_down"], "Y3_Asymmetry"),
    run_industry_difference_test(panel, ["d_curv_up", "d_curv_down"], "Curvature_Asymmetry"),
], ignore_index=True)

industry_difference_tests = add_bh_correction(industry_difference_tests, group_cols=("model",))

diff_path = TABDIR / "industry_difference_tests.csv"
industry_difference_tests.to_csv(diff_path, index=False, encoding="utf-8-sig")
display(industry_difference_tests.round(6))

,model,variable,f_stat,p_value,df_num,n_industries,n_obs,n_dates,significant_5pct,p_value_bh,bh_n_tests,bh_family,significant_5pct_bh
0,Y3_Asymmetry,d_y3_up,3.516189,0.001817,6,7,17157,2451,True,0.003633,2,Y3_Asymmetry,True
1,Y3_Asymmetry,d_y3_down,3.130383,0.004652,6,7,17157,2451,True,0.004652,2,Y3_Asymmetry,True
2,Curvature_Asymmetry,d_curv_up,3.007698,0.006245,6,7,17157,2451,True,0.012491,2,Curvature_Asymmetry,True
3,Curvature_Asymmetry,d_curv_down,1.374417,0.221086,6,7,17157,2451,False,0.221086,2,Curvature_Asymmetry,False


In [78]:
label_map = {
    ("Y3_Asymmetry", "d_y3_up"): "3년물 상승",
    ("Y3_Asymmetry", "d_y3_down"): "3년물 하락",
    ("Curvature_Asymmetry", "d_curv_up"): "곡률 상승",
    ("Curvature_Asymmetry", "d_curv_down"): "곡률 하락",
}
dd = industry_difference_tests.copy()
dd["label"] = [label_map[(m, v)] for m, v in zip(dd["model"], dd["variable"])]
order = ["3년물 상승", "3년물 하락", "곡률 상승", "곡률 하락"]
dd = dd.set_index("label").loc[order].reset_index()

# F값은 부호가 없는 값이므로 부호색(SIGN_COLORS neg/pos)을 쓰지 않는다.
# 03_asymmetry_wald_comparison.png 와 동일한 중립색(#5c8cbc)으로 BH 5% 유의를 표시하고,
# 비유의는 config.SIGN_COLORS["ns"] 로 표시한다.
SIG_COLOR = "#5c8cbc"

fig, ax = plt.subplots(figsize=(9.5, 5.4), dpi=130)
colors = [SIG_COLOR if p < 0.05 else config.SIGN_COLORS["ns"] for p in dd["p_value_bh"]]
bars = ax.bar(dd["label"], dd["f_stat"], color=colors, width=0.55, alpha=0.92)
for b, f, p in zip(bars, dd["f_stat"], dd["p_value_bh"]):
    ax.text(b.get_x() + b.get_width() / 2, f + 0.06, f"F={f:.2f}\nBH p={p:.4f}",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold" if p < 0.05 else "normal")

ax.set_ylabel("산업 간 차이 F값 (날짜 클러스터 SE)", fontsize=11)
ax.set_title("산업 간 계수 이질성 -- 방향 분리 후 비교", fontsize=14, fontweight="bold", pad=14)
ax.set_ylim(0, dd["f_stat"].max() * 1.35)
ax.grid(axis="y", ls=":", color="gray", alpha=0.5)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor=SIG_COLOR, label="BH 5% 유의"),
    plt.Rectangle((0, 0), 1, 1, facecolor=config.SIGN_COLORS["ns"], label="비유의"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=9.5, framealpha=1)

fig.tight_layout()

fig04_path = FIGDIR / "04_industry_difference_comparison.png"
fig.savefig(fig04_path, dpi=200, bbox_inches="tight")
plt.show()

/sessions/rcw-01lahd1zbbtcdxvfafycm7kq/tmp/ipykernel_6/2006182543.py:17: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(9.5, 5.4), dpi=130)


**관찰.** 방향 분리 후에도 산업 간 차이 F검정은 일부 조합에서만 BH 5%를 충족한다. 3년물과 곡률의 결과 패턴은 같지 않다.

**4.A.5 방향더미를 더한 절편 강건성.** 확인할 내용: 표본을 무변화일 제외로 좁히고 상승 방향더미를 추가해도 건설업의 비대칭 결과가 유지되는지.

In [79]:
import warnings

# PART D·E 산출물은 본문과 같은 폴더에 접두로 구분해 저장한다(프로젝트 파일명 규칙).
ROBUST_DIR = config.H2_DIR
ROBUST_TABDIR = TABDIR
ROBUST_FIGDIR = FIGDIR


def append_summary_section(section_text, marker):
    """summary.md 에 절을 덧붙인다. 같은 marker 절이 있으면 교체한다(재실행 안전)."""
    path = config.H2_DIR / "summary.md"
    base = path.read_text(encoding="utf-8") if path.exists() else ""
    cut = base.find(marker)
    if cut != -1:                      # 재실행 시 같은 절이 중복되지 않게 잘라낸다
        base = base[:cut]
    path.write_text(base.rstrip() + "\n\n" + section_text.strip() + "\n", encoding="utf-8")
    return path


def round_numeric(df, n=6):
    """숫자 열만 반올림한다. 날짜 열에 round를 걸면 경고만 나고 효과가 없다."""
    return df.round(dict.fromkeys(df.select_dtypes("number").columns, n))


print("PART D 설정 완료 | 표:", config._rel(ROBUST_TABDIR), "| 그림:", config._rel(ROBUST_FIGDIR))


PART D 설정 완료 | 표: results/H2_asymmetry_regime/tables | 그림: results/H2_asymmetry_regime/figures


In [80]:
# ============================================================
# 3단계. 공통 함수
# ============================================================

def add_bh_correction(
    result,
    group_cols=("model",),
    p_col="p_value",
    alpha=BH_ALPHA,
):
    """지정한 검정족별로 BH 보정을 적용한다."""
    out = result.copy()
    group_cols = list(group_cols)

    out["p_value_bh"] = np.nan
    out["bh_n_tests"] = 0

    # 길이 1 리스트로 그룹화하면 pandas 4에서 키가 스칼라 대신 튜플이 된다. 단일 열이면 스칼라로 넘긴다.
    _by = group_cols[0] if len(group_cols) == 1 else group_cols
    for _, _sub in out.groupby(
        _by,
        dropna=False,
    ):
        idx = _sub.index

        pvals = out.loc[
            idx,
            p_col,
        ].to_numpy(dtype=float)

        out.loc[idx, "p_value_bh"] = multipletests(
            pvals,
            alpha=alpha,
            method="fdr_bh",
        )[1]

        out.loc[idx, "bh_n_tests"] = len(pvals)

    out["bh_family"] = (
        out[group_cols]
        .astype(str)
        .agg(" | ".join, axis=1)
    )

    out["significant_5pct_bh"] = (
        out["p_value_bh"] < alpha
    )

    return out


def fit_asymmetric(
    data,
    up_col,
    down_col,
    hac_lags=HAC_LAGS,
    include_direction_dummy=False,
    drop_zero_for_dummy=True,
):
    """산업 하나의 상승·하락 비대칭 회귀를 추정한다."""

    use_cols = [
        "industry_return",
        "kospi_return",
        up_col,
        down_col,
    ]

    sample = (
        data[use_cols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )

    x_cols = [
        "kospi_return",
        up_col,
        down_col,
    ]

    if include_direction_dummy:

        if drop_zero_for_dummy:
            sample = sample[
                (sample[up_col] > 0)
                | (sample[down_col] < 0)
            ].copy()

        sample["direction_up_dummy"] = (
            sample[up_col] > 0
        ).astype(float)

        x_cols = [
            "kospi_return",
            "direction_up_dummy",
            up_col,
            down_col,
        ]

    X = sm.add_constant(
        sample[x_cols],
        has_constant="add",
    )

    return sm.OLS(
        sample["industry_return"],
        X,
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": hac_lags,
        },
    )


def industry_wald_table(
    data,
    up_col,
    down_col,
    model_name,
    include_direction_dummy=False,
    drop_zero_for_dummy=True,
):
    """
    7개 산업의 계수와
    H0: beta_up = beta_down
    Wald 검정을 계산한다.
    """

    rows = []

    industries = sorted(
        data["industry"]
        .dropna()
        .unique()
    )

    for industry in industries:

        group = data[
            data["industry"] == industry
        ]

        model = fit_asymmetric(
            group,
            up_col,
            down_col,
            include_direction_dummy=include_direction_dummy,
            drop_zero_for_dummy=drop_zero_for_dummy,
        )

        ci = model.conf_int(alpha=0.05)

        wald = model.f_test(
            f"{up_col} = {down_col}"
        )

        row = {
            "model": model_name,
            "industry": industry,

            "beta_up": model.params[up_col],
            "se_up": model.bse[up_col],
            "ci_lower_up": ci.loc[up_col, 0],
            "ci_upper_up": ci.loc[up_col, 1],
            "p_up": model.pvalues[up_col],

            "beta_down": model.params[down_col],
            "se_down": model.bse[down_col],
            "ci_lower_down": ci.loc[down_col, 0],
            "ci_upper_down": ci.loc[down_col, 1],
            "p_down": model.pvalues[down_col],

            "beta_difference": (
                model.params[up_col]
                - model.params[down_col]
            ),

            "f_stat": float(
                np.asarray(
                    wald.fvalue
                ).squeeze()
            ),

            "p_value": float(
                np.asarray(
                    wald.pvalue
                ).squeeze()
            ),

            "nobs": int(model.nobs),
            "r_squared": model.rsquared,

            "direction_dummy_included":
                include_direction_dummy,

            "direction_dummy_beta": (
                model.params["direction_up_dummy"]
                if "direction_up_dummy"
                in model.params.index
                else np.nan
            ),

            "direction_dummy_p": (
                model.pvalues["direction_up_dummy"]
                if "direction_up_dummy"
                in model.params.index
                else np.nan
            ),
        }

        rows.append(row)

    result = pd.DataFrame(rows)

    return add_bh_correction(
        result,
        group_cols=("model",),
        p_col="p_value",
    )


def coefficient_table(
    data,
    up_col,
    down_col,
    model_name,
):
    """상승·하락 개별 계수의 p값을 14행으로 반환한다."""

    rows = []

    industries = sorted(
        data["industry"]
        .dropna()
        .unique()
    )

    for industry in industries:

        group = data[
            data["industry"] == industry
        ]

        model = fit_asymmetric(
            group,
            up_col,
            down_col,
        )

        ci = model.conf_int(alpha=0.05)

        for direction, variable in [
            ("up", up_col),
            ("down", down_col),
        ]:
            rows.append(
                {
                    "model": model_name,
                    "direction": direction,
                    "industry": industry,
                    "variable": variable,
                    "beta": model.params[variable],
                    "se": model.bse[variable],
                    "p_value": model.pvalues[variable],
                    "ci_lower_95": ci.loc[variable, 0],
                    "ci_upper_95": ci.loc[variable, 1],
                }
            )

    return pd.DataFrame(rows)


def _build_pooled_design(
    data,
    up_col,
    down_col,
):
    """전 산업 공동검정 설계행렬과 제약행렬을 만든다."""

    required = [
        "date",
        "industry",
        "industry_return",
        "kospi_return",
        up_col,
        down_col,
    ]

    d = (
        data[required]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .sort_values(["date", "industry"])
        .copy()
    )

    dummies = pd.get_dummies(
        d["industry"],
        prefix="ind",
        dtype=float,
    )

    X = pd.concat(
        [
            dummies,

            dummies.mul(
                d["kospi_return"].to_numpy(),
                axis=0,
            ).add_prefix("mkt_"),

            dummies.mul(
                d[up_col].to_numpy(),
                axis=0,
            ).add_prefix(f"{up_col}__"),

            dummies.mul(
                d[down_col].to_numpy(),
                axis=0,
            ).add_prefix(f"{down_col}__"),
        ],
        axis=1,
    )

    names = list(X.columns)

    up_idx = [
        i
        for i, name in enumerate(names)
        if name.startswith(f"{up_col}__")
    ]

    down_idx = [
        i
        for i, name in enumerate(names)
        if name.startswith(f"{down_col}__")
    ]

    assert (
        len(up_idx)
        == len(down_idx)
        == d["industry"].nunique()
    )

    R = np.zeros(
        (
            len(up_idx),
            len(names),
        )
    )

    for row, (u_idx, d_idx) in enumerate(
        zip(up_idx, down_idx)
    ):
        R[row, u_idx] = 1.0
        R[row, d_idx] = -1.0

    date_codes = pd.factorize(
        d["date"],
        sort=True,
    )[0]

    return d, X, R, date_codes


def joint_symmetry_test(
    data,
    up_col,
    down_col,
    variable_name,
    covariance="date_cluster",
    maxlags=HAC_LAGS,
):
    """
    모든 산업에서
    H0: beta_up = beta_down
    을 공동검정한다.
    """

    d, X, R, date_codes = _build_pooled_design(
        data,
        up_col,
        down_col,
    )

    base_model = sm.OLS(
        d["industry_return"].to_numpy(),
        X.to_numpy(),
    )

    if covariance == "date_cluster":

        result = base_model.fit(
            cov_type="cluster",
            cov_kwds={
                "groups": date_codes,
                "use_correction": True,
            },
        )

        covariance_label = "Date-cluster"

    elif covariance == "driscoll_kraay":

        result = base_model.fit(
            cov_type="hac-groupsum",
            cov_kwds={
                "time": date_codes,
                "maxlags": maxlags,
                "use_correction": "cluster",

                # Driscoll-Kraay p값이 NaN이 되는
                # 자유도 보정 문제를 방지
                "df_correction": False,
            },
        )

        covariance_label = (
            f"Driscoll-Kraay lag={maxlags}"
        )

    else:
        raise ValueError(
            "지원하지 않는 covariance입니다."
        )

    wald = result.f_test(R)

    f_stat = float(
        np.asarray(
            wald.fvalue
        ).squeeze()
    )

    p_value = float(
        np.asarray(
            wald.pvalue
        ).squeeze()
    )

    if not np.isfinite(p_value):
        raise ValueError(
            f"{covariance_label}: "
            "공동검정 p-value를 계산하지 못했습니다."
        )

    return {
        "variable": variable_name,
        "covariance": covariance,
        "covariance_label": covariance_label,

        "maxlags": (
            np.nan
            if covariance == "date_cluster"
            else maxlags
        ),

        "n_restrictions": R.shape[0],
        "f_stat": f_stat,
        "p_value": p_value,
        "nobs": int(result.nobs),
        "n_dates": int(d["date"].nunique()),
    }


def extract_industry_row(
    table,
    industry="건설업",
):
    """결과표에서 특정 산업 한 행을 가져온다."""

    match = table[
        table["industry"] == industry
    ]

    if len(match) != 1:
        raise ValueError(
            f"{industry} 행이 정확히 "
            f"1개가 아닙니다: {len(match)}개"
        )

    return match.iloc[0]


print("3단계 공통 함수 정의 완료")

3단계 공통 함수 정의 완료


In [81]:
nonzero_panel = panel[
    (panel["d_y3_up"] > 0) | (panel["d_y3_down"] < 0)
].copy()

direction_specs = []

baseline_full = industry_wald_table(
    panel,
    "d_y3_up",
    "d_y3_down",
    model_name="baseline_full",
)
direction_specs.append(baseline_full)

baseline_nonzero = industry_wald_table(
    nonzero_panel,
    "d_y3_up",
    "d_y3_down",
    model_name="nonzero_no_dummy",
)
direction_specs.append(baseline_nonzero)

dummy_nonzero = industry_wald_table(
    nonzero_panel,
    "d_y3_up",
    "d_y3_down",
    model_name="nonzero_with_up_dummy",
    include_direction_dummy=True,
    drop_zero_for_dummy=True,
)
direction_specs.append(dummy_nonzero)

direction_dummy_comparison = pd.concat(
    direction_specs,
    ignore_index=True,
)

direction_dummy_comparison.to_csv(
    ROBUST_TABDIR / "robustness_direction_dummy_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

construction_direction_dummy = direction_dummy_comparison[
    direction_dummy_comparison["industry"] == "건설업"
].copy()

display(
    construction_direction_dummy[
        [
            "model",
            "beta_up",
            "beta_down",
            "direction_dummy_beta",
            "direction_dummy_p",
            "f_stat",
            "p_value",
            "p_value_bh",
            "nobs",
        ]
    ].round(6)
)

,model,beta_up,beta_down,direction_dummy_beta,direction_dummy_p,f_stat,p_value,p_value_bh,nobs
0,baseline_full,-0.021607,0.031027,NaN,NaN,10.417801,0.001265,0.008852,2451
7,nonzero_no_dummy,-0.021568,0.031028,NaN,NaN,10.246743,0.001387,0.009710,2400
14,nonzero_with_up_dummy,-0.027109,0.024652,0.000707,0.338407,10.066691,0.001529,0.010702,2400


**관찰.** 방향더미를 추가한 사양에서도 건설업의 상승·하락 계수 패턴은 크게 흔들리지 않는다.

**4.A.6 크기계수의 조건별 산업 간 차이.** 확인할 내용: |Δy3|(크기)와 Δy3(방향)를 한 모형에 함께 넣었을 때 산업 간 차이가 극단값 제거·기간 분할·산업 제외 등 조건을 바꿔도 유지되는지.

In [82]:
panel = pd.read_csv(config.EXTENDED_PANEL_PATH, encoding="utf-8-sig")
panel["date"] = pd.to_datetime(panel["date"], errors="coerce")

required_panel_cols = [
    "date", "industry", "industry_return", "kospi_return",
    "d_y3", "ns_curvature", "credit_spread",
]
missing_cols = [c for c in required_panel_cols if c not in panel.columns]
if missing_cols:
    raise ValueError("확장 패널에 필요한 열이 없습니다: " + ", ".join(missing_cols))

panel = panel.sort_values(["date", "industry"]).reset_index(drop=True)

assert "credit_spread" in panel.columns, "구버전 패널입니다. data/processed/ 를 갱신하세요."
assert abs(panel["ns_curvature"].mean() + 0.3999) < 0.01, \
    "lambda=0.5 패널이 아닙니다. data/processed/ 를 갱신하세요."

# 방향 변수(STEP7과 동일)와 크기 변수를 함께 만든다.
panel["d_y3_up"] = panel["d_y3"].clip(lower=0)
panel["d_y3_down"] = panel["d_y3"].clip(upper=0)
panel["abs_d_y3"] = panel["d_y3"].abs()

valid = panel["d_y3"].notna()
# 분해 항등식: Δy = Δy_up + Δy_down, |Δy| = Δy_up - Δy_down
assert np.allclose(panel.loc[valid, "d_y3"],
                   panel.loc[valid, "d_y3_up"] + panel.loc[valid, "d_y3_down"])
assert np.allclose(panel.loc[valid, "abs_d_y3"],
                   panel.loc[valid, "d_y3_up"] - panel.loc[valid, "d_y3_down"])
assert (panel["d_y3_up"].dropna() >= 0).all()
assert (panel["d_y3_down"].dropna() <= 0).all()

INDUSTRIES = sorted(panel["industry"].dropna().unique())

print("ns_curvature 평균:", round(panel["ns_curvature"].mean(), 4),
      "| config.NS_LAMBDA =", config.NS_LAMBDA)
print("분석 기간:", panel["date"].min().date(), "~", panel["date"].max().date())
print("산업 수:", len(INDUSTRIES), "| 산업 목록:", INDUSTRIES)
print("변수 생성·검증 완료")
display(panel[["date", "industry", "d_y3", "d_y3_up", "d_y3_down", "abs_d_y3"]].head())

ns_curvature 평균: -0.3999 | config.NS_LAMBDA = 0.5
분석 기간: 2016-01-04 ~ 2025-12-30
산업 수: 7 | 산업 목록: ['건설업', '금융업', '운수장비', '의약품', '전기전자', '철강금속', '화학']
변수 생성·검증 완료


,date,industry,d_y3,d_y3_up,d_y3_down,abs_d_y3
0,2016-01-04,건설업,NaN,NaN,NaN,NaN
1,2016-01-04,금융업,NaN,NaN,NaN,NaN
2,2016-01-04,운수장비,NaN,NaN,NaN,NaN
3,2016-01-04,의약품,NaN,NaN,NaN,NaN
4,2016-01-04,전기전자,NaN,NaN,NaN,NaN


In [83]:
def fit_updown(data, hac_lags=HAC_LAGS):
    """R = a + g*kospi + b_up*Δy_up + b_down*Δy_down (HAC). STEP7과 동일 사양."""
    cols = ["industry_return", "kospi_return", "d_y3_up", "d_y3_down"]
    s = data[cols].replace([np.inf, -np.inf], np.nan).dropna()
    X = sm.add_constant(s[["kospi_return", "d_y3_up", "d_y3_down"]], has_constant="add")
    return sm.OLS(s["industry_return"], X).fit(
        cov_type="HAC", cov_kwds={"maxlags": hac_lags})


def fit_absdir(data, hac_lags=HAC_LAGS):
    """R = a + g*kospi + b_abs*|Δy| + b_dir*Δy (HAC). 위 모형의 재모수화."""
    cols = ["industry_return", "kospi_return", "abs_d_y3", "d_y3"]
    s = data[cols].replace([np.inf, -np.inf], np.nan).dropna()
    X = sm.add_constant(s[["kospi_return", "abs_d_y3", "d_y3"]], has_constant="add")
    return sm.OLS(s["industry_return"], X).fit(
        cov_type="HAC", cov_kwds={"maxlags": hac_lags})


rows = []
for ind in INDUSTRIES:
    g = panel[panel["industry"] == ind]
    m1, m2 = fit_updown(g), fit_absdir(g)
    b_abs_implied = (m1.params["d_y3_up"] - m1.params["d_y3_down"]) / 2
    b_dir_implied = (m1.params["d_y3_up"] + m1.params["d_y3_down"]) / 2
    rows.append({
        "industry": ind,
        "beta_up": m1.params["d_y3_up"],
        "beta_down": m1.params["d_y3_down"],
        "beta_abs_implied": b_abs_implied,
        "beta_abs_direct": m2.params["abs_d_y3"],
        "abs_diff": abs(b_abs_implied - m2.params["abs_d_y3"]),
        "beta_dir_implied": b_dir_implied,
        "beta_dir_direct": m2.params["d_y3"],
        "dir_diff": abs(b_dir_implied - m2.params["d_y3"]),
        "r2_updown": m1.rsquared,
        "r2_absdir": m2.rsquared,
        "r2_diff": abs(m1.rsquared - m2.rsquared),
        "nobs": int(m2.nobs),
    })

uncert_reparameterization = pd.DataFrame(rows)

# 항등식이 수치오차 범위에서 성립하는지 확인한다.
assert (uncert_reparameterization["abs_diff"] < 1e-10).all()
assert (uncert_reparameterization["dir_diff"] < 1e-10).all()
assert (uncert_reparameterization["r2_diff"] < 1e-12).all()

uncert_reparameterization.to_csv(
    TABDIR / "uncert_reparameterization.csv", index=False, encoding="utf-8-sig")
print("저장:", config._rel(TABDIR / "uncert_reparameterization.csv"))
display(uncert_reparameterization.round(6))

저장: results/H2_asymmetry_regime/tables/uncert_reparameterization.csv


,industry,beta_up,beta_down,beta_abs_implied,beta_abs_direct,abs_diff,beta_dir_implied,beta_dir_direct,dir_diff,r2_updown,r2_absdir,r2_diff,nobs
0,건설업,-0.021607,0.031027,-0.026317,-0.026317,0.0,0.004710,0.004710,0.0,0.400090,0.400090,0.0,2451
1,금융업,0.005664,0.017479,-0.005908,-0.005908,0.0,0.011571,0.011571,0.0,0.623040,0.623040,0.0,2451
2,운수장비,0.017446,0.016986,0.000230,0.000230,0.0,0.017216,0.017216,0.0,0.471961,0.471961,0.0,2451
3,의약품,-0.017156,-0.038179,0.010512,0.010512,0.0,-0.027667,-0.027667,0.0,0.243546,0.243546,0.0,2451
4,전기전자,0.007840,-0.005425,0.006632,0.006632,0.0,0.001208,0.001208,0.0,0.719434,0.719434,0.0,2451
5,철강금속,0.028131,0.008412,0.009859,0.009859,0.0,0.018272,0.018272,0.0,0.383669,0.383669,0.0,2451
6,화학,0.000943,0.004369,-0.001713,-0.001713,0.0,0.002656,0.002656,0.0,0.643654,0.643654,0.0,2451


In [84]:
# add_bh_correction -- nb06(06_yield_curve_exposure.ipynb) cell 42 원문 그대로.
def add_bh_correction(result, group_cols=("model", "factor"),
                      p_col="p_value", alpha=0.05):
    out = result.copy()
    group_cols = list(group_cols)

    out["p_value_bh"] = np.nan
    out["bh_n_tests"] = 0

    # 길이 1 리스트로 그룹화하면 pandas 4에서 키가 스칼라 대신 튜플이 된다. 단일 열이면 스칼라로 넘긴다.
    _by = group_cols[0] if len(group_cols) == 1 else group_cols
    for _, _sub in out.groupby(_by):
        idx = _sub.index
        pvals = out.loc[idx, p_col].to_numpy(dtype=float)
        out.loc[idx, "p_value_bh"] = multipletests(pvals, method="fdr_bh")[1]
        out.loc[idx, "bh_n_tests"] = len(pvals)

    out["bh_family"] = out[group_cols].astype(str).agg(" | ".join, axis=1)
    out["significant_5pct_bh"] = out["p_value_bh"] < alpha

    return out


rows = []
for ind in INDUSTRIES:
    g = panel[panel["industry"] == ind]
    m = fit_absdir(g)
    ci = m.conf_int(alpha=0.05)
    for comp, var in [("size", "abs_d_y3"), ("direction", "d_y3")]:
        # p값은 STEP7·STEP7a와 같은 Wald F 기준을 쓴다. t^2 = F 이므로 같은 검정이며,
        # 참조분포만 달라 t 기준으로 쓰면 소수점 넷째 자리에서 값이 어긋난다.
        rows.append({
            "model": "Y3_SizeDirection",
            "industry": ind,
            "component": comp,
            "beta": m.params[var],
            "se_hac": m.bse[var],
            "t_value": m.tvalues[var],
            "p_value": float(m.f_test(f"{var}=0").pvalue),
            "ci_lower_95": ci.loc[var, 0],
            "ci_upper_95": ci.loc[var, 1],
            "nobs": int(m.nobs),
            "r_squared": m.rsquared,
            "hac_lags": HAC_LAGS,
        })

uncert_size_direction_betas = add_bh_correction(
    pd.DataFrame(rows), group_cols=("model", "component"), alpha=BH_ALPHA)

uncert_size_direction_betas.to_csv(
    TABDIR / "uncert_size_direction_betas.csv", index=False, encoding="utf-8-sig")
print("저장:", config._rel(TABDIR / "uncert_size_direction_betas.csv"))
display(uncert_size_direction_betas.round(6))

저장: results/H2_asymmetry_regime/tables/uncert_size_direction_betas.csv


,model,industry,component,beta,se_hac,t_value,p_value,ci_lower_95,ci_upper_95,nobs,r_squared,hac_lags,p_value_bh,bh_n_tests,bh_family,significant_5pct_bh
0,Y3_SizeDirection,건설업,size,-0.026317,0.008154,-3.227662,0.001265,-0.042297,-0.010336,2451,0.400090,5,0.008852,7,Y3_SizeDirection | size,True
1,Y3_SizeDirection,건설업,direction,0.004710,0.008032,0.586364,0.557685,-0.011033,0.020452,2451,0.400090,5,0.685433,7,Y3_SizeDirection | direction,False
2,Y3_SizeDirection,금융업,size,-0.005908,0.005598,-1.055347,0.291371,-0.016880,0.005064,2451,0.623040,5,0.429648,7,Y3_SizeDirection | size,False
3,Y3_SizeDirection,금융업,direction,0.011571,0.003793,3.050880,0.002306,0.004138,0.019005,2451,0.623040,5,0.006875,7,Y3_SizeDirection | direction,True
4,Y3_SizeDirection,운수장비,size,0.000230,0.007203,0.031957,0.974509,-0.013887,0.014347,2451,0.471961,5,0.974509,7,Y3_SizeDirection | size,False
5,Y3_SizeDirection,운수장비,direction,0.017216,0.005784,2.976266,0.002946,0.005879,0.028554,2451,0.471961,5,0.006875,7,Y3_SizeDirection | direction,True
6,Y3_SizeDirection,의약품,size,0.010512,0.009966,1.054751,0.291644,-0.009021,0.030045,2451,0.243546,5,0.429648,7,Y3_SizeDirection | size,False
7,Y3_SizeDirection,의약품,direction,-0.027667,0.007892,-3.505679,0.000464,-0.043136,-0.012199,2451,0.243546,5,0.003245,7,Y3_SizeDirection | direction,True
8,Y3_SizeDirection,전기전자,size,0.006632,0.005156,1.286412,0.198421,-0.003473,0.016737,2451,0.719434,5,0.429648,7,Y3_SizeDirection | size,False
9,Y3_SizeDirection,전기전자,direction,0.001208,0.004184,0.288630,0.772889,-0.006993,0.009409,2451,0.719434,5,0.772889,7,Y3_SizeDirection | direction,False


In [85]:
# run_industry_difference_test -- nb06 cell 68 원문 그대로
def run_industry_difference_test(data, test_vars, model_name,
                                 ret_col="industry_return",
                                 mkt_col="kospi_return",
                                 industry_col="industry",
                                 date_col="date"):
    """H0: 모든 산업의 계수가 같다. 요인별로 F-검정한다."""
    test_vars = list(test_vars)
    d = data.dropna(subset=[ret_col, mkt_col, industry_col, date_col] + test_vars).copy()

    date_codes = pd.factorize(d[date_col])[0]
    if len(np.unique(date_codes)) < 2:
        raise ValueError("날짜 그룹이 2개 이상이어야 클러스터 SE를 쓸 수 있습니다.")

    dummies = pd.get_dummies(d[industry_col], prefix="ind", dtype=float)
    blocks = [dummies, dummies.mul(d[mkt_col].values, axis=0).add_prefix("mkt_")]

    for v in test_vars:
        blocks.append(dummies.mul(d[v].values, axis=0).add_prefix(f"{v}__"))

    X = pd.concat(blocks, axis=1)
    names = list(X.columns)

    model = sm.OLS(d[ret_col].values, X.values).fit(
        cov_type="cluster", cov_kwds={"groups": date_codes})

    rows = []
    for v in test_vars:
        idx = [i for i, c in enumerate(names) if c.startswith(f"{v}__")]

        # "첫 산업 계수 = 나머지 산업 계수" 제약행렬
        R = np.zeros((len(idx) - 1, len(names)))
        for r, j in enumerate(idx[1:]):
            R[r, idx[0]] = 1.0
            R[r, j] = -1.0

        t = model.f_test(R)

        rows.append({
            "model": model_name,
            "variable": v,
            "f_stat": float(t.fvalue),
            "p_value": float(t.pvalue),
            "df_num": int(len(idx) - 1),
            "n_industries": int(dummies.shape[1]),
            "n_obs": int(model.nobs),
            "n_dates": int(len(np.unique(date_codes))),
            "significant_5pct": bool(t.pvalue < 0.05),
        })

    return pd.DataFrame(rows)


# 크기·방향 두 변수를 한 모형에 함께 넣어 검정한다.
diff_new = run_industry_difference_test(
    panel, ["abs_d_y3", "d_y3"], "Y3_SizeDirection")
# 한 모형에서 두 변수를 검정하므로 STEP7과 같은 방식으로 BH 보정한다
# (07의 industry_difference_tests.csv 도 검정족 2개로 보정했다).
diff_new = add_bh_correction(diff_new, group_cols=("model",), alpha=BH_ALPHA)
diff_new["source"] = "07b (신규)"

# STEP7 기존 결과(읽기 전용)를 참고로 붙인다.
_ref = TABDIR / "industry_difference_tests.csv"
if _ref.exists():
    ref = pd.read_csv(_ref, encoding="utf-8-sig")
    ref = ref[ref["model"] == "Y3_Asymmetry"][
        ["model", "variable", "f_stat", "p_value", "df_num",
         "n_industries", "n_obs", "n_dates", "significant_5pct",
         "p_value_bh", "bh_n_tests", "significant_5pct_bh"]].copy()
    ref["source"] = "07 (기존, 참고)"
    uncert_industry_difference = pd.concat([diff_new, ref], ignore_index=True)
else:
    uncert_industry_difference = diff_new

uncert_industry_difference.to_csv(
    TABDIR / "uncert_industry_difference.csv", index=False, encoding="utf-8-sig")
print("저장:", config._rel(TABDIR / "uncert_industry_difference.csv"))
display(uncert_industry_difference.round(6))

저장: results/H2_asymmetry_regime/tables/uncert_industry_difference.csv


,model,variable,f_stat,p_value,df_num,n_industries,n_obs,n_dates,significant_5pct,p_value_bh,bh_n_tests,bh_family,significant_5pct_bh,source
0,Y3_SizeDirection,abs_d_y3,2.212868,0.039202,6,7,17157,2451,True,0.039202,2,Y3_SizeDirection,True,07b (신규)
1,Y3_SizeDirection,d_y3,4.137249,0.000385,6,7,17157,2451,True,0.000770,2,Y3_SizeDirection,True,07b (신규)
2,Y3_Asymmetry,d_y3_up,3.516189,0.001817,6,7,17157,2451,True,0.003633,2,NaN,True,"07 (기존, 참고)"
3,Y3_Asymmetry,d_y3_down,3.130383,0.004652,6,7,17157,2451,True,0.004652,2,NaN,True,"07 (기존, 참고)"


In [86]:
def spec_row(data, label, note):
    """조건 하나에 대해 산업 간 크기계수 차이 검정을 돌린다."""
    diff = run_industry_difference_test(data, ["abs_d_y3", "d_y3"], label)
    row_abs = diff[diff["variable"] == "abs_d_y3"].iloc[0]
    row_dir = diff[diff["variable"] == "d_y3"].iloc[0]

    return {
        "specification": label,
        "note": note,
        "n_obs": int(row_abs["n_obs"]),
        "n_dates": int(row_abs["n_dates"]),
        "n_industries": int(row_abs["n_industries"]),
        "size_diff_f": row_abs["f_stat"],
        "size_diff_p": row_abs["p_value"],
        "size_keeps_5pct": bool(row_abs["p_value"] < 0.05),
        "direction_diff_f": row_dir["f_stat"],
        "direction_diff_p": row_dir["p_value"],
        "direction_keeps_5pct": bool(row_dir["p_value"] < 0.05),
    }


base = panel.dropna(subset=["industry_return", "kospi_return", "d_y3"]).copy()
year = base["date"].dt.year

# 극단일 기준은 날짜 단위(중복 제거)로 계산한다.
_daily_abs = base.drop_duplicates("date")["abs_d_y3"]
q99, q995 = _daily_abs.quantile(0.99), _daily_abs.quantile(0.995)

wins = base.copy()
_lo, _hi = wins["d_y3"].quantile([0.01, 0.99])
wins["d_y3"] = wins["d_y3"].clip(_lo, _hi)
wins["abs_d_y3"] = wins["d_y3"].abs()

specs = [
    (base, "전체 2016-2025", "기준 사양"),
    (base[~year.isin([2022, 2023])], "2022-2023 제외", "금리 급변동기 제외"),
    (base[~year.isin([2020, 2021])], "2020-2021 제외", "코로나 구간 제외"),
    (base[year <= 2020], "2016-2020", "전반기"),
    (base[year >= 2021], "2021-2025", "후반기"),
    (base[base["abs_d_y3"] < q99], "절대변화 상위1% 제외", "극단일 제외"),
    (base[base["abs_d_y3"] < q995], "절대변화 상위0.5% 제외", "극단일 제외"),
    (wins, "Winsorize 1%", "상하위 1% 절단"),
    (base[base["industry"] != "건설업"], "건설업 제외", "특정 산업 의존 확인"),
]

uncert_specification_sensitivity = pd.DataFrame(
    [spec_row(d, lab, note) for d, lab, note in specs])

uncert_specification_sensitivity.to_csv(
    TABDIR / "uncert_specification_sensitivity.csv", index=False, encoding="utf-8-sig")
print("저장:", config._rel(TABDIR / "uncert_specification_sensitivity.csv"))
display(uncert_specification_sensitivity.round(6))

저장: results/H2_asymmetry_regime/tables/uncert_specification_sensitivity.csv


,specification,note,n_obs,n_dates,n_industries,size_diff_f,size_diff_p,size_keeps_5pct,direction_diff_f,direction_diff_p,direction_keeps_5pct
0,전체 2016-2025,기준 사양,17157,2451,7,2.212868,0.039202,True,4.137249,0.000385,True
1,2022-2023 제외,금리 급변동기 제외,13720,1960,7,2.604541,0.016188,True,1.299685,0.253852,False
2,2020-2021 제외,코로나 구간 제외,13685,1955,7,2.491447,0.020992,True,2.930844,0.007538,True
3,2016-2020,전반기,8582,1226,7,1.170622,0.319453,False,1.156767,0.327153,False
4,2021-2025,후반기,8575,1225,7,3.027835,0.006077,True,5.919151,0.000004,True
5,절대변화 상위1% 제외,극단일 제외,16982,2426,7,2.065306,0.054196,False,5.162098,0.000028,True
6,절대변화 상위0.5% 제외,극단일 제외,17066,2438,7,1.917946,0.074362,False,4.947780,0.000048,True
7,Winsorize 1%,상하위 1% 절단,17157,2451,7,2.294873,0.032656,True,4.763209,0.000078,True
8,건설업 제외,특정 산업 의존 확인,14706,2451,6,0.950690,0.446922,False,4.863104,0.000198,True


In [87]:
spec = uncert_specification_sensitivity.copy()
y = np.arange(len(spec))
h = 0.38

fig, ax = plt.subplots(figsize=(10, 6))
NEUTRAL, GREY = "#5c8cbc", config.SIGN_COLORS["ns"]

s_p = spec["size_diff_p"].to_numpy(float)
d_p = spec["direction_diff_p"].to_numpy(float)

ax.barh(y - h / 2, s_p, height=h,
        color=[NEUTRAL if v < 0.05 else GREY for v in s_p], edgecolor="white")
ax.barh(y + h / 2, d_p, height=h,
        color=[NEUTRAL if v < 0.05 else GREY for v in d_p],
        edgecolor="white", hatch="//")

ax.axvline(0.05, color="#c0392b", linewidth=1.2, linestyle="--")
ax.text(0.055, -0.75, "p = 0.05", color="#c0392b", fontsize=9)
ax.set_yticks(y)
ax.set_yticklabels(spec["specification"])
ax.invert_yaxis()
ax.set_xlabel("산업 간 계수 차이 검정의 p 값")
ax.set_title("조건을 바꿔도 산업 간 차이가 남는가")
ax.grid(axis="x", alpha=0.25)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="white", edgecolor="#444444", label="무늬 없음 = 크기계수 차이"),
    Patch(facecolor="white", edgecolor="#444444", hatch="//", label="빗금 = 방향계수 차이 (비교용)"),
    Patch(facecolor=NEUTRAL, edgecolor="white", label="파랑 = 5% 기준 충족"),
    Patch(facecolor=GREY, edgecolor="white", label="회색 = 미충족"),
], loc="lower right", frameon=False, fontsize=8.5)

fig.tight_layout()
fig.savefig(FIGDIR / "11_uncert_specification_sensitivity.png", dpi=200, bbox_inches="tight")
print("저장:", config._rel(FIGDIR / "11_uncert_specification_sensitivity.png"))
plt.show()

저장: results/H2_asymmetry_regime/figures/11_uncert_specification_sensitivity.png


**관찰.** 산업 간 크기계수 차이는 대부분 사양에서 5% 기준을 유지하지만 일부 조건(건설업 제외 등)에서는 넘지 못한다. 방향계수 차이보다 사양에 덜 민감하다.

**4.A.7 |Δy3| 충격의 경제적 크기.** 확인할 내용: |Δy3| 1 표준편차 충격이 산업 일별 수익률 표준편차 대비 얼마나 큰 변화를 일으키는지, KOSPI 1 표준편차 효과와 비교하면 어느 정도인지.

In [88]:
rows = []
for ind in INDUSTRIES:
    g = panel[panel["industry"] == ind]
    m = fit_absdir(g)
    s = g[["industry_return", "kospi_return", "abs_d_y3", "d_y3"]].dropna()

    ret_sd = s["industry_return"].std()
    abs_sd = s["abs_d_y3"].std()
    kospi_sd = s["kospi_return"].std()

    b_abs = m.params["abs_d_y3"]
    rows.append({
        "industry": ind,
        "beta_abs": b_abs,
        "p_value": m.pvalues["abs_d_y3"],
        "abs_d_y3_sd_pp": abs_sd,
        "abs_d_y3_sd_bp": abs_sd * 100,
        "ret_sd_pct": ret_sd * 100,
        # 1SD 충격 시 수익률 변화(%)
        "effect_1sd_pct": b_abs * abs_sd * 100,
        # 수익률 표준편차 대비 비율(%)
        "ratio_to_ret_sd_pct": abs(b_abs * abs_sd) / ret_sd * 100,
        # 10bp 변동 시 수익률 변화(%)
        "effect_10bp_pct": b_abs * 0.1 * 100,
        # 비교 기준: KOSPI 1SD 효과
        "kospi_effect_1sd_pct": m.params["kospi_return"] * kospi_sd * 100,
        "kospi_ratio_to_ret_sd_pct": abs(m.params["kospi_return"] * kospi_sd) / ret_sd * 100,
        "nobs": int(m.nobs),
    })

uncert_economic_magnitude = pd.DataFrame(rows)
uncert_economic_magnitude.to_csv(
    TABDIR / "uncert_economic_magnitude.csv", index=False, encoding="utf-8-sig")
print("저장:", config._rel(TABDIR / "uncert_economic_magnitude.csv"))
display(uncert_economic_magnitude.round(4))

저장: results/H2_asymmetry_regime/tables/uncert_economic_magnitude.csv


,industry,beta_abs,p_value,abs_d_y3_sd_pp,abs_d_y3_sd_bp,ret_sd_pct,effect_1sd_pct,ratio_to_ret_sd_pct,effect_10bp_pct,kospi_effect_1sd_pct,kospi_ratio_to_ret_sd_pct,nobs
0,건설업,-0.0263,0.0012,0.0295,2.948,1.7870,-0.0776,4.3414,-0.2632,1.1206,62.7051,2451
1,금융업,-0.0059,0.2913,0.0295,2.948,1.3031,-0.0174,1.3365,-0.0591,1.0282,78.8987,2451
2,운수장비,0.0002,0.9745,0.0295,2.948,1.6231,0.0007,0.0418,0.0023,1.1167,68.8038,2451
3,의약품,0.0105,0.2915,0.0295,2.948,1.8051,0.0310,1.7168,0.1051,0.8815,48.8340,2451
4,전기전자,0.0066,0.1983,0.0295,2.948,1.5792,0.0196,1.2381,0.0663,1.3416,84.9548,2451
5,철강금속,0.0099,0.3068,0.0295,2.948,1.8379,0.0291,1.5814,0.0986,1.1426,62.1678,2451
6,화학,-0.0017,0.7834,0.0295,2.948,1.4600,-0.0051,0.3459,-0.0171,1.1713,80.2267,2451


In [89]:
mag = uncert_economic_magnitude.set_index("industry").loc[INDUSTRIES]
y = np.arange(len(mag))
h = 0.38

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(y - h / 2, mag["ratio_to_ret_sd_pct"], height=h,
        color="#5c8cbc", edgecolor="white", label="|Δy3| 1SD 효과")
ax.barh(y + h / 2, mag["kospi_ratio_to_ret_sd_pct"], height=h,
        color=config.SIGN_COLORS["ns"], edgecolor="white", label="KOSPI 1SD 효과")

for k, (ind, r) in enumerate(mag.iterrows()):
    ax.text(r["ratio_to_ret_sd_pct"] + 1.2, k - h / 2,
            f"{r['ratio_to_ret_sd_pct']:.2f}%", va="center", fontsize=8)

ax.set_yticks(y)
ax.set_yticklabels(mag.index)
ax.invert_yaxis()
ax.set_xlabel("산업 일별 수익률 표준편차 대비 비율 (%)")
ax.set_title("1 표준편차 충격의 크기 - 금리 변동 크기 대 시장요인")
ax.grid(axis="x", alpha=0.25)
ax.set_xlim(0, max(mag["kospi_ratio_to_ret_sd_pct"]) * 1.18)
ax.legend(loc="lower right", frameon=False, bbox_to_anchor=(1.0, 0.02))

fig.tight_layout()
fig.savefig(FIGDIR / "12_uncert_economic_magnitude.png", dpi=200, bbox_inches="tight")
print("저장:", config._rel(FIGDIR / "12_uncert_economic_magnitude.png"))
plt.show()

저장: results/H2_asymmetry_regime/figures/12_uncert_economic_magnitude.png


**관찰.** |Δy3| 1 표준편차 충격의 효과는 KOSPI 1 표준편차 효과보다 작다. 금리 충격의 절대적 크기는 시장요인보다 작지만 0은 아니다.

**4.A.8 정책국면별(5국면) 금리베타.** 확인할 내용: 한국은행 정책금리 사이클 5개 국면 각각에서 산업별 금리베타(β₁~β₅)와 95% 신뢰구간이 어떻게 달라지는지.

In [90]:
# 0. 환경 설정 · 경로 · 라이브러리

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
# 저장소 루트에서 `pip install -e .` 를 한 번 실행했다면 아래 탐색은 건너뛴다.
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(
        f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}\n"
        f"저장소 루트({_root})에서 `pip install -e .` 를 실행하세요."
    )
# --- 부트스트랩 끝 ---

# 한글 폰트 공통 설정 (config.py 5-c절).
config.setup_korean_font()

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# 공통 상수는 config에서 가져온다 (STYLE_GUIDE 13·14절).
HAC_LAGS = config.HAC_MAXLAGS
ALPHA = config.BH_ALPHA

# 경로는 전부 config에서 가져온다. 개인 PC 절대경로를 출력에 남기지 않는다(config._rel 사용).
PANEL_FILE = config.PROCESSED_DIR / "industry_rate_panel_daily.csv"
REGIME_FILE = config.RAW_DIR / "market" / "model_D_policy_regime_daily.csv"

# 모형 D 산출물은 H2(비대칭·국면) 폴더에 regime_ 접두로 저장한다 (HYPOTHESIS_MODEL_MAP).
OUTPUT_DIR = config.H2_TABDIR
FIG_DIR = config.H2_FIGDIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("config      :", config._rel(Path(config.__file__)))
print("PANEL_FILE  :", config._rel(PANEL_FILE), "(존재:", PANEL_FILE.exists(), ")")
print("REGIME_FILE :", config._rel(REGIME_FILE), "(존재:", REGIME_FILE.exists(), ")")
print("OUTPUT_DIR  :", config._rel(OUTPUT_DIR))

그래프 폰트: Noto Sans CJK JP
config      : config.py
PANEL_FILE  : data/processed/industry_rate_panel_daily.csv (존재: True )
REGIME_FILE : data/raw/market/model_D_policy_regime_daily.csv (존재: False )
OUTPUT_DIR  : results/H2_asymmetry_regime/tables


In [91]:
# 1-1. 최종 패널 불러오기
if not PANEL_FILE.exists():
    raise FileNotFoundError(
        f"최종 패널을 찾을 수 없습니다: {PANEL_FILE}\n"
        "먼저 src/data_prep.py의 run_data_preparation() / run_variable_generation()을 실행하세요."
    )

panel = pd.read_csv(PANEL_FILE, encoding="utf-8-sig")
panel["date"] = pd.to_datetime(panel["date"])

required_cols = ["date", "industry", "industry_return", "kospi_return", "d_y3"]
missing_cols = [c for c in required_cols if c not in panel.columns]
if missing_cols:
    raise ValueError(f"최종 패널에 필요한 컬럼이 없습니다: {missing_cols}")

print("패널 크기:", panel.shape)
print("기간:", panel["date"].min().date(), "~", panel["date"].max().date())
print("산업 수:", panel["industry"].nunique())
print("산업:", sorted(panel["industry"].dropna().unique().tolist()))
display(panel[required_cols].head())

패널 크기: (17164, 12)
기간: 2016-01-04 ~ 2025-12-30
산업 수: 7
산업: ['건설업', '금융업', '운수장비', '의약품', '전기전자', '철강금속', '화학']


,date,industry,industry_return,kospi_return,d_y3
0,2016-01-04,건설업,NaN,NaN,NaN
1,2016-01-04,금융업,NaN,NaN,NaN
2,2016-01-04,운수장비,NaN,NaN,NaN
3,2016-01-04,의약품,NaN,NaN,NaN
4,2016-01-04,전기전자,NaN,NaN,NaN


In [92]:
# 2-1. 정책국면 데이터 불러오기 또는 자동 생성
def find_regime_file():
    if REGIME_FILE is not None:
        p = Path(REGIME_FILE)
        return p if p.exists() else None

    candidates = [
        config.PROCESSED_DIR / "model_D_policy_regime_daily.csv",
        config.PROCESSED_DIR / "policy_regime_daily.csv",
        config.INTERIM_DIR / "model_D_policy_regime_daily.csv",
        config.INTERIM_DIR / "policy_regime_daily.csv",
        Path.cwd() / "model_D_policy_regime_daily.csv",
        Path.cwd() / "policy_regime_daily.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def build_regime_map_from_dates():
    specs = [
        (1, "2016-01-04", "2017-11-29", 0, "저금리·완화 국면", "완화·인하"),
        (2, "2017-11-30", "2019-07-17", 1, "인상·정상화 사이클", "인상·긴축"),
        (3, "2019-07-18", "2021-08-25", 0, "인하·초저금리 사이클", "완화·인하"),
        (4, "2021-08-26", "2024-10-10", 1, "인상·고금리 유지 사이클", "인상·긴축"),
        (5, "2024-10-11", "2025-12-30", 0, "인하 재개 사이클", "완화·인하"),
    ]

    parts = []
    for regime, start, end, tightening, policy_cycle, direction in specs:
        dates = pd.date_range(start=start, end=end, freq="D")
        parts.append(pd.DataFrame({
            "date": dates,
            "regime": regime,
            "tightening": tightening,
            "policy_cycle": policy_cycle,
            "direction": direction,
        }))
    return pd.concat(parts, ignore_index=True)


regime_path = find_regime_file()

if regime_path is not None:
    regime_map = pd.read_csv(regime_path, encoding="utf-8-sig")
    regime_map["date"] = pd.to_datetime(regime_map["date"])
    print("정책국면 파일 사용:", config._rel(Path(regime_path)))
else:
    regime_map = build_regime_map_from_dates()
    print("정책국면 파일을 찾지 못해 합의된 날짜 기준으로 자동 생성했습니다.")

keep_cols = [c for c in ["date", "regime", "tightening", "policy_cycle", "direction"] if c in regime_map.columns]
regime_map = regime_map[keep_cols].copy()

for c in ["date", "regime", "tightening"]:
    if c not in regime_map.columns:
        raise ValueError(f"정책국면 데이터에 필수 컬럼 `{c}`가 없습니다.")

if regime_map["date"].duplicated().any():
    dup = regime_map.loc[regime_map["date"].duplicated(), "date"].head().tolist()
    raise ValueError(f"정책국면 데이터에 중복 날짜가 있습니다. 예: {dup}")

display(regime_map.head())

정책국면 파일을 찾지 못해 합의된 날짜 기준으로 자동 생성했습니다.


,date,regime,tightening,policy_cycle,direction
0,2016-01-04,1,0,저금리·완화 국면,완화·인하
1,2016-01-05,1,0,저금리·완화 국면,완화·인하
2,2016-01-06,1,0,저금리·완화 국면,완화·인하
3,2016-01-07,1,0,저금리·완화 국면,완화·인하
4,2016-01-08,1,0,저금리·완화 국면,완화·인하


In [93]:
# 2-2. 기존 패널에 regime / tightening 병합
model_d_panel = panel.merge(
    regime_map,
    on="date",
    how="left",
    validate="many_to_one",
)

analysis_mask = model_d_panel["date"].between("2016-01-04", "2025-12-30")
missing_regime = model_d_panel.loc[analysis_mask, "regime"].isna().sum()

print("병합 후 크기:", model_d_panel.shape)
print("분석기간 내 regime 누락 행:", missing_regime)

if missing_regime > 0:
    bad_dates = (
        model_d_panel.loc[analysis_mask & model_d_panel["regime"].isna(), "date"]
        .drop_duplicates()
        .sort_values()
        .head(20)
        .tolist()
    )
    raise ValueError(f"정책국면이 매핑되지 않은 날짜가 있습니다. 예: {bad_dates}")

model_d_panel["regime"] = model_d_panel["regime"].astype(int)
model_d_panel["tightening"] = model_d_panel["tightening"].astype(int)

display(model_d_panel[[
    "date", "industry", "industry_return", "kospi_return", "d_y3",
    "regime", "tightening"
]].head())

병합 후 크기: (17164, 16)
분석기간 내 regime 누락 행: 0


,date,industry,industry_return,kospi_return,d_y3,regime,tightening
0,2016-01-04,건설업,NaN,NaN,NaN,1,0
1,2016-01-04,금융업,NaN,NaN,NaN,1,0
2,2016-01-04,운수장비,NaN,NaN,NaN,1,0
3,2016-01-04,의약품,NaN,NaN,NaN,1,0
4,2016-01-04,전기전자,NaN,NaN,NaN,1,0


In [94]:
# D-1-1. 통합회귀 함수
REGIMES = [1, 2, 3, 4, 5]

def build_regime_design(df):
    # 전체 절편은 넣지 않고 regime별 절편 5개를 직접 넣는다.
    X = pd.DataFrame(index=df.index)

    for r in REGIMES:
        d = (df["regime"] == r).astype(float)
        X[f"const_r{r}"] = d
        X[f"mkt_r{r}"] = d * df["kospi_return"]
        X[f"rate_r{r}"] = d * df["d_y3"]

    return X


def run_regime_model(data, industry, hac_lags=HAC_LAGS):
    sub = (
        data.loc[
            data["industry"] == industry,
            ["date", "industry_return", "kospi_return", "d_y3", "regime"]
        ]
        .dropna()
        .sort_values("date")
        .copy()
    )

    present_regimes = sorted(sub["regime"].unique().tolist())
    if present_regimes != REGIMES:
        raise ValueError(
            f"{industry}: 5개 국면이 모두 존재하지 않습니다. 현재 국면={present_regimes}"
        )

    X = build_regime_design(sub)
    y = sub["industry_return"]

    model = sm.OLS(y, X)
    result = model.fit(
        cov_type="HAC",
        cov_kwds={"maxlags": hac_lags},
    )
    return result, sub


preferred_order = ["건설업", "금융업", "화학", "운수장비", "전기전자", "철강금속", "의약품"]
industries = [x for x in preferred_order if x in model_d_panel["industry"].unique()]
industries += [
    x for x in sorted(model_d_panel["industry"].unique())
    if x not in industries
]

model_results = {}
model_samples = {}

for industry in industries:
    res, sample = run_regime_model(model_d_panel, industry)
    model_results[industry] = res
    model_samples[industry] = sample
    print(f"{industry}: n={int(res.nobs)}, adj.R²={res.rsquared_adj:.4f}")

print("\n완료된 산업 수:", len(model_results))

건설업: n=2451, adj.R²=0.4042
금융업: n=2451, adj.R²=0.6259
화학: n=2451, adj.R²=0.6452
운수장비: n=2451, adj.R²=0.4886
전기전자: n=2451, adj.R²=0.7275
철강금속: n=2451, adj.R²=0.3853


의약품: n=2451, adj.R²=0.2499

완료된 산업 수: 7


In [95]:
# D-1-2. 국면별 금리베타 추출
def result_series(result):
    names = result.model.exog_names
    params = pd.Series(np.asarray(result.params), index=names)
    bse = pd.Series(np.asarray(result.bse), index=names)
    pvalues = pd.Series(np.asarray(result.pvalues), index=names)
    ci = pd.DataFrame(
        np.asarray(result.conf_int(alpha=ALPHA)),
        index=names,
        columns=["ci_low", "ci_high"],
    )
    return params, bse, pvalues, ci


d1_rows = []

for industry, res in model_results.items():
    params, bse, pvalues, ci = result_series(res)

    for r in REGIMES:
        name = f"rate_r{r}"
        d1_rows.append({
            "industry": industry,
            "regime": r,
            "beta": params[name],
            "hac_se": bse[name],
            "p_value": pvalues[name],
            "ci_low": ci.loc[name, "ci_low"],
            "ci_high": ci.loc[name, "ci_high"],
            "nobs_total_industry": int(res.nobs),
            "nobs_regime": int((model_samples[industry]["regime"] == r).sum()),
        })

d1_long = pd.DataFrame(d1_rows)

d1_beta_wide = (
    d1_long
    .pivot(index="industry", columns="regime", values="beta")
    .rename(columns=lambda r: f"{r}기 β")
    .reindex(industries)
)

print("[D-1 국면별 금리베타]")
display(d1_beta_wide.round(6))

print("[D-1 상세 결과]")
display(d1_long.round(6))

d1_long.to_csv(OUTPUT_DIR / "regime_beta_long.csv", index=False, encoding="utf-8-sig")
d1_beta_wide.to_csv(OUTPUT_DIR / "regime_beta_wide.csv", encoding="utf-8-sig")

[D-1 국면별 금리베타]


regime,1기 β,2기 β,3기 β,4기 β,5기 β
industry,,,,,
건설업,0.036986,0.095305,-0.010244,-0.002941,-0.039238
금융업,0.050240,0.024372,0.001273,0.006019,0.034307
화학,0.005831,0.017157,-0.017168,0.007710,-0.019556
운수장비,0.081579,0.034323,-0.017115,0.016247,0.002448
전기전자,-0.054118,-0.042118,0.022710,0.002758,-0.000860
철강금속,0.070965,0.042810,0.009131,0.024990,-0.057606
의약품,-0.023950,-0.032675,0.001379,-0.035623,-0.029533


[D-1 상세 결과]


,industry,regime,beta,hac_se,p_value,ci_low,ci_high,nobs_total_industry,nobs_regime
0,건설업,1,0.036986,0.022547,0.100912,-0.007204,0.081177,2451,468
1,건설업,2,0.095305,0.038858,0.014181,0.019145,0.171466,2451,398
2,건설업,3,-0.010244,0.027542,0.709930,-0.064226,0.043737,2451,522
3,건설업,4,-0.002941,0.009504,0.756993,-0.021567,0.015686,2451,765
4,건설업,5,-0.039238,0.025906,0.129875,-0.090014,0.011538,2451,298
5,금융업,1,0.050240,0.014782,0.000677,0.021267,0.079212,2451,468
6,금융업,2,0.024372,0.015056,0.105487,-0.005136,0.053881,2451,398
7,금융업,3,0.001273,0.014325,0.929184,-0.026803,0.029350,2451,522
8,금융업,4,0.006019,0.003937,0.126315,-0.001698,0.013735,2451,765
9,금융업,5,0.034307,0.021873,0.116775,-0.008563,0.077177,2451,298


In [96]:
# D-1-3. 산업별 β + 95% CI 그래프

# 한글 폰트·음수기호는 config.setup_korean_font()(0셀)에서 이미 설정됨.


# --------------------------------------------------
# 2. 산업별 β + 95% CI 그래프
# --------------------------------------------------
for industry in industries:

    sub = (
        d1_long[d1_long["industry"] == industry]
        .sort_values("regime")
    )

    x = sub["regime"].to_numpy()
    y = sub["beta"].to_numpy()

    lower = y - sub["ci_low"].to_numpy()
    upper = sub["ci_high"].to_numpy() - y

    fig, ax = plt.subplots(figsize=(8, 4.8))

    ax.errorbar(
        x,
        y,
        yerr=np.vstack([lower, upper]),
        fmt="o",
        capsize=5,
        markersize=6,
    )

    # β = 0 기준선
    ax.axhline(0, linewidth=1)

    ax.set_xticks(REGIMES)
    ax.set_xlabel("정책국면")
    ax.set_ylabel("국고채 3년 금리베타")
    ax.set_title(f"{industry} — 정책국면별 금리베타 (95% CI)")

    ax.grid(axis="y", alpha=0.25)

    fig.tight_layout()

    # 저장
    out = FIG_DIR / f"regime_{industry}_beta_ci.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")

    plt.show()
    plt.close(fig)


print("그래프 저장 폴더:", config._rel(FIG_DIR))

그래프 저장 폴더: results/H2_asymmetry_regime/figures


**관찰.** 국면별 베타는 산업·국면 조합에 따라 부호와 크기가 달라진다. 이는 H2 본문의 정책국면 보조분석(완화·인상 2구간 비교)을 5구간으로 세분한 결과이며, 구간을 세분할수록 신뢰구간이 넓어져 개별 국면의 유의성 판정은 더 보수적으로 봐야 한다.

In [97]:
# 4.A에서 바뀐 전역 상태(panel, industries)를 Baseline 시점 값으로 되돌린다.
# H3·Appendix는 원래의 baseline panel/industries를 쓴다(새 계산 없음, 상태 복원뿐).
panel = _panel_baseline.copy()
industries = list(_industries_baseline)

# 4. H3 — 산업 재무특성과 금리베타의 탐색적 연관

H3는 재무특성이 산업별 금리베타 차이의 원인인지를 검정하지 않는다. Baseline에서 의약품이 포함된 비교에서 차이가 가장 뚜렷했던 점을 출발점으로, 배당수익률과 부채비율의 산업 순위가 금리베타와 어떻게 정렬되는지 살펴본다.


## 방법
산업별 대표기업 10개사의 2016–2025년 재무특성을 2024년 시가총액가중 평균과 중앙값으로 집계한다. 산업 수가 7개이므로 7!=5,040개 전체 순열을 이용한 양측 정확 순열검정으로 Spearman $\rho$의 p-value를 계산한다.


In [98]:
# 의약품을 제외한 6개 산업만으로 Baseline 공동검정을 다시 수행 (동일한 검정을 표본만 축소)
joint_ex_pharma = models.cross_industry_Ftest(
    panel[panel["industry"] != "의약품"]
    .rename(columns={"industry_return": "ret", "kospi_return": "mkt", "d_y3": "dY"})
    [["industry", "ret", "mkt", "dY", "date"]]
)
print(f"의약품 제외 6개 산업 공동검정: F={float(joint_ex_pharma.fvalue):.3f}, p={float(joint_ex_pharma.pvalue):.4f}")

의약품 제외 6개 산업 공동검정: F=2.204, p=0.0513


**의약품 제외 시 공동검정은 5% 유의수준을 소폭 상회했다.** 5%에서는 귀무가설을 기각하지 못하지만 10% 기준에서는 경계적이다. 검정통계량과 p-value는 위 결과표에 제시한다. 따라서 의약품만이 산업 차이의 원인이라고 단정할 수 없다. 산업 간 차이는 의약품을 포함한 비교에서 가장 뚜렷했다.


In [99]:
INDUSTRIES = industries
RAW_FIRM_DIR = config.RAW_DIR / "firm"

fin = pd.read_csv(config.PROCESSED_DIR / "firm_financials_recovered.csv", dtype={"stock_code": str})
fin["stock_code"] = fin["stock_code"].str.zfill(6)
sel = pd.read_csv(RAW_FIRM_DIR / "selected_top10.csv", dtype={"stock_code": str})
sel["stock_code"] = sel["stock_code"].str.zfill(6)
weight = sel[["stock_code", "market_cap_2024"]].drop_duplicates("stock_code")

# Baseline 절에서 이미 추정한 산업별 금리베타를 그대로 재사용한다.
beta = beta_table.set_index("industry")["beta_rate"].reindex(INDUSTRIES)

fpa = (
    fin.groupby(["industry", "stock_code"])
    .agg(부채비율=("debt_ratio", "mean"), 배당수익률=("dividend_yield", "mean"))
    .reset_index().merge(weight, on="stock_code", how="left")
)


def wavg(g, col, w="market_cap_2024"):
    v, ww = g[col], g[w]
    m = v.notna() & ww.notna() & (ww > 0)
    return np.average(v[m], weights=ww[m]) if m.any() else np.nan


rows = []
for ind, g in fpa.groupby("industry"):
    rows.append({
        "industry": ind,
        "부채비율_시총가중": wavg(g, "부채비율"), "부채비율_중앙값": g["부채비율"].median(),
        "배당수익률_시총가중": wavg(g, "배당수익률"), "배당수익률_중앙값": g["배당수익률"].median(),
    })
char = pd.DataFrame(rows).set_index("industry").reindex(INDUSTRIES)
from decimal import Decimal, ROUND_HALF_UP


def format_three_decimals_half_up(value):
    if pd.isna(value):
        return ""
    rounded = Decimal(f"{value:.12f}").quantize(Decimal("0.001"), rounding=ROUND_HALF_UP)
    return f"{rounded:.3f}"


char.apply(lambda column: column.map(format_three_decimals_half_up))


,부채비율_시총가중,부채비율_중앙값,배당수익률_시총가중,배당수익률_중앙값
industry,,,,
건설업,1.609,1.875,1.743,1.405
금융업,9.116,10.526,3.763,4.105
운수장비,1.784,2.073,1.931,0.469
의약품,0.493,0.628,0.139,0.360
전기전자,0.580,0.721,1.615,0.752
철강금속,0.791,0.913,2.763,1.830
화학,1.008,0.808,1.609,1.645


의약품은 부채비율과 배당수익률의 두 집계 기준에서 모두 7개 산업 중 가장 낮다.


In [100]:
def perm_p(x, y):
    """정확 순열검정(양측): |rho| >= |rho_0|인 순열의 비율."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    rho0 = spearmanr(x, y).correlation
    cnt = tot = 0
    for pm in permutations(range(len(x))):
        if abs(spearmanr(x, y[list(pm)]).correlation) >= abs(rho0) - 1e-12:
            cnt += 1
        tot += 1
    return rho0, cnt / tot


def crit(n, alpha=0.05):
    """n개 순위에서 양측 5% 기준을 만족하는 |rho|의 정확 임계값."""
    base = list(range(n))
    denom = n * (n * n - 1)
    rhos = [1 - 6 * sum((a - b) ** 2 for a, b in zip(base, p)) / denom for p in permutations(base)]
    N = len(rhos)
    absr = sorted({abs(round(r, 6)) for r in rhos}, reverse=True)
    last = 1.0
    for threshold in absr:
        if sum(1 for r in rhos if abs(r) >= threshold - 1e-9) / N > alpha:
            break
        last = threshold
    return last


spearman_rows = []


def spear(char_col, label, subset=None):
    b, c = beta.copy(), char[char_col].copy()
    if subset is not None:
        b, c = b.loc[subset], c.loc[subset]
    df = pd.concat([b.rename("beta"), c.rename("v")], axis=1).dropna()
    rho, p_value = perm_p(df["beta"].values, df["v"].values)
    spearman_rows.append({
        "관계": label,
        "n": len(df),
        "rho": rho,
        "순열 p-value": p_value,
        "임계값(0.05)": crit(len(df)),
    })


ex_fin = [i for i in INDUSTRIES if i != "금융업"]
spear("배당수익률_시총가중", "배당수익률 (시총가중)")
spear("배당수익률_중앙값", "배당수익률 (중앙값)")
spear("부채비율_시총가중", "부채비율 (시총가중)")
spear("부채비율_중앙값", "부채비율 (중앙값)")
spear("부채비율_시총가중", "부채비율 (금융업 제외)", subset=ex_fin)

# 배당 미공시 연도를 0이 아닌 결측으로 처리하는 대안
fin_alt = fin.copy()
fin_alt["배당_대안"] = np.where(fin_alt["div_status"] == "ok", fin_alt["dividend_yield"], np.nan)
fpa_alt = (
    fin_alt.groupby(["industry", "stock_code"])
    .agg(배당_대안=("배당_대안", "mean"))
    .reset_index().merge(weight, on="stock_code", how="left")
)
alt_rows = []
for ind, group in fpa_alt.groupby("industry"):
    alt_rows.append({"industry": ind, "배당_시총가중_대안": wavg(group, "배당_대안")})
char_alt = pd.DataFrame(alt_rows).set_index("industry").reindex(INDUSTRIES)
alt_df = pd.concat([
    beta.rename("beta"),
    char_alt["배당_시총가중_대안"].rename("v"),
], axis=1).dropna()
rho_alt, p_alt = perm_p(alt_df["beta"].values, alt_df["v"].values)
spearman_rows.append({
    "관계": "배당수익률 (미공시 연도 결측)",
    "n": len(alt_df),
    "rho": rho_alt,
    "순열 p-value": p_alt,
    "임계값(0.05)": crit(len(alt_df)),
})

spearman_results = pd.DataFrame(spearman_rows)
spearman_results.round(4)


,관계,n,rho,순열 p-value,임계값(0.05)
0,배당수익률 (시총가중),7,0.8571,0.0238,0.7857
1,배당수익률 (중앙값),7,0.5357,0.2357,0.7857
2,부채비율 (시총가중),7,0.6071,0.1667,0.7857
3,부채비율 (중앙값),7,0.7500,0.0663,0.7857
4,부채비율 (금융업 제외),6,0.6571,0.1750,0.8857
5,배당수익률 (미공시 연도 결측),7,0.7857,0.0480,0.7857


## 배당수익률

시가총액가중 기준에서는 순위정렬이 관찰되지만 중앙값 기준에서는 약해진다. 미공시 연도를 결측으로 처리한 대안은 5% 유의수준의 경계에 놓인다. 상관계수와 exact p-value는 위 결과표에 제시한다. **배당수익률 결과는 특정 집계기준에서 관찰된 순위정렬로 해석한다.**

## 부채비율

시가총액가중, 중앙값, 금융업 제외 사양 모두 5% 수준에 도달하지 않았다. **부채비율은 주요 사양에서 안정적인 관계가 확인되지 않았다.** 철강금속 제외 시 순위상관이 크게 변하는 결과는 강한 관계의 발견이 아니라 소표본의 민감성을 보여준다. 상관계수와 p-value는 위 결과표에 제시한다.


아래 산점도는 시가총액가중 재무특성과 산업 금리베타의 순위 배치를 보여준다. 회귀선이나 인과관계를 제시하는 그림은 아니다.


In [101]:
# 금리베타 vs 재무특성 산점도 (BH 유의성 기준 색상)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
panels = [("부채비율_시총가중", "부채비율 (탐색적, 로그축)", True), ("배당수익률_시총가중", "배당수익률 (%)", False)]
sig_map = beta_table.set_index("industry")["p_value_bh"].reindex(INDUSTRIES) < 0.05

for ax, (col, label, logx) in zip(axes, panels):
    for ind in INDUSTRIES:
        x, y = char.loc[ind, col], beta.loc[ind]
        color = (NEG_COLOR if y < 0 else POS_COLOR) if sig_map.loc[ind] else NS_COLOR
        ax.scatter(x, y, s=65, color=color, edgecolor="white", linewidth=0.6, zorder=3)
        ax.annotate(ind, (x, y), fontsize=8.5, xytext=(5, 4), textcoords="offset points")
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(label)
    ax.set_ylabel("산업 금리베타")
fig.suptitle("금리베타 vs 산업 재무특성 (점 색: BH 유의성)", fontweight="bold")
fig.tight_layout()
plt.show()

In [102]:
# 시가총액가중 결과가 소수 대형주에 얼마나 좌우되는지 확인한다.
def concentration(group):
    group = group.dropna(subset=["market_cap_2024"]).sort_values(
        "market_cap_2024", ascending=False
    ).reset_index(drop=True)
    total = group["market_cap_2024"].sum()
    share = group["market_cap_2024"] / total * 100
    hhi = float(((group["market_cap_2024"] / total) ** 2).sum())
    n = len(group)
    return {
        "top1_share_pct": share.iloc[0] if n >= 1 else np.nan,
        "top3_share_pct": share.iloc[:3].sum() if n >= 3 else np.nan,
        "hhi": hhi,
        "n_firms": n,
    }


pharma_summary = concentration(sel[sel["industry"] == "의약품"][["market_cap_2024"]].copy())
sup = pd.read_csv(config.PROCESSED_DIR / "pharma_rnd_support.csv").dropna(subset=["rnd_to_sales_pct"])
print(
    f"의약품 시총 집중도: Top1 {pharma_summary['top1_share_pct']:.1f}% · "
    f"Top3 {pharma_summary['top3_share_pct']:.1f}% · HHI {pharma_summary['hhi']:.3f}"
)
print(
    f"의약품 R&D/매출 중앙값: {sup['rnd_to_sales_pct'].median():.1f}% "
    f"({len(sup)}개사 전체, 회계기준 혼재)"
)


의약품 시총 집중도: Top1 50.6% · Top3 88.3% · HHI 0.357
의약품 R&D/매출 중앙값: 11.6% (10개사 전체, 회계기준 혼재)


의약품의 시가총액 집중도와 R&D/매출 표본 요약은 아래 표에 제시한다. 시총가중 배당수익률은 소수 대형주의 영향이 크다.

| 보조지표 | 값 |
|---|---:|
| 2024년 Top1 시가총액 비중 | 50.6% |
| 2024년 Top3 시가총액 비중 | 88.3% |
| 2024년 시가총액 HHI | 0.357 |
| R&D/매출 중앙값 | 11.6% |
| R&D/매출 표본수 | 10개사 |

사업보고서 연구개발비 총계와 DART 경상연구개발비가 혼재하므로 R&D/매출 값은 기업 간 강도 비교나 duration의 직접 측정치로 사용하지 않는다.¹

¹ 사전적 제외 기준이 없어 10개사를 모두 포함했다.

## 만기 구조와의 대조

| 설명변수 | 의약품 원계수 | 10bp 변화 시 예상 수익률 변화(%) | p-value |
|---|---:|---:|---:|
| 국고 3년 | −0.02707 | −0.27 | 0.000548 |
| 국고 10년 | −0.01598 | −0.16 | 0.0697 |
| NS Level | −0.01382 | −0.14 | 0.383 |
| NS Slope | −0.00901 | −0.09 | 0.534 |
| NS Curvature | −0.00860 | −0.09 | 0.00236 |

장기금리와 Level 반응은 5% 수준에 도달하지 않았다. 이는 duration 해석을 기각하는 직접 검정은 아니지만, 의약품의 음(-) 베타를 duration 하나로 설명하지 않아야 하는 반대증거다.


### 5.A 추가 민감도 — Final Notebook 미포함

`10_firm_characteristics_analysis.ipynb`의 부채비율 강건성 점검을 옮긴다.

**5.A.1 부채비율의 공통기간·집계기준 민감도.** 확인할 내용: 부채비율 순위상관이 자료가용성이 낮은 금융업 문제, 공통기간(2023–2025) 재계산, 개별 산업 제외에 얼마나 민감한지.

In [103]:
# [셀 1] 설정 · 수집 결과 로드 (import·경로는 여기서 한 번만)

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
import sys
from pathlib import Path
_anchors = []
if "__vsc_ipynb_file__" in globals():
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))
import config
if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(f"의도하지 않은 config.py: {config.__file__}")
# --- 부트스트랩 끝 ---

import warnings
from itertools import permutations
import numpy as np, pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
config.setup_korean_font()

RAW_FIRM_DIR = config.RAW_DIR / "firm"
PROCESSED_DIR = config.PROCESSED_DIR
H3_TABDIR = config.H3_TABDIR
H3_FIGDIR = config.H3_FIGDIR
for d in (PROCESSED_DIR, H3_TABDIR, H3_FIGDIR):
    d.mkdir(parents=True, exist_ok=True)
INDUSTRIES = ["건설업", "금융업", "운수장비", "의약품", "전기전자", "철강금속", "화학"]
YEARS = list(range(2016, 2026))

FIN_CSV = RAW_FIRM_DIR / "firm_financials.csv"
FIN_RECOVERED_CSV = PROCESSED_DIR / "firm_financials_recovered.csv"  # STEP9에서 금융업 부채비율 결측 보완(DART 사업보고서 원문)
SEL_CSV = RAW_FIRM_DIR / "selected_top10.csv"
BASELINE_BETA = config.BASELINE_DIR / "tables" / "rate_beta_by_industry_hac.csv"
JOINT_CSV = config.BASELINE_DIR / "tables" / "industry_beta_joint_test.csv"
PAIRWISE_CSV = config.BASELINE_DIR / "tables" / "industry_beta_pairwise_tests.csv"
for _p in (FIN_CSV, BASELINE_BETA, JOINT_CSV, PAIRWISE_CSV):
    assert _p.exists(), f"[필수입력 없음] {_p}"

# 부채비율 계산에는 STEP9 보완 파일을 우선 사용(없으면 원본으로 대체) — 배당수익률·R&D는 영향 없음
fin = pd.read_csv(FIN_RECOVERED_CSV if FIN_RECOVERED_CSV.exists() else FIN_CSV, dtype={"stock_code": str})
fin["stock_code"] = fin["stock_code"].str.zfill(6)
sel = pd.read_csv(SEL_CSV, dtype={"stock_code": str}); sel["stock_code"] = sel["stock_code"].str.zfill(6)
weight = sel[["stock_code", "market_cap_2024"]].drop_duplicates("stock_code")
beta = pd.read_csv(BASELINE_BETA).set_index("industry")["beta_rate"].reindex(INDUSTRIES)
print(f"[로드] firm_financials {fin.shape} · 산업별 금리베타 {beta.notna().sum()}개")
print(beta.round(4).to_string())

그래프 폰트: Noto Sans CJK JP


[로드] firm_financials (700, 14) · 산업별 금리베타 7개
industry
건설업     0.0032
금융업     0.0112
운수장비    0.0172
의약품    -0.0271
전기전자    0.0016
철강금속    0.0188
화학      0.0026


In [104]:
# [셀 4] Spearman — 정확 임계값 + 순열 p + 강건성(시총가중/중앙값/금융제외/LOO/결측대안)
def perm_p(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    rho0 = spearmanr(x, y).correlation
    tot = cnt = 0
    for pm in permutations(range(len(x))):
        if abs(spearmanr(x, y[list(pm)]).correlation) >= abs(rho0) - 1e-12:
            cnt += 1
        tot += 1
    return rho0, cnt / tot

def crit(n, alpha=0.05):
    base = list(range(n)); denom = n * (n * n - 1)
    rhos = [1 - 6 * sum((a - b) ** 2 for a, b in zip(base, p)) / denom for p in permutations(base)]
    N = len(rhos); absr = sorted({abs(round(r, 6)) for r in rhos}, reverse=True); last = 1.0
    for t in absr:
        if sum(1 for r in rhos if abs(r) >= t - 1e-9) / N > alpha:
            break
        last = t
    return last

res = []
def spear(char_col, label, subset=None):
    b, c = beta.copy(), char[char_col].copy()
    if subset is not None:
        b, c = b.loc[subset], c.loc[subset]
    df = pd.concat([b.rename("beta"), c.rename("v")], axis=1).dropna()
    rho, p = perm_p(df["beta"].values, df["v"].values)
    res.append({"관계": label, "n": len(df), "rho": round(rho, 4), "순열_p": round(p, 4),
                "임계값(0.05)": round(crit(len(df)), 4), "유의": bool(abs(rho) >= crit(len(df)))})
    return rho, p

ex_fin = [i for i in INDUSTRIES if i != "금융업"]
spear("배당수익률_시총가중", "금리베타 vs 배당수익률 (7산업·시총가중)")
spear("배당수익률_중앙값", "금리베타 vs 배당수익률 (7산업·중앙값)")
spear("부채비율_시총가중", "금리베타 vs 부채비율 (7산업·시총가중, 탐색적)")
spear("부채비율_시총가중", "금리베타 vs 부채비율 (금융제외 6산업, 탐색적)", subset=ex_fin)
spear("부채비율_중앙값", "금리베타 vs 부채비율 (7산업·중앙값, 탐색적)")
resdf = pd.DataFrame(res)
resdf.to_csv(H3_TABDIR / "spearman_results.csv", index=False, encoding="utf-8-sig")
print(resdf.to_string(index=False))
_n7 = len(INDUSTRIES)
print(f"\n※ n={_n7} 정확 임계값 |rho|≥{crit(_n7):.3f}. 순열 p 이산성으로 결과는 탐색적 참고.")

# --- leave-one-out (n=6): 어느 산업 하나가 결과를 견인하는지 점검 ---
loo_rows = []
for excl in INDUSTRIES:
    sub = [i for i in INDUSTRIES if i != excl]
    for col, label in [("배당수익률_시총가중", "배당_시총가중"), ("배당수익률_중앙값", "배당_중앙값"),
                        ("부채비율_시총가중", "부채_시총가중")]:
        df = pd.concat([beta.loc[sub].rename("beta"), char.loc[sub, col].rename("v")], axis=1).dropna()
        if len(df) < 3:
            continue
        rho, p = perm_p(df["beta"].values, df["v"].values)
        loo_rows.append({"제외산업": excl, "변수": label, "n": len(df), "rho": round(rho, 4),
                          "순열_p": round(p, 4), "임계값(0.05)": round(crit(len(df)), 4)})
loo_df = pd.DataFrame(loo_rows)
loo_df.to_csv(H3_TABDIR / "spearman_leave_one_out.csv", index=False, encoding="utf-8-sig")
print("\n[Leave-one-out, n=6]")
print(loo_df.pivot(index="제외산업", columns="변수", values="rho").reindex(INDUSTRIES).round(4).to_string())

# --- 배당수익률 결측처리 대안: 미공시연도를 0이 아닌 결측으로 처리 ---
fin_alt = fin.copy()
fin_alt["배당_대안"] = np.where(fin_alt["div_status"] == "ok", fin_alt["dividend_yield"], np.nan)
fpa_alt = (fin_alt.groupby(["industry", "stock_code"]).agg(배당_대안=("배당_대안", "mean"))
           .reset_index().merge(weight, on="stock_code", how="left"))
rows_alt = []
for ind, g in fpa_alt.groupby("industry"):
    rows_alt.append({"industry": ind, "배당_시총가중_대안": wavg(g, "배당_대안")})
char_alt = pd.DataFrame(rows_alt).set_index("industry").reindex(INDUSTRIES)
df_alt = pd.concat([beta.rename("beta"), char_alt["배당_시총가중_대안"].rename("v")], axis=1).dropna()
rho_alt, p_alt = perm_p(df_alt["beta"].values, df_alt["v"].values)
rho_cur = resdf.loc[resdf["관계"].str.contains("배당수익률.*시총가중"), "rho"].iloc[0]
p_cur = resdf.loc[resdf["관계"].str.contains("배당수익률.*시총가중"), "순열_p"].iloc[0]
print(f"\n[배당 결측처리 대안 비교] 현행(미공시=0) rho={rho_cur:.4f}, p={p_cur:.4f} "
      f"↔ 대안(미공시=결측 제외) rho={rho_alt:.4f}, p={p_alt:.4f}, n={len(df_alt)}, 임계값={crit(len(df_alt)):.4f}")

                          관계  n    rho   순열_p  임계값(0.05)    유의
    금리베타 vs 배당수익률 (7산업·시총가중)  7 0.8571 0.0238     0.7857  True
     금리베타 vs 배당수익률 (7산업·중앙값)  7 0.5357 0.2357     0.7857 False
금리베타 vs 부채비율 (7산업·시총가중, 탐색적)  7 0.6071 0.1667     0.7857 False
금리베타 vs 부채비율 (금융제외 6산업, 탐색적)  6 0.6571 0.1750     0.8857 False
 금리베타 vs 부채비율 (7산업·중앙값, 탐색적)  7 0.7500 0.0663     0.7857 False

※ n=7 정확 임계값 |rho|≥0.786. 순열 p 이산성으로 결과는 탐색적 참고.



[Leave-one-out, n=6]
변수    배당_시총가중  배당_중앙값  부채_시총가중
제외산업                          
건설업    0.7714  0.5429   0.6000
금융업    0.9429  0.6000   0.6571
운수장비   0.8857  0.8857   0.6571
의약품    0.7714  0.2571   0.3714
전기전자   0.8286  0.4857   0.3714
철강금속   0.8857  0.3714   0.9429
화학     0.8286  0.5429   0.6000



[배당 결측처리 대안 비교] 현행(미공시=0) rho=0.8571, p=0.0238 ↔ 대안(미공시=결측 제외) rho=0.7857, p=0.0480, n=7, 임계값=0.7857


In [105]:
# [셀 5] 부채비율 자료가용성 · 공통기간(2023-2025) 민감도 점검
fin_industry = fin[fin["industry"] == "금융업"]
cov_firm = fin_industry.groupby("company_name")["debt_ratio"].apply(lambda s: int(s.notna().sum()))
n_ok, n_total = int(fin_industry["debt_ratio"].notna().sum()), len(fin_industry)
print(f"[금융업 부채비율 자료가용성] {n_ok}/{n_total} firm-year 확보 ({n_ok/n_total:.0%})")
print(cov_firm.to_string())
print("\n주: fnlttSinglAcntAll·fnlttSinglAcnt 두 API 엔드포인트는 금융지주·은행·보험 8개사의 2022년 이전 "
      "XBRL 상세 재무제표를 반환하지 않는다(CFS·OFS 모두 013 조회된 데이타가 없습니다, 라이브 확인). "
      "이는 API 경로의 자료 공백일 뿐 사업보고서 자체는 존재하므로, STEP9에서 사업보고서 원문의 "
      "요약연결재무정보(부채총계·자본총계)를 직접 추출해 대부분 복구했다(data/raw/firm/"
      "dart_debt_ratio_recovery.csv, 출처 rcept_no 기록). 남은 결측은 우리금융지주 2016~2018년(3칸)뿐이며, "
      "이는 수집 실패가 아니라 우리금융지주가 2019년에 금융지주회사로 신설되어 그 이전 사업보고서 자체가 "
      "존재하지 않는 구조적 사유다.")

# 공통기간(2023-2025)으로 전 산업 재계산 — 기간을 통일해도 결론이 개선되는지 확인
fin_common = fin[fin["fiscal_year"].isin([2023, 2024, 2025])]
fpa_c = (fin_common.groupby(["industry", "stock_code"]).agg(부채비율=("debt_ratio", "mean"))
         .reset_index().merge(weight, on="stock_code", how="left"))
rows_c = []
for ind, g in fpa_c.groupby("industry"):
    rows_c.append({"industry": ind, "부채비율_시총가중_공통기간": wavg(g, "부채비율"),
                   "부채비율_중앙값_공통기간": g["부채비율"].median()})
char_c = pd.DataFrame(rows_c).set_index("industry").reindex(INDUSTRIES)

df_cw = pd.concat([beta.rename("beta"), char_c["부채비율_시총가중_공통기간"].rename("v")], axis=1).dropna()
rho_cw, p_cw = perm_p(df_cw["beta"].values, df_cw["v"].values)
df_cm = pd.concat([beta.rename("beta"), char_c["부채비율_중앙값_공통기간"].rename("v")], axis=1).dropna()
rho_cm, p_cm = perm_p(df_cm["beta"].values, df_cm["v"].values)

rho_full_w = resdf.loc[resdf["관계"].str.contains("부채비율.*7산업.*시총가중"), "rho"].iloc[0]
rho_full_m = resdf.loc[resdf["관계"].str.contains("부채비율.*7산업.*중앙값"), "rho"].iloc[0]
print(f"\n[공통기간(2023-2025) 재계산] 시총가중 rho={rho_cw:.4f}, p={p_cw:.4f} "
      f"(전체기간 rho={rho_full_w:.4f}) | 중앙값 rho={rho_cm:.4f}, p={p_cm:.4f} (전체기간 rho={rho_full_m:.4f})")

robustness = pd.DataFrame([
    {"점검": "금융업 firm-year 확보율", "값": f"{n_ok}/{n_total} ({n_ok/n_total:.0%})"},
    {"점검": "공통기간 재계산 vs 전체기간(시총가중)", "값": f"{rho_cw:.4f} vs {rho_full_w:.4f}"},
    {"점검": "공통기간 재계산 vs 전체기간(중앙값)", "값": f"{rho_cm:.4f} vs {rho_full_m:.4f}"},
    {"점검": "금융제외 6산업(시총가중)", "값": f"{resdf.loc[resdf['관계'].str.contains('금융제외'), 'rho'].iloc[0]:.4f}"},
    {"점검": "LOO 최댓값(시총가중, 제외산업)", "값": f"{loo_df[loo_df['변수']=='부채_시총가중']['rho'].max():.4f} "
     f"({loo_df[loo_df['변수']=='부채_시총가중'].sort_values('rho', ascending=False).iloc[0]['제외산업']} 제외)"},
])
robustness.to_csv(H3_TABDIR / "debt_ratio_robustness.csv", index=False, encoding="utf-8-sig")
print("\n[부채비율 강건성 요약]")
print(robustness.to_string(index=False))

[금융업 부채비율 자료가용성] 97/100 firm-year 확보 (97%)
company_name
HD한국조선해양    10
KB금융        10
LG          10
기업은행        10
메리츠금융지주     10
삼성생명        10
삼성화재        10
신한지주        10
우리금융지주       7
하나금융지주      10

주: fnlttSinglAcntAll·fnlttSinglAcnt 두 API 엔드포인트는 금융지주·은행·보험 8개사의 2022년 이전 XBRL 상세 재무제표를 반환하지 않는다(CFS·OFS 모두 013 조회된 데이타가 없습니다, 라이브 확인). 이는 API 경로의 자료 공백일 뿐 사업보고서 자체는 존재하므로, STEP9에서 사업보고서 원문의 요약연결재무정보(부채총계·자본총계)를 직접 추출해 대부분 복구했다(data/raw/firm/dart_debt_ratio_recovery.csv, 출처 rcept_no 기록). 남은 결측은 우리금융지주 2016~2018년(3칸)뿐이며, 이는 수집 실패가 아니라 우리금융지주가 2019년에 금융지주회사로 신설되어 그 이전 사업보고서 자체가 존재하지 않는 구조적 사유다.



[공통기간(2023-2025) 재계산] 시총가중 rho=0.6071, p=0.1667 (전체기간 rho=0.6071) | 중앙값 rho=0.3929, p=0.3956 (전체기간 rho=0.7500)

[부채비율 강건성 요약]
                    점검                값
     금융업 firm-year 확보율     97/100 (97%)
공통기간 재계산 vs 전체기간(시총가중) 0.6071 vs 0.6071
 공통기간 재계산 vs 전체기간(중앙값) 0.3929 vs 0.7500
        금융제외 6산업(시총가중)           0.6571
   LOO 최댓값(시총가중, 제외산업) 0.9429 (철강금속 제외)


**관찰.** 공통기간·집계기준·개별산업 제외에서 부채비율 순위상관은 5% 기준을 충족하지 못한다. 철강금속을 제외하면 순위상관이 크게 변하며, 상세 계수는 위 결과표에 제시한다. 이 관계가 소수 산업, 특히 철강금속에 좌우된다는 뜻이다.

In [106]:
# 5.A에서 바뀐 INDUSTRIES를 FINAL[37] 기준값으로 되돌린다(Appendix D가 이 순서를 쓴다).
INDUSTRIES = list(industries)

# 5. 종합 결과와 해석 범위

| 단계 | 질문 | 핵심 결과 | 해석 범위 |
|---|---|---|---|
| Baseline | 산업별 3년 금리베타가 같은가? | 동일성 기각 | 조건부 관련성 |
| H1 | 어느 곡선요인에서 차이가 나타나는가? | $\lambda=0.5$ Curvature에서 관찰 | $\lambda$와 공통변동에 민감 |
| H2 | 상승·하락 계수가 같은가? | 건설업에서만 BH 유의 | 전 산업 검정은 경계적 |
| H3 | 재무특성과 베타가 정렬되는가? | 배당은 일부 사양에서 정렬, 부채는 안정적 관계 미확인 | n=7 탐색적 결과 |

### 데이터가 직접 보여주는 것

- **산업별 3년 금리베타의 동일성은 기각되었다.**
- Curvature의 산업 간 차이는 $\lambda=0.5$에서 관찰되지만 Level·Slope·Credit Spread에서는 확인되지 않았다.
- 건설업의 3년물 상승·하락 계수는 서로 달랐고 Curvature 방향분해는 비유의였다.
- 배당수익률은 시가총액가중에서 순위정렬이 관찰됐지만 중앙값에서 약해졌다. 부채비율은 주요 사양에서 안정적 관계가 확인되지 않았다.

### 경제적으로 가능한 해석

의약품의 음(-) 금리베타, 낮은 배당수익률, R&D 지출은 미래 현금흐름 비중이 큰 기업에 대한 이론과 모순되지 않는다. 건설업의 방향별 반응은 자금조달 민감 가능성과 정합적일 수 있다. 두 해석 모두 본 분석이 직접 검정한 인과경로는 아니다.

### 이 연구만으로 확정할 수 없는 것

- 관측 금리변화에서 순수 정책충격과 경기·물가 정보효과를 분리하지 않았다.
- **대체 $\lambda$에서는 Curvature 산업 차이가 5% 수준에서 유지되지 않았다.**
- H2 전 산업 동시검정은 5% 기준에서 경계적이며, 두 요인 전체를 하나의 다중검정족으로 묶으면 판단이 약해진다.
- H3는 소수 산업의 집계량에 기초하며 duration과 인과관계를 직접 측정하지 않는다.


# Appendix

## A. Baseline 강건성

### 21개 쌍별 비교와 표준오차 방식

공동검정이 유의해도 어느 산업쌍이 다른지는 알 수 없다. 7개 산업의 21개 조합을 모두 비교하고, HAC와 날짜 클러스터 표준오차 각각에 BH 보정을 적용한다.


In [107]:
# 통합 회귀를 HAC와 날짜 클러스터 표준오차로 각각 추정한다.
baseline_long = panel[["date", "industry", "industry_return", "kospi_return", "d_y3"]].dropna()
baseline_formula = "industry_return ~ 0 + C(industry) + C(industry):kospi_return + C(industry):d_y3"
pooled_hac = smf.ols(baseline_formula, data=baseline_long).fit(
    cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS}
)
date_codes = pd.factorize(baseline_long["date"])[0]
pooled_cluster = smf.ols(baseline_formula, data=baseline_long).fit(
    cov_type="cluster", cov_kwds={"groups": date_codes}
)


def rate_param(industry):
    return f"C(industry)[{industry}]:d_y3"


param_names = list(pooled_hac.params.index)
pairwise_rows = []
for industry_a, industry_b in combinations(industries, 2):
    contrast = np.zeros(len(param_names))
    contrast[param_names.index(rate_param(industry_a))] = 1.0
    contrast[param_names.index(rate_param(industry_b))] = -1.0
    test_hac = pooled_hac.t_test(contrast)
    test_cluster = pooled_cluster.t_test(contrast)
    pairwise_rows.append({
        "industry_a": industry_a,
        "industry_b": industry_b,
        "beta_difference": float(np.asarray(test_hac.effect).squeeze()),
        "p_value_hac": float(np.asarray(test_hac.pvalue).squeeze()),
        "p_value_cluster": float(np.asarray(test_cluster.pvalue).squeeze()),
    })

pairwise_table = pd.DataFrame(pairwise_rows)
pairwise_table["BH p-value (HAC)"] = multipletests(
    pairwise_table["p_value_hac"], method="fdr_bh"
)[1]
pairwise_table["BH p-value (날짜 클러스터)"] = multipletests(
    pairwise_table["p_value_cluster"], method="fdr_bh"
)[1]
pairwise_table["유의 (HAC)"] = pairwise_table["BH p-value (HAC)"] < 0.05
pairwise_table["유의 (날짜 클러스터)"] = pairwise_table["BH p-value (날짜 클러스터)"] < 0.05
pairwise_table = pairwise_table.sort_values("BH p-value (날짜 클러스터)").reset_index(drop=True)

print(
    "BH 5% 유의 쌍 — HAC:", int(pairwise_table["유의 (HAC)"].sum()),
    "/ 21 · 날짜 클러스터:", int(pairwise_table["유의 (날짜 클러스터)"].sum()), "/ 21"
)
pairwise_table.round(5)


BH 5% 유의 쌍 — HAC: 6 / 21 · 날짜 클러스터: 6 / 21


,industry_a,industry_b,beta_difference,p_value_hac,p_value_cluster,BH p-value (HAC),BH p-value (날짜 클러스터),유의 (HAC),유의 (날짜 클러스터)
0,금융업,의약품,0.03830,0.00002,0.00002,0.00018,0.00025,True,True
1,운수장비,의약품,0.04430,0.00001,0.00001,0.00018,0.00025,True,True
2,의약품,철강금속,-0.04590,0.00004,0.00006,0.00030,0.00045,True,True
3,의약품,화학,-0.02962,0.00118,0.00142,0.00620,0.00745,True,True
4,의약품,전기전자,-0.02865,0.00188,0.00212,0.00790,0.00892,True,True
5,건설업,의약품,0.03027,0.00449,0.00514,0.01571,0.01798,True,True
6,운수장비,화학,0.01467,0.04080,0.03071,0.10711,0.08061,False,False
7,철강금속,화학,0.01628,0.03337,0.02747,0.10011,0.08061,False,False
8,전기전자,철강금속,-0.01725,0.07274,0.07860,0.15275,0.15006,False,False
9,운수장비,전기전자,0.01564,0.05698,0.07171,0.13295,0.15006,False,False


두 표준오차 방식에서 BH 5% 유의 쌍은 같은 여섯 쌍이며 모두 의약품을 포함한다. 쌍별 비교의 판정은 표준오차 방식에 따라 바뀌지 않았다.


In [108]:
# 동일한 여섯 개 제약으로 공동검정의 HAC·날짜 클러스터 결과를 비교한다.
reference = industries[0]
R = []
for industry in industries[1:]:
    restriction = np.zeros(len(param_names))
    restriction[param_names.index(rate_param(reference))] = 1.0
    restriction[param_names.index(rate_param(industry))] = -1.0
    R.append(restriction)
R = np.vstack(R)

joint_hac = pooled_hac.f_test(R)
joint_cluster_appendix = pooled_cluster.f_test(R)
se_comparison = pd.DataFrame([
    {
        "표준오차 방식": f"HAC (lag={HAC_LAGS})",
        "F": float(joint_hac.fvalue),
        "p-value": float(joint_hac.pvalue),
        "5% 판정": "기각" if float(joint_hac.pvalue) < 0.05 else "기각 못함",
    },
    {
        "표준오차 방식": "날짜 클러스터",
        "F": float(joint_cluster_appendix.fvalue),
        "p-value": float(joint_cluster_appendix.pvalue),
        "5% 판정": "기각" if float(joint_cluster_appendix.pvalue) < 0.05 else "기각 못함",
    },
])
se_comparison.round(6)


,표준오차 방식,F,p-value,5% 판정
0,HAC (lag=5),4.269420,0.000264,기각
1,날짜 클러스터,4.189156,0.000338,기각


## B. H1 민감도와 기술적 진단

### $\lambda$ 설정에 따른 결과 변화

**확인할 문제.** Curvature 결과가 $\lambda$ 설정에 의존하는가?


In [109]:
# lambda 민감도 — 개별 산업의 3요인 동시유의성(own-joint) p값이 lambda에 얼마나 안정적인가
MATURITIES = np.array([1.0, 3.0, 5.0, 10.0])


def nelson_siegel_loadings(maturities, lambda_value):
    level = np.ones_like(maturities)
    slope_loading = (1.0 - np.exp(-lambda_value * maturities)) / (lambda_value * maturities)
    curvature_loading = slope_loading - np.exp(-lambda_value * maturities)
    return np.column_stack([level, slope_loading, curvature_loading])


# lambda별 요인 변화(diff)는 국고채 원자료 고유의 날짜(2,462일)를 기준으로 계산해야 한다.
# extended_panel(산업 패널과 병합 후 2,452일)에서 다시 만들면, 병합 과정에서 빠진 날짜
# 바로 다음 거래일의 diff가 실제보다 넓은 간격으로 계산되어 값이 미세하게 달라진다.
_yield_raw = pd.read_csv(config.RAW_YIELD_CURVE_FILE, encoding="utf-8-sig")
_maturity_map = {"국고채(1년)": "y1", "국고채(3년)": "y3_new", "국고채(5년)": "y5", "국고채(10년)": "y10_new"}
_yield_selected = _yield_raw[_yield_raw["계정항목"].isin(_maturity_map.keys())].copy()
_yield_selected["maturity_name"] = _yield_selected["계정항목"].map(_maturity_map)
_meta_cols = [c for c in ["통계표", "계정항목", "단위", "변환"] if c in _yield_selected.columns]
_date_cols = [c for c in _yield_selected.columns if c not in _meta_cols + ["maturity_name"]]
_yield_long = _yield_selected.melt(id_vars=["maturity_name"], value_vars=_date_cols, var_name="date", value_name="yield")
_yield_long["date"] = pd.to_datetime(_yield_long["date"], errors="coerce")
_yield_long["yield"] = pd.to_numeric(_yield_long["yield"], errors="coerce")
yield_wide = (
    _yield_long.pivot(index="date", columns="maturity_name", values="yield")
    .reset_index().sort_values("date").dropna().reset_index(drop=True)
)
yield_wide.columns.name = None
yield_wide = yield_wide[["date", "y1", "y3_new", "y5", "y10_new"]]
yield_matrix = yield_wide[["y1", "y3_new", "y5", "y10_new"]].to_numpy(dtype=float)

# 검증: lambda=0.5에서 이렇게 재구성한 d_ns_curvature가 본문 lambda=0.5 변수과 정확히 일치하는지 확인
_chk_X = nelson_siegel_loadings(MATURITIES, config.NS_LAMBDA)
_chk_betas = yield_matrix @ np.linalg.pinv(_chk_X).T
_chk = pd.DataFrame({"date": yield_wide["date"], "d_C_chk": _chk_betas[:, 2]})
_chk["d_C_chk"] = _chk["d_C_chk"].diff()
_chk_merge = _chk.merge(extended_panel[["date", "d_ns_curvature"]].drop_duplicates("date"), on="date", how="inner")
_max_diff = (_chk_merge["d_C_chk"] - _chk_merge["d_ns_curvature"]).abs().max()
print(f"검증: 원자료 기준 재구성 d_ns_curvature vs 본문 lambda=0.5 변수, 최대 절대차 = {_max_diff:.2e}")

LAMBDA_GRID = [0.2, 0.3, 0.4, 0.5, 0.6, config.NS_LAMBDA_COMPARE, 1.0, 1.5, 2.0]


def lambda_case(lam):
    X = nelson_siegel_loadings(MATURITIES, lam)
    B = yield_matrix @ np.linalg.pinv(X).T
    f = pd.DataFrame(B, index=yield_wide["date"], columns=["L", "S", "C"]).diff().dropna()
    f.columns = ["d_L", "d_S", "d_C"]
    f = f.reset_index()
    dd = extended_panel[["date", "industry", "industry_return", "kospi_return"]].merge(f, on="date", how="inner")
    joint = {}
    for ind, g in dd.groupby("industry"):
        z = g[["industry_return", "kospi_return", "d_L", "d_S", "d_C"]].dropna()
        m = sm.OLS(z["industry_return"], sm.add_constant(z[["kospi_return", "d_L", "d_S", "d_C"]], has_constant="add")
                   ).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})
        joint[ind] = float(m.f_test("d_L = 0, d_S = 0, d_C = 0").pvalue)
    return {"lambda": lam, "동시검정_유의산업수": int(sum(v < 0.05 for v in joint.values())),
            "의약품_p": round(joint.get("의약품", np.nan), 5), "운수장비_p": round(joint.get("운수장비", np.nan), 5)}


lambda_sensitivity = pd.DataFrame([lambda_case(l) for l in LAMBDA_GRID])
lambda_sensitivity

검증: 원자료 기준 재구성 d_ns_curvature vs 본문 lambda=0.5 변수, 최대 절대차 = 7.19e-15


,lambda,동시검정_유의산업수,의약품_p,운수장비_p
0,0.2000,3,0.00440,0.01317
1,0.3000,3,0.00371,0.01310
2,0.4000,3,0.00314,0.01309
3,0.5000,3,0.00268,0.01313
4,0.6000,3,0.00232,0.01322
5,0.8041,3,0.00181,0.01348
6,1.0000,3,0.00151,0.01376
7,1.5000,3,0.00117,0.01434
8,2.0000,3,0.00106,0.01463


$\lambda=0.2$~2.0에서 개별 산업의 3요인 동시검정 결과를 먼저 확인한다. 이 검정은 산업 자신의 세 계수가 모두 0인지 묻는 것으로, 산업 간 Curvature 동일성 검정과는 다른 질문이다.


In [110]:
# Curvature 산업 간 차이 검정의 lambda 민감도 (본문 검정을 lambda 그리드 전체로 확장)
def industry_difference_by_lambda(lam):
    X = nelson_siegel_loadings(MATURITIES, lam)
    B = yield_matrix @ np.linalg.pinv(X).T
    f = pd.DataFrame(B, index=yield_wide["date"], columns=["L", "S", "C"]).diff().dropna()
    f.columns = ["d_L", "d_S", "d_C"]
    f = f.reset_index()
    dd = extended_panel[["date", "industry", "industry_return", "kospi_return"]].merge(f, on="date", how="inner")
    r = run_industry_difference_test(dd, ["d_L", "d_S", "d_C"], "lambda_grid")
    r["p_value_bh"] = multipletests(r["p_value"], method="fdr_bh")[1]
    return {"lambda": lam, "F_curvature": r.loc[r["variable"] == "d_C", "f_stat"].iloc[0],
            "BH_p_curvature": r.loc[r["variable"] == "d_C", "p_value_bh"].iloc[0]}


lambda_diff_sensitivity = pd.DataFrame([industry_difference_by_lambda(l) for l in LAMBDA_GRID])
lambda_diff_sensitivity.round(4)

,lambda,F_curvature,BH_p_curvature
0,0.2000,1.4090,0.2072
1,0.3000,3.0799,0.0158
2,0.4000,3.9124,0.0020
3,0.5000,3.5800,0.0047
4,0.6000,3.0786,0.0158
5,0.8041,2.2866,0.0998
6,1.0000,1.8091,0.2804
7,1.5000,1.2489,0.4171
8,2.0000,1.0679,0.4489


In [111]:
# 기존 lambda 격자 결과를 그림으로 표시한다.
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(
    lambda_diff_sensitivity["lambda"],
    lambda_diff_sensitivity["BH_p_curvature"],
    color=POS_COLOR,
    marker="o",
    linewidth=1.8,
)
ax.axhline(0.05, color=NEG_COLOR, linestyle="--", linewidth=1.2, label="BH p=0.05")

highlight_specs = [(0.5, "$\\lambda=0.5$"), (config.NS_LAMBDA_COMPARE, "$\\lambda=0.8041$")]
for lam, label in highlight_specs:
    point = lambda_diff_sensitivity.loc[
        np.isclose(lambda_diff_sensitivity["lambda"], lam)
    ].iloc[0]
    ax.scatter(point["lambda"], point["BH_p_curvature"], s=65, color=NEG_COLOR, zorder=4)
    ax.annotate(
        f"{label}\nBH p={point['BH_p_curvature']:.4f}",
        (point["lambda"], point["BH_p_curvature"]),
        xytext=(7, 9),
        textcoords="offset points",
        fontsize=9,
    )

ax.set_xticks(lambda_diff_sensitivity["lambda"])
ax.set_xticklabels([f"{x:g}" for x in lambda_diff_sensitivity["lambda"]])
ax.set_xlabel("Nelson–Siegel $\\lambda$")
ax.set_ylabel("Curvature 산업 간 동일성 검정 BH p-value")
ax.set_title("$\\lambda$ 설정에 따른 Curvature 산업 차이의 민감도", loc="left", fontweight="bold")
ax.legend(frameon=True, fontsize=9)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()


**판단.** 산업 간 Curvature 차이는 제한된 $\lambda$ 구간에서만 5% 수준에 도달하고, 곡선 재구성 오차가 가장 작은 대체값에서는 유지되지 않는다. 상세 $\lambda$별 검정값은 위 결과표에 제시한다. 결과는 전체 격자에서 안정적이지 않다.


In [112]:
# PCA 전체 적재값
YIELD_CHANGE_COLS = ["d_y1", "d_y3_new", "d_y5", "d_y10_new"]
MATURITY_LABELS = ["1년", "3년", "5년", "10년"]
Z = yield_changes[YIELD_CHANGE_COLS].to_numpy(dtype=float)
Zc = Z - Z.mean(axis=0)
eigval, eigvec = np.linalg.eigh(np.cov(Zc, rowvar=False))
order = np.argsort(eigval)[::-1]
eigval, eigvec = eigval[order], eigvec[:, order]
explained = eigval / eigval.sum()

pca_table = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(eigval))],
    "explained_ratio": explained,
    "cumulative_ratio": np.cumsum(explained),
})
for j, label in enumerate(MATURITY_LABELS):
    pca_table[f"loading_{label}"] = eigvec[j, :]
pca_table.round(4)

,component,explained_ratio,cumulative_ratio,loading_1년,loading_3년,loading_5년,loading_10년
0,PC1,0.9209,0.9209,-0.2363,-0.5334,-0.5849,-0.5635
1,PC2,0.0479,0.9688,-0.4581,-0.5357,-0.0096,0.7092
2,PC3,0.0224,0.9912,-0.8526,0.3826,0.2446,-0.2585
3,PC4,0.0088,1.0000,-0.0853,0.5311,-0.7733,0.3356


In [113]:
# VIF·요인 상관행렬
VIF_VARS = ["kospi_return", "d_ns_level", "d_ns_slope", "d_ns_curvature", "d_credit_spread"]
factor_daily = extended_panel[["date"] + VIF_VARS].drop_duplicates(subset="date").dropna()
vif_design = sm.add_constant(factor_daily[VIF_VARS], has_constant="add")
vif_values = {
    var: variance_inflation_factor(vif_design.to_numpy(dtype=float), i)
    for i, var in enumerate(vif_design.columns) if var != "const"
}
pd.Series(vif_values, name="VIF").round(3).to_frame()

,VIF
kospi_return,1.006
d_ns_level,7.254
d_ns_slope,7.014
d_ns_curvature,2.447
d_credit_spread,1.228


PC1 적재량과 VIF는 요인들의 공통변동을 확인하는 진단이다. Level과 Slope의 VIF 및 상관은 공통변동이 크다는 점을 보여준다. Curvature와 Credit Spread의 VIF는 상대적으로 낮지만, 이 값만으로 두 변수의 경제적 효과를 분리할 수는 없다. 상세 수치는 위 결과표에 제시한다.


In [114]:
# Gram–Schmidt 직교화와 FWL 검증
def orthogonalize_factors(daily, factor_cols):
    out = daily.copy()
    built = []
    for col in factor_cols:
        target = f"o_{col}"
        if not built:
            out[target] = out[col]
        else:
            X = sm.add_constant(out[built], has_constant="add")
            out[target] = sm.OLS(out[col], X).fit().resid
        built.append(target)
    return out, built


base_factors = ["d_ns_level", "d_ns_slope", "d_ns_curvature"]
orth_daily, orth_cols = orthogonalize_factors(
    extended_panel[["date"] + base_factors].drop_duplicates("date").dropna(),
    base_factors,
)
orth_panel = extended_panel.merge(
    orth_daily[["date"] + orth_cols], on="date", how="left", validate="many_to_one"
)
orth_betas = run_yield_curve_factor_model(
    orth_panel,
    x_cols=["kospi_return"] + orth_cols,
    factor_labels={
        "o_d_ns_level": "Level_orth",
        "o_d_ns_slope": "Slope_orth",
        "o_d_ns_curvature": "Curvature_orth",
    },
)
fwl_check = (
    yield_curve_factor_betas[yield_curve_factor_betas["factor"] == "Curvature"]
    [["industry", "coefficient", "std_error_hac", "p_value"]]
    .rename(columns={
        "coefficient": "coef_3factor",
        "std_error_hac": "se_3factor",
        "p_value": "p_3factor",
    })
    .merge(
        orth_betas[orth_betas["factor"] == "Curvature_orth"]
        [["industry", "coefficient", "std_error_hac", "p_value"]]
        .rename(columns={
            "coefficient": "coef_orth",
            "std_error_hac": "se_orth",
            "p_value": "p_orth",
        }),
        on="industry",
        validate="one_to_one",
    )
)
fwl_check["계수 절대차"] = (fwl_check["coef_3factor"] - fwl_check["coef_orth"]).abs()
fwl_check["SE 절대차"] = (fwl_check["se_3factor"] - fwl_check["se_orth"]).abs()
print("직교화 후 최대 요인상관:", orth_daily[orth_cols].corr().where(
    ~np.eye(len(orth_cols), dtype=bool)
).abs().max().max())
print("FWL 계수·SE 최대 절대차:", fwl_check[["계수 절대차", "SE 절대차"]].to_numpy().max())
fwl_check.round(8)


직교화 후 최대 요인상관: 1.6543716315681066e-15
FWL 계수·SE 최대 절대차: 9.952975943416931e-17


,industry,coef_3factor,se_3factor,p_3factor,coef_orth,se_orth,p_orth,계수 절대차,SE 절대차
0,건설업,-0.002338,0.003058,0.444568,-0.002338,0.003058,0.444568,0.0,0.0
1,금융업,0.002413,0.001370,0.078067,0.002413,0.001370,0.078067,0.0,0.0
2,운수장비,0.006207,0.002310,0.007206,0.006207,0.002310,0.007206,0.0,0.0
3,의약품,-0.008599,0.002828,0.002363,-0.008599,0.002828,0.002363,0.0,0.0
4,전기전자,0.001441,0.001582,0.362277,0.001441,0.001582,0.362277,0.0,0.0
5,철강금속,0.004148,0.002878,0.149483,0.004148,0.002878,0.149483,0.0,0.0
6,화학,-0.000080,0.001704,0.962436,-0.000080,0.001704,0.962436,0.0,0.0


**판단.** 마지막에 직교화한 Curvature의 계수와 표준오차는 원래 3요인 모형과 수치오차 범위에서 같다. 이는 FWL 정리의 기계적 결과다. 직교화는 공유변동을 앞 요인에 배정하며 순서에 의존하므로 강건한 인과증거로 사용하지 않는다.


In [115]:
# Curvature와 Credit Spread의 공유변동 및 FWL 성질
credit_daily = (
    extended_panel[["date", "kospi_return", "d_ns_level", "d_ns_slope",
                    "d_ns_curvature", "d_credit_spread"]]
    .drop_duplicates("date").dropna().reset_index(drop=True)
)
credit_X = sm.add_constant(
    credit_daily[["kospi_return", "d_ns_level", "d_ns_slope", "d_ns_curvature"]],
    has_constant="add",
)
credit_projection = sm.OLS(credit_daily["d_credit_spread"], credit_X).fit()
credit_daily["d_credit_orth"] = credit_projection.resid
credit_orth_panel = extended_panel.merge(
    credit_daily[["date", "d_credit_orth"]], on="date", how="left", validate="many_to_one"
)
credit_orth_betas = run_yield_curve_factor_model(
    credit_orth_panel,
    x_cols=["kospi_return", "d_ns_level", "d_ns_slope", "d_ns_curvature", "d_credit_orth"],
    factor_labels={
        "d_ns_level": "Level",
        "d_ns_slope": "Slope",
        "d_ns_curvature": "Curvature",
        "d_credit_orth": "CreditSpread_orth",
    },
)
credit_fwl = (
    yield_curve_factor_betas[yield_curve_factor_betas["factor"] == "Curvature"]
    [["industry", "coefficient"]]
    .rename(columns={"coefficient": "coef_3factor"})
    .merge(
        credit_orth_betas[credit_orth_betas["factor"] == "Curvature"]
        [["industry", "coefficient"]]
        .rename(columns={"coefficient": "coef_credit_orth"}),
        on="industry",
        validate="one_to_one",
    )
)
credit_fwl["절대차"] = (credit_fwl["coef_3factor"] - credit_fwl["coef_credit_orth"]).abs()
print(f"Credit Spread가 시장·곡선 3요인과 공유하는 변동(R²): {credit_projection.rsquared:.4f}")
print(f"3요인 vs Credit 직교화 모형 Curvature 계수 최대 절대차: {credit_fwl['절대차'].max():.2e}")
credit_fwl.round(8)


Credit Spread가 시장·곡선 3요인과 공유하는 변동(R²): 0.1859
3요인 vs Credit 직교화 모형 Curvature 계수 최대 절대차: 2.06e-16


,industry,coef_3factor,coef_credit_orth,절대차
0,건설업,-0.002338,-0.002338,0.0
1,금융업,0.002413,0.002413,0.0
2,운수장비,0.006207,0.006207,0.0
3,의약품,-0.008599,-0.008599,0.0
4,전기전자,0.001441,0.001441,0.0
5,철강금속,0.004148,0.004148,0.0
6,화학,-0.000080,-0.000080,0.0


Credit Spread 변동의 약 18.6%가 시장수익률과 곡선요인으로 설명된다. 직교화 뒤 Curvature 계수가 보존되는 것은 FWL의 필연적 결과다. **이 진단은 Curvature의 산업 차이가 Credit Spread로 흡수되지 않는다는 주장까지 식별하지 않는다.**


In [116]:
# 20년·30년을 추가했을 때 곡선 적합도와 요인 분리가 개선되는지 재계산한다.
def yield_rows_to_wide(raw, item_map):
    selected = raw[raw["계정항목"].isin(item_map)].copy()
    selected["maturity"] = selected["계정항목"].map(item_map)
    meta = [c for c in ["통계표", "계정항목", "단위", "변환"] if c in selected.columns]
    date_cols = [c for c in selected.columns if c not in meta + ["maturity"]]
    long = selected.melt(
        id_vars=["maturity"], value_vars=date_cols, var_name="date", value_name="yield"
    )
    long["date"] = pd.to_datetime(long["date"], errors="coerce")
    long["yield"] = pd.to_numeric(long["yield"], errors="coerce")
    return long.pivot(index="date", columns="maturity", values="yield").sort_index()


base_raw = pd.read_csv(config.RAW_YIELD_CURVE_FILE, encoding="utf-8-sig")
extra_raw = pd.read_csv(
    config.RAW_DIR / "rates" / "treasury_yield_curve_extended.csv.csv",
    encoding="utf-8-sig",
)
base_wide = yield_rows_to_wide(base_raw, {
    "국고채(1년)": "y1", "국고채(3년)": "y3", "국고채(5년)": "y5", "국고채(10년)": "y10",
})
extra_wide = yield_rows_to_wide(extra_raw, {
    "국고채(20년)": "y20", "국고채(30년)": "y30",
})
curve_wide = base_wide.join(extra_wide, how="inner").dropna()


def maturity_set_diagnostic(columns, maturities):
    matrix = curve_wide[columns].to_numpy(dtype=float)

    def rmse(lambda_value):
        loadings = nelson_siegel_loadings(np.asarray(maturities, float), lambda_value)
        factors = matrix @ np.linalg.pinv(loadings).T
        return float(np.sqrt(np.mean((matrix - factors @ loadings.T) ** 2)))

    optimum = minimize_scalar(rmse, bounds=(0.05, 3.0), method="bounded")
    loadings = nelson_siegel_loadings(np.asarray(maturities, float), optimum.x)
    factors = matrix @ np.linalg.pinv(loadings).T
    changes = pd.DataFrame(factors, columns=["L", "S", "C"]).diff().dropna()
    design = sm.add_constant(changes, has_constant="add")
    max_vif = max(
        variance_inflation_factor(design.to_numpy(dtype=float), i)
        for i in range(1, design.shape[1])
    )
    eigenvalues = np.linalg.eigvalsh(np.cov(np.diff(matrix, axis=0), rowvar=False))[::-1]
    return {
        "관측일": len(matrix),
        "최적 lambda": optimum.x,
        "곡선 RMSE": optimum.fun,
        "최대 VIF": max_vif,
        "PC1 설명비중(%)": eigenvalues[0] / eigenvalues.sum() * 100,
    }


maturity_extension = pd.DataFrame([
    {"만기 세트": "1·3·5·10년", **maturity_set_diagnostic(["y1", "y3", "y5", "y10"], [1, 3, 5, 10])},
    {"만기 세트": "+20·30년", **maturity_set_diagnostic(["y1", "y3", "y5", "y10", "y20", "y30"], [1, 3, 5, 10, 20, 30])},
])
maturity_extension.round(4)


,만기 세트,관측일,최적 lambda,곡선 RMSE,최대 VIF,PC1 설명비중(%)
0,1·3·5·10년,2462,0.8041,0.0108,9.4592,92.0854
1,+20·30년,2462,1.1339,0.0384,18.3560,88.9208


**판단.** 20년·30년을 추가하면 최적 $\lambda$에서 곡선 RMSE가 약 3.5배 커지고 최대 VIF도 약 9.46에서 18.36으로 높아진다. 만기를 늘려도 요인 분리가 개선되지 않아 본분석의 만기 세트는 바꾸지 않는다.


In [117]:
# HAC lag 5·8·10·20에서 3요인·4요인 계수의 유의성 판정을 비교한다.
HAC_LAG_GRID = [5, 8, 10, 20]
hac_frames = []
for lag_value in HAC_LAG_GRID:
    three = run_yield_curve_factor_model(extended_panel, hac_lags=lag_value).assign(
        model="3요인", hac_lags=lag_value
    )
    four = run_yield_curve_factor_model(
        extended_panel,
        hac_lags=lag_value,
        x_cols=["kospi_return", "d_ns_level", "d_ns_slope", "d_ns_curvature", "d_credit_spread"],
        factor_labels={
            "d_ns_level": "Level",
            "d_ns_slope": "Slope",
            "d_ns_curvature": "Curvature",
            "d_credit_spread": "CreditSpread",
        },
    ).assign(model="4요인", hac_lags=lag_value)
    hac_frames.extend([three, four])

hac_sensitivity = pd.concat(hac_frames, ignore_index=True)
hac_sensitivity["significant_5pct"] = hac_sensitivity["p_value"] < 0.05
hac_flip = (
    hac_sensitivity.groupby(["model", "industry", "factor"])["significant_5pct"]
    .nunique().reset_index(name="판정 수")
)
hac_summary = (
    hac_sensitivity.groupby(["model", "hac_lags"])
    .agg(유의_조합=("significant_5pct", "sum"), 전체_조합=("significant_5pct", "size"))
    .reset_index()
)
print("lag에 따라 5% 판정이 바뀐 조합:", int((hac_flip["판정 수"] > 1).sum()))
hac_summary


lag에 따라 5% 판정이 바뀐 조합: 0


,model,hac_lags,유의_조합,전체_조합
0,3요인,5,3,21
1,3요인,8,3,21
2,3요인,10,3,21
3,3요인,20,3,21
4,4요인,5,3,28
5,4요인,8,3,28
6,4요인,10,3,28
7,4요인,20,3,28


HAC 시차는 계수를 바꾸지 않고 표준오차와 p-value를 바꾼다. 위 표와 판정 변화 수는 lag 선택에 따른 유의성 민감도를 직접 보여준다. 산업 간 동일성 검정은 날짜 클러스터 표준오차를 사용하므로 이 HAC lag 비교의 대상이 아니다.


## C. H2 강건성·민감도 분석

아래 강건성 결과는 본 노트북에서 다시 계산한다. `07_asymmetric_rate_response.ipynb`에서 산출·검증된 표는 수치 일치를 확인하는 기준으로만 사용한다.


In [118]:
# Curvature 상승·하락 계수 동일성의 산업별 Wald 검정
curvature_wald_tests.round(5)


,industry,f_stat,p_value,p_value_bh
0,건설업,0.52478,0.46888,0.78409
1,금융업,0.01180,0.91351,0.91351
2,운수장비,1.11124,0.29192,0.78409
3,의약품,0.07584,0.78304,0.91351
4,전기전자,0.33969,0.56006,0.78409
5,철강금속,2.49889,0.11406,0.78409
6,화학,0.47997,0.48850,0.78409


### 극단값과 기간 의존성

**확인할 문제.** 건설업 비대칭이 일부 급변동일이나 특정 기간에 의해 만들어졌는가?


In [119]:
# H2 강건성 사양을 live 재계산하고 07의 검증표와 대조한다.
h2_panel = extended_panel.copy()


def h2_industry_wald(data, up_col, down_col, model_name, include_direction_dummy=False):
    rows = []
    for industry in industries:
        use_cols = ["industry_return", "kospi_return", up_col, down_col]
        sample = data.loc[data["industry"] == industry, use_cols].replace(
            [np.inf, -np.inf], np.nan
        ).dropna().copy()
        x_cols = ["kospi_return", up_col, down_col]
        if include_direction_dummy:
            sample = sample[(sample[up_col] > 0) | (sample[down_col] < 0)].copy()
            sample["direction_up_dummy"] = (sample[up_col] > 0).astype(float)
            x_cols = ["kospi_return", "direction_up_dummy", up_col, down_col]

        X = sm.add_constant(sample[x_cols], has_constant="add")
        model = sm.OLS(sample["industry_return"], X).fit(
            cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS}
        )
        ci = model.conf_int(alpha=0.05)
        wald = model.f_test(f"{up_col} = {down_col}")
        rows.append({
            "model": model_name,
            "industry": industry,
            "beta_up": model.params[up_col],
            "se_up": model.bse[up_col],
            "ci_lower_up": ci.loc[up_col, 0],
            "ci_upper_up": ci.loc[up_col, 1],
            "p_up": model.pvalues[up_col],
            "beta_down": model.params[down_col],
            "se_down": model.bse[down_col],
            "ci_lower_down": ci.loc[down_col, 0],
            "ci_upper_down": ci.loc[down_col, 1],
            "p_down": model.pvalues[down_col],
            "beta_difference": model.params[up_col] - model.params[down_col],
            "f_stat": float(np.asarray(wald.fvalue).squeeze()),
            "p_value": float(np.asarray(wald.pvalue).squeeze()),
            "nobs": int(model.nobs),
            "r_squared": model.rsquared,
        })
    return add_bh_correction(pd.DataFrame(rows), group_cols=("model",))


def assert_h2_reference_match(live, filename, keys, numeric_cols, tolerance=1e-8):
    reference = pd.read_csv(config.H2_TABDIR / filename)
    left = live[keys + numeric_cols].copy()
    right = reference[keys + numeric_cols].copy()
    for key in keys:
        left[key] = left[key].astype(str)
        right[key] = right[key].astype(str)
    compared = left.merge(right, on=keys, suffixes=("_live", "_reference"), validate="one_to_one")
    if len(compared) != len(left) or len(compared) != len(right):
        raise AssertionError(f"{filename}: live 결과와 검증표의 행 구성이 다릅니다.")
    differences = []
    for col in numeric_cols:
        diff = (compared[f"{col}_live"] - compared[f"{col}_reference"]).abs().max()
        differences.append(0.0 if pd.isna(diff) else float(diff))
    max_diff = max(differences, default=0.0)
    if max_diff >= tolerance:
        raise AssertionError(f"{filename}: 최대 수치 차이 {max_diff:.3e}")
    print(f"검증표 대조 완료: {filename} · 최대 차이 {max_diff:.2e}")


def add_rate_split(data, source_col, prefix):
    out = data.copy()
    out[f"{prefix}_up"] = out[source_col].clip(lower=0)
    out[f"{prefix}_down"] = out[source_col].clip(upper=0)
    return out


def subset_h2_dates(data, start=None, end=None, exclude_start=None, exclude_end=None):
    out = data.copy()
    if start is not None:
        out = out[out["date"] >= pd.Timestamp(start)]
    if end is not None:
        out = out[out["date"] <= pd.Timestamp(end)]
    if exclude_start is not None and exclude_end is not None:
        excluded = out["date"].between(pd.Timestamp(exclude_start), pd.Timestamp(exclude_end))
        out = out[~excluded]
    return out.copy()


daily_abs = (
    h2_panel[["date", "d_y3"]].drop_duplicates("date").dropna()
    .assign(abs_d_y3=lambda frame: frame["d_y3"].abs())
)
q01, q99 = h2_panel["d_y3"].dropna().quantile([0.01, 0.99])
winsor_panel = add_rate_split(
    h2_panel.assign(d_y3_winsor_1pct=h2_panel["d_y3"].clip(q01, q99)),
    "d_y3_winsor_1pct",
    "d_y3_winsor_1pct",
)
abs_q99 = daily_abs["abs_d_y3"].quantile(0.99)
abs_q995 = daily_abs["abs_d_y3"].quantile(0.995)

extreme_specs = {
    "baseline": (h2_panel, "d_y3_up", "d_y3_down"),
    "winsor_1pct": (winsor_panel, "d_y3_winsor_1pct_up", "d_y3_winsor_1pct_down"),
    "trim_abs_top_1pct": (
        h2_panel[~h2_panel["date"].isin(daily_abs.loc[daily_abs["abs_d_y3"] > abs_q99, "date"])],
        "d_y3_up", "d_y3_down",
    ),
    "trim_abs_top_0_5pct": (
        h2_panel[~h2_panel["date"].isin(daily_abs.loc[daily_abs["abs_d_y3"] > abs_q995, "date"])],
        "d_y3_up", "d_y3_down",
    ),
}
extreme_parts = []
for specification, (data, up_col, down_col) in extreme_specs.items():
    result = h2_industry_wald(data, up_col, down_col, specification)
    result["specification"] = specification
    extreme_parts.append(result)
extreme_all = pd.concat(extreme_parts, ignore_index=True)
extreme_construction = extreme_all[extreme_all["industry"] == "건설업"].copy()

period_specs = {
    "full_2016_2025": h2_panel,
    "split_2016_2020": subset_h2_dates(h2_panel, "2016-01-01", "2020-12-31"),
    "split_2021_2025": subset_h2_dates(h2_panel, "2021-01-01", "2025-12-31"),
    "period_2016_2019": subset_h2_dates(h2_panel, "2016-01-01", "2019-12-31"),
    "period_2020_2021": subset_h2_dates(h2_panel, "2020-01-01", "2021-12-31"),
    "period_2022_2025": subset_h2_dates(h2_panel, "2022-01-01", "2025-12-31"),
    "exclude_2020_2021": subset_h2_dates(h2_panel, exclude_start="2020-01-01", exclude_end="2021-12-31"),
    "exclude_2022_2023": subset_h2_dates(h2_panel, exclude_start="2022-01-01", exclude_end="2023-12-31"),
}
period_parts = []
for specification, data in period_specs.items():
    result = h2_industry_wald(data, "d_y3_up", "d_y3_down", specification)
    result["specification"] = specification
    period_parts.append(result)
period_all = pd.concat(period_parts, ignore_index=True)
period_construction = period_all[period_all["industry"] == "건설업"].copy()

nonzero_panel = h2_panel[(h2_panel["d_y3_up"] > 0) | (h2_panel["d_y3_down"] < 0)].copy()
direction_parts = [
    h2_industry_wald(h2_panel, "d_y3_up", "d_y3_down", "baseline_full"),
    h2_industry_wald(nonzero_panel, "d_y3_up", "d_y3_down", "nonzero_no_dummy"),
    h2_industry_wald(
        nonzero_panel, "d_y3_up", "d_y3_down", "nonzero_with_up_dummy",
        include_direction_dummy=True,
    ),
]
direction_all = pd.concat(direction_parts, ignore_index=True)
direction_all["specification"] = direction_all["model"]
direction_construction = direction_all[direction_all["industry"] == "건설업"].copy()

master_parts = []
for category, part in [
    ("extreme_value", extreme_construction),
    ("period", period_construction),
    ("direction_intercept", direction_construction),
]:
    selected = part[["specification", "beta_up", "beta_down", "f_stat", "p_value", "p_value_bh", "nobs"]].copy()
    selected.insert(0, "category", category)
    master_parts.append(selected)
construction_robustness = pd.concat(master_parts, ignore_index=True)
construction_robustness["robust_bh5"] = construction_robustness["p_value_bh"] < 0.05

assert_h2_reference_match(
    construction_robustness,
    "robustness_construction_master.csv",
    ["category", "specification"],
    ["beta_up", "beta_down", "f_stat", "p_value", "p_value_bh", "nobs"],
)

spec_labels = {
    "baseline": "기준 사양", "baseline_full": "기준 사양",
    "trim_abs_top_0_5pct": "절대변화 상위 0.5% 제거",
    "trim_abs_top_1pct": "절대변화 상위 1% 제거",
    "winsor_1pct": "1%/99% Winsorize",
    "exclude_2020_2021": "2020–2021 제외", "exclude_2022_2023": "2022–2023 제외",
    "full_2016_2025": "2016–2025 전체", "period_2016_2019": "2016–2019",
    "period_2020_2021": "2020–2021", "period_2022_2025": "2022–2025",
    "split_2016_2020": "2016–2020 분할", "split_2021_2025": "2021–2025 분할",
    "nonzero_no_dummy": "무변화일 제외", "nonzero_with_up_dummy": "상승일 절편 추가",
}
category_labels = {"extreme_value": "극단값", "period": "기간", "direction_intercept": "절편 사양"}
construction_robustness["사양"] = construction_robustness["specification"].map(spec_labels)
construction_robustness["구분"] = construction_robustness["category"].map(category_labels)
construction_robustness[[
    "구분", "사양", "beta_up", "beta_down", "f_stat", "p_value_bh", "nobs", "robust_bh5"
]].round(4)


검증표 대조 완료: robustness_construction_master.csv · 최대 차이 2.58e-13


,구분,사양,beta_up,beta_down,f_stat,p_value_bh,nobs,robust_bh5
0,극단값,기준 사양,-0.0216,0.0310,10.4178,0.0089,2451,True
1,극단값,1%/99% Winsorize,-0.0215,0.0395,7.7450,0.0380,2451,True
2,극단값,절대변화 상위 1% 제거,-0.0090,0.0369,3.7392,0.1864,2426,False
3,극단값,절대변화 상위 0.5% 제거,-0.0168,0.0379,5.4182,0.1401,2438,False
4,기간,2016–2025 전체,-0.0216,0.0310,10.4178,0.0089,2451,True
5,기간,2016–2020 분할,0.0168,0.0386,0.2286,0.7651,1226,False
6,기간,2021–2025 분할,-0.0355,0.0267,11.1433,0.0061,1225,True
7,기간,2016–2019,0.0353,0.0675,0.4010,0.7826,978,False
8,기간,2020–2021,-0.0198,-0.0011,0.1340,0.7720,496,False
9,기간,2022–2025,-0.0387,0.0251,10.0713,0.0109,977,True


**판단.** 부호 패턴은 여러 사양에서 유지되지만 상위 0.5%·1% 제거, 일부 기간 제외와 초기 기간 분할에서는 BH 5% 기준을 넘지 못한다. 건설업 결과는 일부 급변동일과 최근 기간에 민감할 수 있다.


In [120]:
# 급변동일 한 날짜씩 제외한 결과와 3년 Rolling Window 결과를 live 재계산한다.
top_extreme_dates = daily_abs.nlargest(10, "abs_d_y3")[["date", "d_y3", "abs_d_y3"]]
leave_one_out_rows = []
for _, extreme_day in top_extreme_dates.iterrows():
    result = h2_industry_wald(
        h2_panel[h2_panel["date"] != extreme_day["date"]],
        "d_y3_up", "d_y3_down", f"drop_{extreme_day['date'].date()}",
    )
    construction = result[result["industry"] == "건설업"].iloc[0]
    leave_one_out_rows.append({
        "excluded_date": extreme_day["date"].strftime("%Y-%m-%d"),
        "excluded_d_y3": extreme_day["d_y3"],
        "excluded_abs_d_y3": extreme_day["abs_d_y3"],
        "beta_up": construction["beta_up"], "beta_down": construction["beta_down"],
        "f_stat": construction["f_stat"], "p_value": construction["p_value"],
        "p_value_bh": construction["p_value_bh"], "nobs": construction["nobs"],
    })
extreme_date_loo = pd.DataFrame(leave_one_out_rows)

rolling_rows = []
min_year, max_year, window_years = int(h2_panel["date"].dt.year.min()), int(h2_panel["date"].dt.year.max()), 3
for start_year in range(min_year, max_year - window_years + 2):
    end_year = start_year + window_years - 1
    rolling_data = subset_h2_dates(
        h2_panel, start=f"{start_year}-01-01", end=f"{end_year}-12-31"
    )
    result = h2_industry_wald(
        rolling_data, "d_y3_up", "d_y3_down", f"rolling_{start_year}_{end_year}"
    )
    construction = result[result["industry"] == "건설업"].iloc[0]
    rolling_rows.append({
        "window": f"{start_year}-{end_year}", "start_year": start_year, "end_year": end_year,
        "n_dates": rolling_data["date"].nunique(),
        **construction[[
            "beta_up", "se_up", "ci_lower_up", "ci_upper_up",
            "beta_down", "se_down", "ci_lower_down", "ci_upper_down",
            "f_stat", "p_value", "p_value_bh",
        ]].to_dict(),
    })
rolling_construction = pd.DataFrame(rolling_rows)

assert_h2_reference_match(
    extreme_date_loo, "robustness_leave_one_out_extreme_dates.csv", ["excluded_date"],
    ["excluded_d_y3", "excluded_abs_d_y3", "beta_up", "beta_down", "f_stat", "p_value", "p_value_bh", "nobs"],
)
assert_h2_reference_match(
    rolling_construction, "robustness_rolling_3y_construction.csv", ["window"],
    ["n_dates", "beta_up", "beta_down", "f_stat", "p_value", "p_value_bh"],
)

print(
    "상위 급변동일을 하나씩 제외했을 때 BH 5% 유지:",
    f"{int((extreme_date_loo['p_value_bh'] < 0.05).sum())}/{len(extreme_date_loo)}",
)
rolling_construction[["window", "beta_up", "beta_down", "f_stat", "p_value_bh"]].round(4)


검증표 대조 완료: robustness_leave_one_out_extreme_dates.csv · 최대 차이 1.88e-13
검증표 대조 완료: robustness_rolling_3y_construction.csv · 최대 차이 1.45e-13
상위 급변동일을 하나씩 제외했을 때 BH 5% 유지: 10/10


,window,beta_up,beta_down,f_stat,p_value_bh
0,2016-2018,0.0556,0.0892,0.2973,0.7072
1,2017-2019,0.0349,0.0742,0.3928,0.7709
2,2018-2020,-0.0003,0.0491,0.6452,0.4221
3,2019-2021,-0.0175,0.0043,0.2369,0.7310
4,2020-2022,-0.0247,0.0081,2.6454,0.3650
5,2021-2023,-0.0237,0.0289,7.2905,0.0496
6,2022-2024,-0.0316,0.0254,7.6566,0.0203
7,2023-2025,-0.0377,0.0404,6.4400,0.0450


Rolling Window에서는 2021–2023 이후 창에서 주로 5% 수준에 도달한다. 전체기간 결과를 모든 시기에 안정적인 구조로 해석하기 어렵다.


In [121]:
# 표준오차 방식과 다중검정 범위를 바꾼 결과를 live 재계산한다.
def h2_joint_covariance(data, up_col, down_col, variable, covariance, maxlags=HAC_LAGS):
    required = ["date", "industry", "industry_return", "kospi_return", up_col, down_col]
    d = data[required].replace([np.inf, -np.inf], np.nan).dropna().sort_values(["date", "industry"])
    dummies = pd.get_dummies(d["industry"], prefix="ind", dtype=float)
    X = pd.concat([
        dummies,
        dummies.mul(d["kospi_return"].to_numpy(), axis=0).add_prefix("mkt_"),
        dummies.mul(d[up_col].to_numpy(), axis=0).add_prefix(f"{up_col}__"),
        dummies.mul(d[down_col].to_numpy(), axis=0).add_prefix(f"{down_col}__"),
    ], axis=1)
    names = list(X.columns)
    up_idx = [i for i, name in enumerate(names) if name.startswith(f"{up_col}__")]
    down_idx = [i for i, name in enumerate(names) if name.startswith(f"{down_col}__")]
    R = np.zeros((len(up_idx), len(names)))
    for row, (up_i, down_i) in enumerate(zip(up_idx, down_idx)):
        R[row, up_i], R[row, down_i] = 1.0, -1.0
    date_codes = pd.factorize(d["date"], sort=True)[0]
    base = sm.OLS(d["industry_return"].to_numpy(), X.to_numpy())
    if covariance == "date_cluster":
        result = base.fit(
            cov_type="cluster", cov_kwds={"groups": date_codes, "use_correction": True}
        )
        label, lag_value = "Date-cluster", np.nan
    else:
        result = base.fit(
            cov_type="hac-groupsum",
            cov_kwds={
                "time": date_codes, "maxlags": maxlags, "use_correction": "cluster",
                "df_correction": False,
            },
        )
        label, lag_value = f"Driscoll-Kraay lag={maxlags}", maxlags
    wald = result.f_test(R)
    return {
        "variable": variable, "covariance": covariance, "covariance_label": label,
        "maxlags": lag_value, "n_restrictions": R.shape[0],
        "f_stat": float(np.asarray(wald.fvalue).squeeze()),
        "p_value": float(np.asarray(wald.pvalue).squeeze()),
        "nobs": int(result.nobs), "n_dates": d["date"].nunique(),
    }


joint_rows = []
for variable, up_col, down_col in [
    ("Y3", "d_y3_up", "d_y3_down"),
    ("Curvature", "d_curv_up", "d_curv_down"),
]:
    joint_rows.append(h2_joint_covariance(h2_panel, up_col, down_col, variable, "date_cluster"))
    for lag in [1, 5, 10]:
        joint_rows.append(h2_joint_covariance(
            h2_panel, up_col, down_col, variable, "driscoll_kraay", maxlags=lag
        ))
covariance_sensitivity = pd.DataFrame(joint_rows)

joint_parts = []
for _, group in covariance_sensitivity.groupby("covariance_label", sort=False):
    group = group.copy()
    group["p_bh_across_2"] = multipletests(group["p_value"], method="fdr_bh")[1]
    group["p_bonferroni_across_2"] = multipletests(group["p_value"], method="bonferroni")[1]
    joint_parts.append(group)
joint_multiple_testing = pd.concat(joint_parts, ignore_index=True)

wald_y3 = h2_industry_wald(h2_panel, "d_y3_up", "d_y3_down", "Y3_Wald")
wald_curvature = h2_industry_wald(h2_panel, "d_curv_up", "d_curv_down", "Curvature_Wald")
wald_multiple_testing = pd.concat([wald_y3, wald_curvature], ignore_index=True).rename(
    columns={"p_value_bh": "p_bh_within_model_7", "significant_5pct_bh": "sig_within_model_7"}
)
wald_multiple_testing["p_bh_all_14"] = multipletests(
    wald_multiple_testing["p_value"], method="fdr_bh"
)[1]

assert_h2_reference_match(
    covariance_sensitivity, "robustness_joint_test_covariance.csv",
    ["variable", "covariance_label"], ["f_stat", "p_value", "nobs", "n_dates"],
)
assert_h2_reference_match(
    joint_multiple_testing, "robustness_multiple_testing_joint.csv",
    ["variable", "covariance_label"], ["p_value", "p_bh_across_2", "p_bonferroni_across_2"],
)
assert_h2_reference_match(
    wald_multiple_testing, "robustness_multiple_testing_wald_families.csv",
    ["model", "industry"], ["p_value", "p_bh_within_model_7", "p_bh_all_14"],
)

display(covariance_sensitivity.round(5))
display(joint_multiple_testing.round(5))
wald_multiple_testing.loc[
    wald_multiple_testing["industry"] == "건설업",
    ["model", "industry", "p_value", "p_bh_within_model_7", "p_bh_all_14"],
].round(5)


검증표 대조 완료: robustness_joint_test_covariance.csv · 최대 차이 8.45e-12
검증표 대조 완료: robustness_multiple_testing_joint.csv · 최대 차이 8.45e-12
검증표 대조 완료: robustness_multiple_testing_wald_families.csv · 최대 차이 7.86e-13


,variable,covariance,covariance_label,maxlags,n_restrictions,f_stat,p_value,nobs,n_dates
0,Y3,date_cluster,Date-cluster,NaN,7,2.10390,0.04007,17157,2451
1,Y3,driscoll_kraay,Driscoll-Kraay lag=1,1.0,7,2.28210,0.02540,17157,2451
2,Y3,driscoll_kraay,Driscoll-Kraay lag=5,5.0,7,2.24310,0.02804,17157,2451
3,Y3,driscoll_kraay,Driscoll-Kraay lag=10,10.0,7,2.26351,0.02662,17157,2451
4,Curvature,date_cluster,Date-cluster,NaN,7,0.76439,0.61729,17157,2451
5,Curvature,driscoll_kraay,Driscoll-Kraay lag=1,1.0,7,0.76009,0.62090,17157,2451
6,Curvature,driscoll_kraay,Driscoll-Kraay lag=5,5.0,7,0.81461,0.57492,17157,2451
7,Curvature,driscoll_kraay,Driscoll-Kraay lag=10,10.0,7,0.79953,0.58756,17157,2451


,variable,covariance,covariance_label,maxlags,n_restrictions,f_stat,p_value,nobs,n_dates,p_bh_across_2,p_bonferroni_across_2
0,Y3,date_cluster,Date-cluster,NaN,7,2.10390,0.04007,17157,2451,0.08014,0.08014
1,Curvature,date_cluster,Date-cluster,NaN,7,0.76439,0.61729,17157,2451,0.61729,1.00000
2,Y3,driscoll_kraay,Driscoll-Kraay lag=1,1.0,7,2.28210,0.02540,17157,2451,0.05079,0.05079
3,Curvature,driscoll_kraay,Driscoll-Kraay lag=1,1.0,7,0.76009,0.62090,17157,2451,0.62090,1.00000
4,Y3,driscoll_kraay,Driscoll-Kraay lag=5,5.0,7,2.24310,0.02804,17157,2451,0.05607,0.05607
5,Curvature,driscoll_kraay,Driscoll-Kraay lag=5,5.0,7,0.81461,0.57492,17157,2451,0.57492,1.00000
6,Y3,driscoll_kraay,Driscoll-Kraay lag=10,10.0,7,2.26351,0.02662,17157,2451,0.05325,0.05325
7,Curvature,driscoll_kraay,Driscoll-Kraay lag=10,10.0,7,0.79953,0.58756,17157,2451,0.58756,1.00000


,model,industry,p_value,p_bh_within_model_7,p_bh_all_14
0,Y3_Wald,건설업,0.00126,0.00885,0.01770
7,Curvature_Wald,건설업,0.46888,0.78409,0.75989


날짜 클러스터와 Driscoll–Kraay에서 3년물 전 산업 검정은 5% 유의수준의 경계에 있다. 3년물과 Curvature 공동검정을 하나의 검정족으로 묶으면 3년물 결과는 5% 수준을 넘지만, 건설업 산업별 Wald는 검정 범위를 넓혀도 유지된다. 상세 p-value는 위 결과표에 제시한다. 검정 단위에 따라 전 산업 결론의 강도가 달라짐을 함께 보고한다.


In [122]:
# 방향성과 변동크기 모형, 산업 간 검정과 경제적 크기를 live 재계산한다.
h2_panel["abs_d_y3"] = h2_panel["d_y3"].abs()


def fit_h2_abs_direction(data):
    cols = ["industry_return", "kospi_return", "abs_d_y3", "d_y3"]
    sample = data[cols].replace([np.inf, -np.inf], np.nan).dropna()
    X = sm.add_constant(sample[["kospi_return", "abs_d_y3", "d_y3"]], has_constant="add")
    return sm.OLS(sample["industry_return"], X).fit(
        cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS}
    )


size_rows, reparameterization_rows, magnitude_rows = [], [], []
for industry in industries:
    group = h2_panel[h2_panel["industry"] == industry]
    updown = fit_asymmetric(group, "d_y3_up", "d_y3_down")
    abs_direction = fit_h2_abs_direction(group)
    ci = abs_direction.conf_int(alpha=0.05)
    for component, variable in [("size", "abs_d_y3"), ("direction", "d_y3")]:
        size_rows.append({
            "model": "Y3_SizeDirection", "industry": industry, "component": component,
            "beta": abs_direction.params[variable], "se_hac": abs_direction.bse[variable],
            "t_value": abs_direction.tvalues[variable],
            "p_value": float(np.asarray(abs_direction.f_test(f"{variable}=0").pvalue).squeeze()),
            "ci_lower_95": ci.loc[variable, 0], "ci_upper_95": ci.loc[variable, 1],
            "nobs": int(abs_direction.nobs), "r_squared": abs_direction.rsquared,
            "hac_lags": HAC_LAGS,
        })

    beta_abs_implied = (updown.params["d_y3_up"] - updown.params["d_y3_down"]) / 2
    beta_dir_implied = (updown.params["d_y3_up"] + updown.params["d_y3_down"]) / 2
    reparameterization_rows.append({
        "industry": industry, "beta_up": updown.params["d_y3_up"],
        "beta_down": updown.params["d_y3_down"],
        "beta_abs_implied": beta_abs_implied, "beta_abs_direct": abs_direction.params["abs_d_y3"],
        "abs_diff": abs(beta_abs_implied - abs_direction.params["abs_d_y3"]),
        "beta_dir_implied": beta_dir_implied, "beta_dir_direct": abs_direction.params["d_y3"],
        "dir_diff": abs(beta_dir_implied - abs_direction.params["d_y3"]),
        "r2_updown": updown.rsquared, "r2_absdir": abs_direction.rsquared,
        "r2_diff": abs(updown.rsquared - abs_direction.rsquared), "nobs": int(abs_direction.nobs),
    })

    sample = group[["industry_return", "kospi_return", "abs_d_y3", "d_y3"]].dropna()
    ret_sd, abs_sd, kospi_sd = (
        sample["industry_return"].std(), sample["abs_d_y3"].std(), sample["kospi_return"].std()
    )
    beta_abs = abs_direction.params["abs_d_y3"]
    magnitude_rows.append({
        "industry": industry, "beta_abs": beta_abs,
        "p_value": abs_direction.pvalues["abs_d_y3"],
        "abs_d_y3_sd_pp": abs_sd, "abs_d_y3_sd_bp": abs_sd * 100,
        "ret_sd_pct": ret_sd * 100, "effect_1sd_pct": beta_abs * abs_sd * 100,
        "ratio_to_ret_sd_pct": abs(beta_abs * abs_sd) / ret_sd * 100,
        "effect_10bp_pct": beta_abs * 0.1 * 100,
        "kospi_effect_1sd_pct": abs_direction.params["kospi_return"] * kospi_sd * 100,
        "kospi_ratio_to_ret_sd_pct": abs(abs_direction.params["kospi_return"] * kospi_sd) / ret_sd * 100,
        "nobs": int(abs_direction.nobs),
    })

size_direction = add_bh_correction(
    pd.DataFrame(size_rows), group_cols=("model", "component")
)
reparameterization = pd.DataFrame(reparameterization_rows)
economic_magnitude = pd.DataFrame(magnitude_rows)

size_difference = add_bh_correction(
    run_industry_difference_test(h2_panel, ["abs_d_y3", "d_y3"], "Y3_SizeDirection"),
    group_cols=("model",),
)
asymmetry_difference = add_bh_correction(
    run_industry_difference_test(h2_panel, ["d_y3_up", "d_y3_down"], "Y3_Asymmetry"),
    group_cols=("model",),
)
size_industry_difference = pd.concat([size_difference, asymmetry_difference], ignore_index=True)

assert_h2_reference_match(
    size_direction, "uncert_size_direction_betas.csv", ["model", "industry", "component"],
    ["beta", "se_hac", "p_value", "p_value_bh", "nobs"],
)
assert_h2_reference_match(
    size_industry_difference, "uncert_industry_difference.csv", ["model", "variable"],
    ["f_stat", "p_value", "p_value_bh"],
)
assert_h2_reference_match(
    economic_magnitude, "uncert_economic_magnitude.csv", ["industry"],
    ["effect_1sd_pct", "ratio_to_ret_sd_pct", "effect_10bp_pct"],
)
assert_h2_reference_match(
    reparameterization, "uncert_reparameterization.csv", ["industry"],
    ["beta_abs_direct", "beta_dir_direct", "abs_diff", "dir_diff", "r2_diff"],
)

display(size_direction[size_direction["component"] == "size"][[
    "industry", "beta", "p_value_bh", "significant_5pct_bh"
]].round(5))
display(size_industry_difference[[
    "model", "variable", "f_stat", "p_value", "p_value_bh", "significant_5pct_bh"
]].round(5))
display(economic_magnitude[economic_magnitude["industry"] == "건설업"][[
    "industry", "effect_1sd_pct", "ratio_to_ret_sd_pct", "effect_10bp_pct"
]].round(4))
print(
    "상승·하락 모형과 크기·방향 재매개변수화의 최대 계수 차:",
    reparameterization[["abs_diff", "dir_diff"]].to_numpy().max(),
)


검증표 대조 완료: uncert_size_direction_betas.csv · 최대 차이 1.11e-16
검증표 대조 완료: uncert_industry_difference.csv · 최대 차이 8.88e-16
검증표 대조 완료: uncert_economic_magnitude.csv · 최대 차이 9.25e-17
검증표 대조 완료: uncert_reparameterization.csv · 최대 차이 9.28e-17


,industry,beta,p_value_bh,significant_5pct_bh
0,건설업,-0.02632,0.00885,True
2,금융업,-0.00591,0.42965,False
4,운수장비,0.00023,0.97451,False
6,의약품,0.01051,0.42965,False
8,전기전자,0.00663,0.42965,False
10,철강금속,0.00986,0.42965,False
12,화학,-0.00171,0.91398,False


,model,variable,f_stat,p_value,p_value_bh,significant_5pct_bh
0,Y3_SizeDirection,abs_d_y3,2.21287,0.03920,0.03920,True
1,Y3_SizeDirection,d_y3,4.13725,0.00038,0.00077,True
2,Y3_Asymmetry,d_y3_up,3.51619,0.00182,0.00363,True
3,Y3_Asymmetry,d_y3_down,3.13038,0.00465,0.00465,True


,industry,effect_1sd_pct,ratio_to_ret_sd_pct,effect_10bp_pct
0,건설업,-0.0776,4.3414,-0.2632


상승·하락 모형과 크기·방향 재매개변수화의 최대 계수 차: 3.148523108897905e-16


**판단.** 건설업은 방향 비대칭과 별개로 변동크기 계수도 BH 보정 후 유의하다. 산업 간 크기계수 차이는 방향계수 차이보다 약하고 건설업을 제외하면 남지 않는다. 상세 p-value는 위 결과표에 제시한다. 이는 H2의 주 검정을 대체하지 않는 보조 관찰이다.


## D. H3 표본 민감도와 보조표

### Leave-one-out

**확인할 문제.** n=7의 순위상관이 한 산업 제외에 얼마나 민감한가?


In [123]:
# Leave-one-out (n=6) — 산업 하나를 제외했을 때 순위상관이 얼마나 흔들리는가
loo_rows = []
for excl in INDUSTRIES:
    sub = [i for i in INDUSTRIES if i != excl]
    for col, label in [("배당수익률_시총가중", "배당_시총가중"), ("배당수익률_중앙값", "배당_중앙값"), ("부채비율_시총가중", "부채_시총가중")]:
        df = pd.concat([beta.loc[sub].rename("beta"), char.loc[sub, col].rename("v")], axis=1).dropna()
        if len(df) < 3:
            continue
        rho, p = perm_p(df["beta"].values, df["v"].values)
        loo_rows.append({"제외산업": excl, "변수": label, "rho": round(rho, 4), "순열_p": round(p, 4)})
loo_df = pd.DataFrame(loo_rows)
loo_df.pivot(index="제외산업", columns="변수", values="rho").reindex(INDUSTRIES).round(4)

변수,배당_시총가중,배당_중앙값,부채_시총가중
제외산업,,,
건설업,0.7714,0.5429,0.6000
금융업,0.9429,0.6000,0.6571
운수장비,0.8857,0.8857,0.6571
의약품,0.7714,0.2571,0.3714
전기전자,0.8286,0.4857,0.3714
철강금속,0.8857,0.3714,0.9429
화학,0.8286,0.5429,0.6000


배당수익률의 시가총액가중 상관은 어느 산업을 제외해도 양(+)을 유지한다. 부채비율은 철강금속 제외 시 순위상관이 크게 변한다. 상세 상관계수는 위 결과표에 제시한다. 이 결과는 부채비율의 강한 관계를 새로 발견한 것이 아니라 소표본 민감성을 보여준다.


In [124]:
# 7개 산업의 시가총액 집중도와 의약품 R&D 보조표
concentration_rows = []
for ind in INDUSTRIES:
    values = concentration(sel[sel["industry"] == ind][["market_cap_2024"]].copy())
    values["industry"] = ind
    concentration_rows.append(values)
industry_concentration = pd.DataFrame(concentration_rows).set_index("industry").sort_values(
    "hhi", ascending=False
)
display(industry_concentration.round(3))
sup[["company_name", "rnd_to_sales_pct", "기준"]].sort_values("rnd_to_sales_pct").round(2)


,top1_share_pct,top3_share_pct,hhi,n_firms
industry,,,,
의약품,50.639,88.340,0.357,10
전기전자,52.955,87.641,0.346,10
철강금속,37.745,84.020,0.296,10
화학,25.889,59.753,0.162,10
운수장비,24.081,59.645,0.154,10
건설업,23.783,52.954,0.140,10
금융업,18.224,42.706,0.112,10


,company_name,rnd_to_sales_pct,기준
9,대웅,0.01,DART 경상연구개발비/매출(비용처리분)
8,유한양행,6.88,DART 경상연구개발비/매출(비용처리분)
7,삼성바이오로직스,8.77,사업보고서 연구개발비/매출(연결·총계)
6,종근당,9.45,DART 경상연구개발비/매출(비용처리분)
5,대웅제약,11.48,DART 경상연구개발비/매출(비용처리분)
4,녹십자,11.63,사업보고서 연구개발비/매출(연결·총계)
3,한올바이오파마,12.53,DART 경상연구개발비/매출(비용처리분)
2,한미약품,14.67,DART 경상연구개발비/매출(비용처리분)
1,셀트리온,15.34,사업보고서 연구개발비/매출(연결·총계)
0,SK바이오사이언스,32.03,사업보고서 연구개발비/매출(연결·총계)


정확 순열검정은 가능한 산업 순위 순열을 모두 사용한다. 이 소표본에서는 양측 5% 기준을 만족하려면 절대 순위상관이 매우 커야 하며, 정확 임계값은 위 결과표에 제시한다.


## 8. 전체 분석 Index

| 분석/요청 가능 내용 | Master 위치 | Final 포함 | Source | 역할 |
|---|---|---|---|---|
| 핵심 금리·곡률 계수의 10bp 환산 해석 | §2·§3·§4·§5 | Y | FINAL/04/06/07 | 전체 본문 공통 |
| 산업별 3년 금리베타(HAC) | §2 | Y | FINAL/04 | Baseline |
| 산업 간 동일성 공동검정 | §2 | Y | FINAL/05 | Baseline |
| 21쌍 산업 쌍별 검정 | §2.A.3 | N | 05 | Baseline 보조 |
| 산업 쌍별 차이 Heatmap | §2.A.3 | N | 05 | Baseline 보조 |
| 산업별 R² 비교 | §2.A.1 | N | 04 | Baseline 보조 |
| 원p vs BH p 비교 | §2.A.2 | N | 04 | Baseline 보조 |
| 만기별(3Y·10Y·장단기차) 베타 비교 | §3 | Y | FINAL/06 | H1 |
| NS Level·Slope·Curvature 3요인 모형 | §3 | Y | FINAL/06 | H1 |
| Curvature 산업 간 차이 검정 | §3 | Y | FINAL/06 | H1 |
| λ 선택 근거(4기준) | §3.A.1 | N | 06 | H1 강건성 |
| λ 민감도(0.2~2.0) | §3.A.1 | N | 06 | H1 강건성 |
| 나비 스프레드로 본 곡률 강건성 | §3.A.2 | N | 06 | H1 강건성 |
| Credit Spread 단독·확장모형 | §3.A.3 | N | 06 | H1 강건성 |
| 수익률곡선 PCA(차원) | §3.A.4 | N | 06 | H1 강건성 |
| 모형 절약성(parsimony) | §3.A.4 | N | 06 | H1 강건성 |
| Rolling PCA·정책국면별 곡선구조 | §3.A.5 | N | 06 | H1 강건성 |
| own-joint(산업 자신 3요인) 검정 | §3.A.6 | N | 06 | H1 강건성 |
| 2요인(Level+Slope) 모형 | §3.A.7 | N | 06 | H1 강건성 |
| 직교화(FWL)·3모형 비교 | §3.A.8 | N | 06 | H1 강건성 |
| HAC lag 시차 민감도 | Appendix B | Y | FINAL/06 | H1 강건성 |
| 20년·30년 만기 확장 검토 | Appendix B | Y | FINAL/06 | H1 강건성 |
| 곡률–시장베타 관계 | §3.A.9 | N | 06 | H1 탐색 |
| 상승·하락 비대칭 회귀(3년물) | §4 | Y | FINAL/07 | H2 |
| 건설업 Wald 검정 | §4 | Y | FINAL/07 | H2 |
| 대칭모형 대비 | §4 | Y | FINAL/07 | H2 |
| 전 산업 동시 대칭검정 | §4 | Y | FINAL/07 | H2 |
| 정책국면(2구간) 보조분석 | §4 | Y | FINAL | H2 |
| 표본균형·기술통계 | §4.A.1 | N | 07 | H2 보조 |
| 상승·하락 계수 10bp 환산표 | §4.A.2 | N | 07 | H2 보조 |
| 곡률 상승·하락 계수 | §4.A.3 | N | 07 | H2 보조 |
| 산업 간 차이(3년물·곡률) | §4.A.4 | N | 07 | H2 보조 |
| 극단값·기간·Rolling·LOO 검증 | Appendix C | Y | FINAL/07 | H2 강건성 |
| SE 방식·다중검정 범위 민감도 | Appendix C | Y | FINAL/07 | H2 강건성 |
| 방향더미 절편 강건성 | §4.A.5 | N | 07 | H2 강건성 |
| 크기계수 조건별 산업 차이 | §4.A.6 | N | 07 | H2 보조 |
| \|Δy3\| 경제적 크기 | §4.A.7 | N | 07 | H2 보조 |
| 정책국면(5국면) β·95%CI | §4.A.8 | N | 08 | H2 확장 |
| 재무특성 vs 금리베타 산점도 | §5 | Y | FINAL/10 | H3 |
| 의약품 제외 공동검정 | §5 | Y | FINAL/10 | H3 |
| 배당수익률 Spearman | §5 | Y | FINAL/10 | H3 |
| 부채비율 Spearman(전체기간) | §5 | Y | FINAL/10 | H3 |
| 시가총액 집중도(HHI)·R&D | §5 | Y | FINAL/10 | H3 |
| Leave-one-out(n=6) | Appendix D | Y | FINAL/10 | H3 강건성 |
| 만기구조 대조(HAC vs 날짜클러스터) | Appendix A | Y | FINAL | 강건성 |
| 부채비율 공통기간·집계 민감도 | §5.A.1 | N | 10 | H3 강건성 |

Master 위치는 최종 셀 배치 기준이며, "Final 포함=Y"는 `FINAL_paper_analysis.ipynb`에 동일 분석이 이미 실려 있음을 뜻한다.